Cell 1 — Check GPU

In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


Cell 2 — Install the packages we need

In [5]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn

Cell 3 — Import libraries

In [6]:
import os
import json
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

print("Imports successful.")

Imports successful.


Cell 4 — Upload medical_rag_results_notebook1_final.csv

In [7]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/classnotes/medical-rag-results-notebook1-final-csv/medical_rag_results_notebook1_final.csv


Cell 5 — Load the file

In [8]:
rag_file = "/kaggle/input/datasets/classnotes/medical-rag-results-notebook1-final-csv/medical_rag_results_notebook1_final.csv"

rag_results_df = pd.read_csv(rag_file)

print("Shape:", rag_results_df.shape)
print("Columns:")
print(rag_results_df.columns.tolist())

Shape: (200, 9)
Columns:
['test_id', 'question', 'retrieved_source_ids', 'retrieved_pubids', 'retrieved_evidence', 'generated_answer', 'reference_answer', 'final_decision', 'insufficient_evidence']


Cell 6 — Inspect the RAG data

In [9]:
display(
    rag_results_df[
        [
            "test_id",
            "question",
            "generated_answer",
            "reference_answer",
            "final_decision"
        ]
    ].head(10)
)

,test_id,question,generated_answer,reference_answer,final_decision
0,521,Pap smears with glandular cell abnormalities: ...,The available evidence is insufficient to answ...,Pap smears with glandular cell abnormalities a...,yes
1,737,Are patients with Werlhof's disease at increas...,"Based on the provided evidence, there is no di...",Patients with WD may possibly undergo cardiac ...,no
2,740,Is there a connection between sublingual varic...,"Based on the provided evidence, there appears ...",An association was found between sublingual va...,yes
3,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...",HIE was associated with reduced repeat imaging...,yes
4,411,The promise of specialty pharmaceuticals: are ...,"Based on the evidence provided, it appears tha...",Current evidence suggests that when used in ta...,yes
5,678,The colour of pain: can patients use colour to...,"Based on the provided evidence, there is no di...","For some patients, using colour to describe th...",yes
6,626,Is fluoroscopy essential for retrieval of lowe...,"No, fluoroscopy is not essential for the retri...",To uphold the notion for radiation exposure to...,no
7,513,Increased neutrophil migratory activity after ...,The available evidence is insufficient to answ...,These data indicate that major blunt trauma en...,yes
8,859,Does the SCL 90-R obsessive-compulsive dimensi...,The available evidence is insufficient to answ...,Our data suggest that the SCL 90-R is best vie...,yes
9,136,Can transcranial direct current stimulation be...,The available evidence is insufficient to answ...,a-tDCS could be useful in identifying residual...,yes


Cell 7 — Check missing values

In [10]:
print("Missing values:")
print(rag_results_df.isna().sum())

Missing values:
test_id                  0
question                 0
retrieved_source_ids     0
retrieved_pubids         0
retrieved_evidence       0
generated_answer         0
reference_answer         0
final_decision           0
insufficient_evidence    0
dtype: int64


Cell 8 — Prepare LettuceDetect input columns

In [11]:
lettuce_df = rag_results_df[
    [
        "test_id",
        "question",
        "retrieved_evidence",
        "generated_answer",
        "reference_answer",
        "final_decision"
    ]
].copy()

lettuce_df = lettuce_df.rename(
    columns={
        "retrieved_evidence": "context",
        "generated_answer": "answer"
    }
)

print("Shape:", lettuce_df.shape)
display(lettuce_df.head(3))

Shape: (200, 6)


,test_id,question,context,answer,reference_answer,final_decision
0,521,Pap smears with glandular cell abnormalities: ...,['To compare adherence to follow-up recommenda...,The available evidence is insufficient to answ...,Pap smears with glandular cell abnormalities a...,yes
1,737,Are patients with Werlhof's disease at increas...,['It was the aim of the present study to elabo...,"Based on the provided evidence, there is no di...",Patients with WD may possibly undergo cardiac ...,no
2,740,Is there a connection between sublingual varic...,['Esophageal varices are present in 30% to 40%...,"Based on the provided evidence, there appears ...",An association was found between sublingual va...,yes


Cell 9 — Convert retrieved evidence into plain context text

In [12]:
import ast

def clean_context(value):
    # Convert string representation of a list into a real list
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return "\n\n".join(str(x) for x in parsed)
        except (ValueError, SyntaxError):
            pass

    # If it is already a list
    if isinstance(value, list):
        return "\n\n".join(str(x) for x in value)

    return str(value)


lettuce_df["context"] = lettuce_df["context"].apply(clean_context)

print("Context conversion complete.")
print()
print("First context:")
print(lettuce_df.loc[0, "context"][:2000])

Context conversion complete.

First context:
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

Participant responses between 2000 and 2007 for urine specimens with a reference diagnosis of high-grade urothelial carcino

Cell 10 — Check the three actual LettuceDetect inputs

In [13]:
print("QUESTION:")
print(lettuce_df.loc[0, "question"])

print("\n" + "=" * 80)

print("CONTEXT:")
print(lettuce_df.loc[0, "context"][:2000])

print("\n" + "=" * 80)

print("ANSWER:")
print(lettuce_df.loc[0, "answer"])

QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

CONTEXT:
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

Participant responses between 2000 and 2007 for urine specime

Cell 11 — Check the LettuceDetect package/repository

In [14]:
!git clone https://github.com/mdwoicke/Hallucination-LettuceDetect.git

Cloning into 'Hallucination-LettuceDetect'...
remote: Enumerating objects: 239, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 239 (delta 63), reused 54 (delta 54), pack-reused 152 (from 1)
Receiving objects: 100% (239/239), 2.23 MiB | 10.73 MiB/s, done.
Resolving deltas: 100% (112/112), done.


In [15]:
import os

print("Repository contents:")
for item in os.listdir("/kaggle/working/Hallucination-LettuceDetect"):
    print(item)

Repository contents:
README.md
scripts
tests
.git
demo
.github
lettucedetect
docs
pyproject.toml
assets
LICENSE
MANIFEST.in
.gitignore


Cell 12 — Inspect the repository structure

In [16]:
!find /kaggle/working/Hallucination-LettuceDetect -maxdepth 2 -type f | head -100

/kaggle/working/Hallucination-LettuceDetect/README.md
/kaggle/working/Hallucination-LettuceDetect/scripts/analyze_tokens.py
/kaggle/working/Hallucination-LettuceDetect/scripts/train.py
/kaggle/working/Hallucination-LettuceDetect/scripts/evaluate.py
/kaggle/working/Hallucination-LettuceDetect/scripts/upload.py
/kaggle/working/Hallucination-LettuceDetect/tests/pytest.ini
/kaggle/working/Hallucination-LettuceDetect/tests/__init__.py
/kaggle/working/Hallucination-LettuceDetect/tests/conftest.py
/kaggle/working/Hallucination-LettuceDetect/tests/run_pytest.py
/kaggle/working/Hallucination-LettuceDetect/tests/test_inference_pytest.py
/kaggle/working/Hallucination-LettuceDetect/.git/HEAD
/kaggle/working/Hallucination-LettuceDetect/.git/description
/kaggle/working/Hallucination-LettuceDetect/.git/config
/kaggle/working/Hallucination-LettuceDetect/.git/packed-refs
/kaggle/working/Hallucination-LettuceDetect/.git/index
/kaggle/working/Hallucination-LettuceDetect/demo/streamlit_demo.py
/kaggle/wor

Cell 13 — Read the official README

In [17]:
with open(
    "/kaggle/working/Hallucination-LettuceDetect/README.md",
    "r",
    encoding="utf-8"
) as f:
    readme = f.read()

print(readme[:12000])

# LettuceDetect 🥬🔍


<p align="center">
  <img src="https://github.com/KRLabsOrg/LettuceDetect/blob/main/assets/lettuce_detective.png?raw=true" alt="LettuceDetect Logo" width="400"/>
  <br><em>Because even AI needs a reality check! 🥬</em>
</p>

LettuceDetect is a lightweight and efficient tool for detecting hallucinations in Retrieval-Augmented Generation (RAG) systems. It identifies unsupported parts of an answer by comparing it to the provided context. The tool is trained and evaluated on the [RAGTruth](https://aclanthology.org/2024.acl-long.585/) dataset and leverages [ModernBERT](https://github.com/AnswerDotAI/ModernBERT) for long-context processing, making it ideal for tasks requiring extensive context windows.

Our models are inspired from the [Luna](https://aclanthology.org/2025.coling-industry.34/) paper which is an encoder-based model and uses a similar token-level approach.

[![PyPI](https://img.shields.io/pypi/v/lettucedetect)](https://pypi.org/project/lettucedetect/)
[![Lic

Cell 14 — Inspect the actual Python package

In [18]:
!find /kaggle/working/Hallucination-LettuceDetect/lettucedetect -type f -maxdepth 3 -print

find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/__init__.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/preprocess/__init__.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/preprocess/preprocess_ragtruth.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/trainer.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/__init__.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/inference.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/evaluator.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/datasets/ragtruth.py
/kaggle/working/Hallucination-LettuceDetect/lettucedetect/datasets/__init__.py


Cell 15 — Inspect the inference test

In [19]:
with open(
    "/kaggle/working/Hallucination-LettuceDetect/tests/test_inference_pytest.py",
    "r",
    encoding="utf-8"
) as f:
    test_code = f.read()

print(test_code)

"""Pytest tests for the inference module."""

from unittest.mock import MagicMock, patch

import pytest
import torch

from lettucedetect.models.inference import HallucinationDetector, TransformerDetector


@pytest.fixture
def mock_tokenizer():
    """Create a mock tokenizer for testing."""
    tokenizer = MagicMock()
    tokenizer.encode.return_value = [101, 102, 103, 104, 105]
    return tokenizer


@pytest.fixture
def mock_model():
    """Create a mock model for testing."""
    model = MagicMock()
    mock_output = MagicMock()
    mock_output.logits = torch.tensor([[[0.1, 0.9], [0.8, 0.2], [0.3, 0.7]]])
    model.return_value = mock_output
    return model


class TestHallucinationDetector:
    """Tests for the HallucinationDetector class."""

    def test_init_with_transformer_method(self):
        """Test initialization with transformer method."""
        with patch("lettucedetect.models.inference.TransformerDetector") as mock_transformer:
            detector = HallucinationDetect

Cell 16 — Install the actual LettuceDetect repository

In [20]:
%cd /kaggle/working/Hallucination-LettuceDetect

!pip install -q -e .

/kaggle/working/Hallucination-LettuceDetect
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 79.0 MB/s eta 0:00:00:00:0100:01
  Building editable for lettucedetect (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requi

Cell 17 — Import the official LettuceDetect API

In [21]:
from lettucedetect.models.inference import HallucinationDetector

print("LettuceDetect imported successfully.")

LettuceDetect imported successfully.


Cell 18 — Load the existing pretrained BASE model

In [22]:
LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

detector = HallucinationDetector(
    method="transformer",
    model_path=LETTUCE_MODEL
)

print("Loaded:", LETTUCE_MODEL)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loaded: KRLabsOrg/lettucedect-base-modernbert-en-v1


Cell 19 — Run LettuceDetect on ONE medical RAG example

In [23]:
sample = lettuce_df.iloc[0]

sample_context = sample["context"]
sample_question = sample["question"]
sample_answer = sample["answer"]

predictions = detector.predict(
    context=[sample_context],
    question=sample_question,
    answer=sample_answer,
    output_format="spans"
)

print("Question:")
print(sample_question)

print("\nAnswer:")
print(sample_answer)

print("\nLettuceDetect predictions:")
print(predictions)

/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/inference.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, device=self.device)
W0917 23:59:06.752000 58 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Question:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

Answer:
The available evidence is insufficient to answer this question.

LettuceDetect predictions:
[]


Cell 20 — Also inspect token-level output

In [24]:
token_predictions = detector.predict(
    context=[sample_context],
    question=sample_question,
    answer=sample_answer,
    output_format="tokens"
)

print("Number of token predictions:", len(token_predictions))

for item in token_predictions[:30]:
    print(item)

Number of token predictions: 11
{'token': 'The', 'pred': 0, 'prob': 0.02611379884183407}
{'token': ' available', 'pred': 0, 'prob': 0.016907986253499985}
{'token': ' evidence', 'pred': 0, 'prob': 0.021537702530622482}
{'token': ' is', 'pred': 0, 'prob': 0.014915622770786285}
{'token': ' insufficient', 'pred': 0, 'prob': 0.01355630811303854}
{'token': ' to', 'pred': 0, 'prob': 0.026834342628717422}
{'token': ' answer', 'pred': 0, 'prob': 0.0279354490339756}
{'token': ' this', 'pred': 0, 'prob': 0.026235096156597137}
{'token': ' question', 'pred': 0, 'prob': 0.039701949805021286}
{'token': '.', 'pred': 0, 'prob': 0.1179070845246315}
{'token': '[SEP]', 'pred': 0, 'prob': 0.0005201817839406431}


/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/inference.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, device=self.device)


Cell 21 — Create a baseline prediction function

In [25]:
def run_lettucedetect_baseline(row):
    predictions = detector.predict(
        context=[row["context"]],
        question=row["question"],
        answer=row["answer"],
        output_format="spans"
    )

    token_predictions = detector.predict(
        context=[row["context"]],
        question=row["question"],
        answer=row["answer"],
        output_format="tokens"
    )

    return predictions, token_predictions


print("Baseline prediction function ready.")

Baseline prediction function ready.


Cell 22 — Test the function on one example

In [26]:
test_spans, test_tokens = run_lettucedetect_baseline(lettuce_df.iloc[0])

print("Span predictions:")
print(test_spans)

print("\nToken predictions:")
for token in test_tokens:
    print(token)

Span predictions:
[]

Token predictions:
{'token': 'The', 'pred': 0, 'prob': 0.02611379884183407}
{'token': ' available', 'pred': 0, 'prob': 0.016907986253499985}
{'token': ' evidence', 'pred': 0, 'prob': 0.021537702530622482}
{'token': ' is', 'pred': 0, 'prob': 0.014915622770786285}
{'token': ' insufficient', 'pred': 0, 'prob': 0.01355630811303854}
{'token': ' to', 'pred': 0, 'prob': 0.026834342628717422}
{'token': ' answer', 'pred': 0, 'prob': 0.0279354490339756}
{'token': ' this', 'pred': 0, 'prob': 0.026235096156597137}
{'token': ' question', 'pred': 0, 'prob': 0.039701949805021286}
{'token': '.', 'pred': 0, 'prob': 0.1179070845246315}
{'token': '[SEP]', 'pred': 0, 'prob': 0.0005201817839406431}


Cell 23 — Run baseline on all 200 examples

In [27]:
baseline_spans = []
baseline_tokens = []

for i, row in lettuce_df.iterrows():
    try:
        spans, tokens = run_lettucedetect_baseline(row)

        baseline_spans.append(spans)
        baseline_tokens.append(tokens)

    except Exception as e:
        print(f"Error at row {i}, test_id={row['test_id']}: {e}")
        baseline_spans.append([])
        baseline_tokens.append([])

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(lettuce_df)}")

print("\nBaseline inference complete.")

Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200

Baseline inference complete.


Cell 24 — Attach predictions to the dataframe

In [28]:
baseline_df = lettuce_df.copy()

baseline_df["lettucedetect_spans"] = baseline_spans
baseline_df["lettucedetect_tokens"] = baseline_tokens

print("Shape:", baseline_df.shape)

display(
    baseline_df[
        [
            "test_id",
            "question",
            "answer",
            "lettucedetect_spans"
        ]
    ].head(10)
)

Shape: (200, 8)


,test_id,question,answer,lettucedetect_spans
0,521,Pap smears with glandular cell abnormalities: ...,The available evidence is insufficient to answ...,[]
1,737,Are patients with Werlhof's disease at increas...,"Based on the provided evidence, there is no di...",[]
2,740,Is there a connection between sublingual varic...,"Based on the provided evidence, there appears ...",[]
3,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...","[{'start': 91, 'end': 116, 'confidence': 0.940..."
4,411,The promise of specialty pharmaceuticals: are ...,"Based on the evidence provided, it appears tha...",[]
5,678,The colour of pain: can patients use colour to...,"Based on the provided evidence, there is no di...",[]
6,626,Is fluoroscopy essential for retrieval of lowe...,"No, fluoroscopy is not essential for the retri...","[{'start': 98, 'end': 100, 'confidence': 0.512..."
7,513,Increased neutrophil migratory activity after ...,The available evidence is insufficient to answ...,[]
8,859,Does the SCL 90-R obsessive-compulsive dimensi...,The available evidence is insufficient to answ...,[]
9,136,Can transcranial direct current stimulation be...,The available evidence is insufficient to answ...,[]


Cell 25 — Count how many examples have detected hallucination spans

In [29]:
num_with_spans = baseline_df["lettucedetect_spans"].apply(
    lambda x: len(x) > 0
).sum()

num_without_spans = len(baseline_df) - num_with_spans

print("Total examples:", len(baseline_df))
print("Examples with detected hallucination spans:", num_with_spans)
print("Examples with no detected hallucination spans:", num_without_spans)

Total examples: 200
Examples with detected hallucination spans: 89
Examples with no detected hallucination spans: 111


Cell 26 — Save the original LettuceDetect baseline

In [30]:
baseline_output_file = "/kaggle/working/lettucedetect_original_baseline.csv"

baseline_df.to_csv(
    baseline_output_file,
    index=False
)

print("Saved:", baseline_output_file)

Saved: /kaggle/working/lettucedetect_original_baseline.csv


Cell 27 — Inspect the LettuceDetect RAGTruth dataset format

In [31]:
# Cell 27 — Inspect LettuceDetect's dataset implementation

%cd /kaggle/working/Hallucination-LettuceDetect

print("=" * 80)
print("ragtruth.py")
print("=" * 80)

with open("lettucedetect/datasets/ragtruth.py", "r", encoding="utf-8") as f:
    ragtruth_code = f.read()

print(ragtruth_code)

print("\n" + "=" * 80)
print("preprocess_ragtruth.py")
print("=" * 80)

with open(
    "lettucedetect/preprocess/preprocess_ragtruth.py",
    "r",
    encoding="utf-8"
) as f:
    preprocess_code = f.read()

print(preprocess_code)

/kaggle/working/Hallucination-LettuceDetect
ragtruth.py
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

from lettucedetect.preprocess.preprocess_ragtruth import RagTruthSample


class RagTruthDataset(Dataset):
    """Dataset for RAG truth data."""

    def __init__(
        self,
        samples: list[RagTruthSample],
        tokenizer: AutoTokenizer,
        max_length: int = 4096,
    ):
        """Initialize the dataset.

        :param samples: List of RagTruthSample objects.
        :param tokenizer: Tokenizer to use for encoding the data.
        :param max_length: Maximum length of the input sequence.
        """
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.samples)

    @classmethod
    def prepare_tokenized_input(
        cls,
        tokenizer: AutoTokenizer,
        context: str,
        answer: str,
        max_length: in

Cell 28 — Create the medical annotation template

In [32]:
# Cell 28 — Create a template for manual medical hallucination annotation

annotation_df = baseline_df[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "final_decision"
    ]
].copy()

# These columns will contain our manual annotations.
annotation_df["overall_label"] = ""
annotation_df["hallucination_spans"] = ""

print("Annotation dataset shape:", annotation_df.shape)

display(
    annotation_df[
        [
            "test_id",
            "question",
            "answer",
            "overall_label",
            "hallucination_spans"
        ]
    ].head(10)
)

Annotation dataset shape: (200, 8)


,test_id,question,answer,overall_label,hallucination_spans
0,521,Pap smears with glandular cell abnormalities: ...,The available evidence is insufficient to answ...,,
1,737,Are patients with Werlhof's disease at increas...,"Based on the provided evidence, there is no di...",,
2,740,Is there a connection between sublingual varic...,"Based on the provided evidence, there appears ...",,
3,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...",,
4,411,The promise of specialty pharmaceuticals: are ...,"Based on the evidence provided, it appears tha...",,
5,678,The colour of pain: can patients use colour to...,"Based on the provided evidence, there is no di...",,
6,626,Is fluoroscopy essential for retrieval of lowe...,"No, fluoroscopy is not essential for the retri...",,
7,513,Increased neutrophil migratory activity after ...,The available evidence is insufficient to answ...,,
8,859,Does the SCL 90-R obsessive-compulsive dimensi...,The available evidence is insufficient to answ...,,
9,136,Can transcranial direct current stimulation be...,The available evidence is insufficient to answ...,,


Cell 29 — Create an annotation helper

In [33]:
# Cell 29 — Helper for reviewing one medical RAG answer

def show_annotation_example(index):
    row = annotation_df.iloc[index]

    print("=" * 100)
    print(f"INDEX: {index}    TEST ID: {row['test_id']}")
    print("=" * 100)

    print("\nQUESTION:")
    print(row["question"])

    print("\n" + "-" * 100)

    print("RETRIEVED CONTEXT:")
    print(row["context"])

    print("\n" + "-" * 100)

    print("GENERATED ANSWER:")
    print(row["answer"])

    print("\n" + "-" * 100)

    print("REFERENCE ANSWER:")
    print(row["reference_answer"])

    print("\n" + "=" * 100)


# Show the first example
show_annotation_example(0)

INDEX: 0    TEST ID: 521

QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the dete

Cell 30 — Add an annotation function

In [34]:
# Cell 30 — Function for recording manual annotations

def annotate_example(index, overall_label, hallucination_spans=None):
    if hallucination_spans is None:
        hallucination_spans = []

    annotation_df.loc[index, "overall_label"] = overall_label
    annotation_df.loc[index, "hallucination_spans"] = json.dumps(
        hallucination_spans,
        ensure_ascii=False
    )

    print(f"Annotated index {index} (test_id={annotation_df.loc[index, 'test_id']})")
    print("Overall label:", overall_label)
    print("Hallucination spans:", hallucination_spans)


# Record the annotation for the example we just reviewed.
annotate_example(
    index=0,
    overall_label="appropriate_abstention",
    hallucination_spans=[]
)

Annotated index 0 (test_id=521)
Overall label: appropriate_abstention
Hallucination spans: []


Cell 31 — Automatically convert hallucinated text into character spans

In [35]:
# Cell 31 — Convert hallucinated text into LettuceDetect character spans

def create_hallucination_span(answer, hallucinated_text):
    """
    Find an exact hallucinated phrase inside the answer
    and return its character-level span.
    """

    if not hallucinated_text:
        return []

    start = answer.find(hallucinated_text)

    if start == -1:
        raise ValueError(
            f"Could not find this text in the answer:\n{hallucinated_text}"
        )

    end = start + len(hallucinated_text)

    return [
        {
            "start": start,
            "end": end,
            "label": "hallucinated"
        }
    ]


# Test the helper
example_answer = "The treatment reduced mortality by 25%."
example_hallucination = "25%"

example_spans = create_hallucination_span(
    example_answer,
    example_hallucination
)

print("Answer:", example_answer)
print("Hallucinated text:", example_hallucination)
print("Generated span:", example_spans)

Answer: The treatment reduced mortality by 25%.
Hallucinated text: 25%
Generated span: [{'start': 35, 'end': 38, 'label': 'hallucinated'}]


Cell 32 — Create a safer annotation helper

In [36]:
# Cell 32 — Annotate an answer using the exact hallucinated text

def annotate_with_text(index, overall_label, hallucinated_texts=None):
    """
    Annotate one answer by providing the exact hallucinated text.
    
    hallucinated_texts should be a list of exact phrases from the answer.
    """

    if hallucinated_texts is None:
        hallucinated_texts = []

    answer = annotation_df.loc[index, "answer"]

    spans = []

    for text in hallucinated_texts:
        start = answer.find(text)

        if start == -1:
            raise ValueError(
                f"Could not find this text in the answer:\n{text}"
            )

        end = start + len(text)

        spans.append({
            "start": start,
            "end": end,
            "label": "hallucinated"
        })

    annotation_df.loc[index, "overall_label"] = overall_label
    annotation_df.loc[index, "hallucination_spans"] = json.dumps(
        spans,
        ensure_ascii=False
    )

    print(f"Annotated index {index} (test_id={annotation_df.loc[index, 'test_id']})")
    print("Overall label:", overall_label)
    print("Hallucination spans:", spans)

Cell 33 — Verify the annotation we already made

In [37]:
# Cell 33 — Verify annotation for test_id 521

print("Test ID:", annotation_df.loc[0, "test_id"])
print("Overall label:", annotation_df.loc[0, "overall_label"])
print("Hallucination spans:", annotation_df.loc[0, "hallucination_spans"])

Test ID: 521
Overall label: appropriate_abstention
Hallucination spans: []


Cell 34 — Save the annotation dataset

In [38]:
# Cell 34 — Save current annotation progress

annotation_file = "/kaggle/working/medical_hallucination_annotations.csv"

annotation_df.to_csv(
    annotation_file,
    index=False
)

print("Saved:", annotation_file)
print("Rows:", len(annotation_df))
print("Annotated rows:", (annotation_df["overall_label"] != "").sum())

Saved: /kaggle/working/medical_hallucination_annotations.csv
Rows: 200
Annotated rows: 1


Cell 35 — Create fixed annotation splits

In [39]:
# Cell 35 — Create fixed train / validation / test splits

from sklearn.model_selection import train_test_split

# Keep the split fixed and reproducible.
annotation_train, annotation_test = train_test_split(
    annotation_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

annotation_train = annotation_train.reset_index(drop=True)
annotation_test = annotation_test.reset_index(drop=True)

# Split the 80% training portion into:
# 70% training and 10% validation of the original 200 examples.
annotation_train, annotation_val = train_test_split(
    annotation_train,
    test_size=0.125,
    random_state=42,
    shuffle=True
)

annotation_train = annotation_train.reset_index(drop=True)
annotation_val = annotation_val.reset_index(drop=True)

print("Training examples:", len(annotation_train))
print("Validation examples:", len(annotation_val))
print("Final test examples:", len(annotation_test))

print("\nTotal:", len(annotation_train) + len(annotation_val) + len(annotation_test))

Training examples: 140
Validation examples: 20
Final test examples: 40

Total: 200


Cell 36 — Record the fixed split IDs

In [40]:
# Cell 36 — Record and inspect the fixed split assignments

annotation_train["split"] = "train"
annotation_val["split"] = "validation"
annotation_test["split"] = "test"

print("TRAIN test_ids:")
print(annotation_train["test_id"].tolist())

print("\nVALIDATION test_ids:")
print(annotation_val["test_id"].tolist())

print("\nFINAL TEST test_ids:")
print(annotation_test["test_id"].tolist())

TRAIN test_ids:
[260, 110, 678, 998, 445, 78, 644, 884, 753, 679, 800, 650, 411, 820, 901, 526, 548, 924, 810, 314, 312, 697, 462, 660, 826, 101, 168, 139, 599, 542, 973, 989, 893, 527, 947, 749, 604, 199, 959, 39, 713, 656, 649, 570, 174, 985, 86, 25, 67, 307, 590, 670, 917, 829, 845, 382, 740, 327, 294, 636, 849, 902, 823, 941, 938, 554, 595, 687, 549, 621, 429, 96, 363, 883, 23, 721, 677, 714, 942, 499, 408, 292, 209, 54, 816, 328, 299, 534, 767, 319, 365, 298, 689, 254, 601, 309, 158, 256, 59, 281, 859, 583, 892, 584, 210, 63, 716, 613, 737, 10, 914, 76, 528, 371, 522, 986, 948, 66, 88, 626, 493, 221, 72, 425, 208, 351, 439, 837, 995, 479, 244, 107, 529, 936, 310, 521, 513, 978, 787, 137]

VALIDATION test_ids:
[370, 451, 261, 213, 974, 120, 802, 247, 494, 321, 764, 235, 741, 811, 198, 879, 332, 184, 578, 44]

FINAL TEST test_ids:
[436, 899, 346, 60, 867, 432, 543, 361, 486, 277, 662, 377, 668, 589, 355, 259, 617, 977, 275, 70, 237, 280, 822, 218, 692, 296, 318, 866, 559, 265, 136, 

Cell 37 — Create a compact annotation queue

In [41]:
# Cell 37 — Create compact annotation queues

train_queue = annotation_train[
    ["test_id", "question", "answer", "overall_label", "hallucination_spans"]
].copy()

val_queue = annotation_val[
    ["test_id", "question", "answer", "overall_label", "hallucination_spans"]
].copy()

test_queue = annotation_test[
    ["test_id", "question", "answer", "overall_label", "hallucination_spans"]
].copy()

print("Training queue:", len(train_queue))
print("Validation queue:", len(val_queue))
print("Final test queue:", len(test_queue))

print("\nCurrently annotated:")
print("Training:", (train_queue["overall_label"] != "").sum())
print("Validation:", (val_queue["overall_label"] != "").sum())
print("Final test:", (test_queue["overall_label"] != "").sum())

Training queue: 140
Validation queue: 20
Final test queue: 40

Currently annotated:
Training: 1
Validation: 0
Final test: 0


Cell 38 — Reset annotations and preserve the examples we've already reviewed

In [42]:
# Cell 38 — Reset and preserve verified annotations

# Start from the original 200-example annotation dataframe
annotation_df = baseline_df[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "final_decision"
    ]
].copy()

# Annotation fields
annotation_df["overall_label"] = ""
annotation_df["hallucination_spans"] = ""
annotation_df["annotator_notes"] = ""

# Preserve the examples we already manually reviewed

verified_annotations = {
    521: {
        "overall_label": "appropriate_abstention",
        "hallucination_spans": [],
        "notes": "Retrieved evidence was insufficient; abstention was appropriate."
    },

    260: {
        "overall_label": "appropriate_abstention",
        "hallucination_spans": [],
        "notes": "Retrieved evidence did not address the question; abstention was appropriate."
    },

    110: {
        "overall_label": "appropriate_abstention",
        "hallucination_spans": [],
        "notes": "Retrieved evidence did not address the mediation question; abstention was appropriate."
    },

    678: {
        "overall_label": "appropriate_abstention",
        "hallucination_spans": [],
        "notes": "Retrieved evidence did not directly support the question; abstention was appropriate."
    },

    998: {
        "overall_label": "partially_supported",
        "hallucination_spans": [
            {
                "start": 447,
                "end": 480,
                "label": "hallucinated"
            }
        ],
        "notes": "Main conclusion was appropriately cautious, but 'in relation to blood transfusions' was unsupported by retrieved context."
    },

    445: {
        "overall_label": "partially_supported",
        "hallucination_spans": [
            {
                "start": 377,
                "end": 502,
                "label": "hallucinated"
            }
        ],
        "notes": "Answer correctly noted lack of direct evidence, but included an unsupported definition/explanation of secular growth acceleration."
    }
}

for test_id, annotation in verified_annotations.items():

    idx = annotation_df.index[
        annotation_df["test_id"] == test_id
    ][0]

    annotation_df.loc[idx, "overall_label"] = annotation["overall_label"]

    annotation_df.loc[idx, "hallucination_spans"] = json.dumps(
        annotation["hallucination_spans"],
        ensure_ascii=False
    )

    annotation_df.loc[idx, "annotator_notes"] = annotation["notes"]


print("Annotation dataframe reset successfully.")

print("Total examples:", len(annotation_df))

print(
    "Already manually reviewed:",
    (annotation_df["overall_label"] != "").sum()
)

print(
    "Remaining:",
    (annotation_df["overall_label"] == "").sum()
)

display(
    annotation_df[
        [
            "test_id",
            "overall_label",
            "hallucination_spans"
        ]
    ][annotation_df["overall_label"] != ""]
)

Annotation dataframe reset successfully.
Total examples: 200
Already manually reviewed: 6
Remaining: 194


,test_id,overall_label,hallucination_spans
0,521,appropriate_abstention,[]
5,678,appropriate_abstention,[]
86,998,partially_supported,"[{""start"": 447, ""end"": 480, ""label"": ""hallucin..."
123,260,appropriate_abstention,[]
127,445,partially_supported,"[{""start"": 377, ""end"": 502, ""label"": ""hallucin..."
172,110,appropriate_abstention,[]


Cell 39 — Generate model-assisted annotations

In [43]:
# Cell 39 — Model-assisted annotation of all 200 examples

import json
import re
import torch

# Load the same Qwen model used for RAG generation
from transformers import AutoTokenizer, AutoModelForCausalLM

ANNOTATION_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

annotation_tokenizer = AutoTokenizer.from_pretrained(
    ANNOTATION_MODEL
)

annotation_model = AutoModelForCausalLM.from_pretrained(
    ANNOTATION_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Annotation model loaded.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Annotation model loaded.


Cell 40 — Annotation function

In [44]:
# Cell 40 — Generate a proposed label and hallucinated spans

def propose_annotation(question, context, answer):
    
    prompt = f"""
You are assisting with annotation of hallucinations in a medical RAG system.

Your task is to determine whether the GENERATED ANSWER is supported by
the RETRIEVED CONTEXT.

IMPORTANT:
- Judge the generated answer ONLY against the retrieved context.
- Do NOT use outside medical knowledge.
- Do NOT treat disagreement with the reference answer as hallucination.
- If the retrieved context does not contain enough information and the
  answer appropriately says that evidence is insufficient, label it
  appropriate_abstention.
- If part of the answer is supported but another specific claim is
  unsupported, label it partially_supported.
- If the answer makes substantive unsupported claims, label it hallucinated.
- If all meaningful claims are supported, label it supported.

Allowed labels:
1. supported
2. partially_supported
3. hallucinated
4. appropriate_abstention

For partially_supported or hallucinated:
identify the EXACT unsupported text from the generated answer.

Return ONLY valid JSON in this format:

{{
  "overall_label": "supported",
  "hallucinated_texts": [],
  "reason": "brief explanation"
}}

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}
""".strip()

    messages = [
        {
            "role": "system",
            "content": "You are a careful annotation assistant. Return only JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = annotation_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = annotation_tokenizer(
        text,
        return_tensors="pt"
    ).to(annotation_model.device)

    with torch.no_grad():
        outputs = annotation_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            temperature=None
        )

    generated = annotation_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Extract JSON
    match = re.search(
        r"\{.*\}",
        generated,
        re.DOTALL
    )

    if not match:
        return {
            "overall_label": "",
            "hallucinated_texts": [],
            "reason": "Could not parse model output.",
            "raw_output": generated
        }

    try:
        result = json.loads(match.group(0))

    except json.JSONDecodeError:
        return {
            "overall_label": "",
            "hallucinated_texts": [],
            "reason": "Invalid JSON returned by model.",
            "raw_output": generated
        }

    return result

Cell 41 — Run it over all 200

In [45]:
# Cell 41 — Generate proposed annotations

proposed_annotations = []

for i, row in annotation_df.iterrows():

    # Don't overwrite the six annotations we already reviewed
    if row["overall_label"] != "":
        proposed_annotations.append({
            "test_id": row["test_id"],
            "overall_label": row["overall_label"],
            "hallucinated_texts": [],
            "reason": row["annotator_notes"],
            "source": "manual"
        })
        continue

    result = propose_annotation(
        question=row["question"],
        context=row["context"],
        answer=row["answer"]
    )

    proposed_annotations.append({
        "test_id": row["test_id"],
        "overall_label": result.get("overall_label", ""),
        "hallucinated_texts": result.get("hallucinated_texts", []),
        "reason": result.get("reason", ""),
        "source": "model_proposed"
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/200")


proposed_df = pd.DataFrame(proposed_annotations)

print("\nProposed annotation generation complete.")
print(proposed_df["overall_label"].value_counts(dropna=False))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200

Proposed annotation generation complete.
overall_label
appropriate_abstention    142
supported                  56
partially_supported         2
Name: count, dtype: int64


Cell 42 — Inspect the proposed annotations

In [46]:
# Cell 42 — Inspect proposed annotation distribution

print("Total examples:", len(proposed_df))

print("\nProposed label distribution:")
print(proposed_df["overall_label"].value_counts(dropna=False))

print("\nExamples by proposed label:")
for label in [
    "hallucinated",
    "partially_supported",
    "supported",
    "appropriate_abstention"
]:
    count = (proposed_df["overall_label"] == label).sum()
    print(f"{label}: {count}")

Total examples: 200

Proposed label distribution:
overall_label
appropriate_abstention    142
supported                  56
partially_supported         2
Name: count, dtype: int64

Examples by proposed label:
hallucinated: 0
partially_supported: 2
supported: 56
appropriate_abstention: 142


Cell 43 — Find potentially suspicious annotations

In [47]:
# Cell 43 — Inspect potentially suspicious model proposals

suspicious = proposed_df[
    proposed_df["overall_label"].isin([
        "hallucinated",
        "partially_supported"
    ])
].copy()

print("Suspicious proposals:", len(suspicious))

display(suspicious)

Suspicious proposals: 2


,test_id,overall_label,hallucinated_texts,reason,source
86,998,partially_supported,[],"Main conclusion was appropriately cautious, bu...",manual
127,445,partially_supported,[],Answer correctly noted lack of direct evidence...,manual


Cell 44 — Create a review sample of the 56 "supported" answers

In [48]:
# Cell 44 — Inspect proposed "supported" examples

supported_proposals = proposed_df[
    proposed_df["overall_label"] == "supported"
].copy()

print("Proposed supported examples:", len(supported_proposals))

display(
    supported_proposals[
        ["test_id", "overall_label", "reason"]
    ].head(20)
)

Proposed supported examples: 56


,test_id,overall_label,reason
2,740,supported,The generated answer acknowledges the lack of ...
6,626,supported,The generated answer accurately reflects the f...
11,76,supported,The generated answer supports the statement ab...
13,973,supported,The generated answer accurately reflects the i...
15,899,supported,The generated answer acknowledges the lack of ...
16,280,supported,The generated answer supports the statement th...
18,761,supported,The generated answer supports the statement th...
25,235,supported,The generated answer acknowledges the general ...
28,902,supported,The retrieved context does not contain any sta...
29,947,supported,The generated answer acknowledges the controve...


Cell 45 — Check the proposed abstentions

In [49]:
# Cell 45 — Inspect proposed abstention examples

abstention_proposals = proposed_df[
    proposed_df["overall_label"] == "appropriate_abstention"
].copy()

print("Proposed appropriate abstentions:", len(abstention_proposals))

display(
    abstention_proposals[
        ["test_id", "overall_label", "reason"]
    ].head(20)
)

Proposed appropriate abstentions: 142


,test_id,overall_label,reason
0,521,appropriate_abstention,Retrieved evidence was insufficient; abstentio...
1,737,appropriate_abstention,The retrieved context does not provide any rel...
3,660,appropriate_abstention,The generated answer acknowledges the lack of ...
4,411,appropriate_abstention,The retrieved context does not contain enough ...
5,678,appropriate_abstention,Retrieved evidence did not directly support th...
7,513,appropriate_abstention,The retrieved context does not provide suffici...
8,859,appropriate_abstention,The retrieved context does not provide suffici...
9,136,appropriate_abstention,The retrieved context does not provide any rel...
10,811,appropriate_abstention,The retrieved context does not provide suffici...
12,636,appropriate_abstention,The retrieved context does not provide direct ...


Cell 46 — Identify answers that LettuceDetect flagged

In [50]:
# Cell 46 — Find examples where original LettuceDetect detected spans

baseline_screening = baseline_df[
    [
        "test_id",
        "question",
        "answer",
        "lettucedetect_spans"
    ]
].copy()

baseline_screening["detected"] = baseline_screening[
    "lettucedetect_spans"
].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

flagged_df = baseline_screening[
    baseline_screening["detected"]
].copy()

print("Total examples:", len(baseline_screening))
print("Flagged by original LettuceDetect:", len(flagged_df))
print("Not flagged:", len(baseline_screening) - len(flagged_df))

display(
    flagged_df[
        [
            "test_id",
            "answer",
            "lettucedetect_spans"
        ]
    ].head(20)
)

Total examples: 200
Flagged by original LettuceDetect: 89
Not flagged: 111


,test_id,answer,lettucedetect_spans
3,660,"Based on Evidence 1, which states that the rad...","[{'start': 91, 'end': 116, 'confidence': 0.940..."
6,626,"No, fluoroscopy is not essential for the retri...","[{'start': 98, 'end': 100, 'confidence': 0.512..."
12,636,The available evidence is insufficient to answ...,"[{'start': 517, 'end': 556, 'confidence': 0.64..."
16,280,"Based on the provided evidence, there is no di...","[{'start': 554, 'end': 566, 'confidence': 0.54..."
18,761,"Based on the evidence provided, surgical excis...","[{'start': 480, 'end': 681, 'confidence': 0.96..."
21,174,The available evidence is insufficient to answ...,"[{'start': 490, 'end': 493, 'confidence': 0.68..."
30,346,"Yes, multiple SNP testing in BRCA2 and BRCA1 f...","[{'start': 624, 'end': 791, 'confidence': 0.99..."
35,198,The available evidence is insufficient to answ...,"[{'start': 318, 'end': 448, 'confidence': 0.99..."
36,687,"Based on the provided evidence, it appears tha...","[{'start': 953, 'end': 962, 'confidence': 0.54..."
40,328,"No, myomas do not really grow significantly du...","[{'start': 96, 'end': 204, 'confidence': 0.987..."


Then Cell 47 — See what those 89 look like

In [51]:
# Cell 47 — Inspect the 89 detector-flagged examples

flagged_ids = flagged_df["test_id"].tolist()

flagged_review = annotation_df[
    annotation_df["test_id"].isin(flagged_ids)
].copy()

flagged_review = flagged_review.merge(
    flagged_df[
        ["test_id", "lettucedetect_spans"]
    ],
    on="test_id",
    how="left"
)

print("Flagged examples available for review:", len(flagged_review))

display(
    flagged_review[
        [
            "test_id",
            "question",
            "answer",
            "lettucedetect_spans"
        ]
    ].head(20)
)

Flagged examples available for review: 89


,test_id,question,answer,lettucedetect_spans
0,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...","[{'start': 91, 'end': 116, 'confidence': 0.940..."
1,626,Is fluoroscopy essential for retrieval of lowe...,"No, fluoroscopy is not essential for the retri...","[{'start': 98, 'end': 100, 'confidence': 0.512..."
2,636,Are stroke patients' reports of home blood pre...,The available evidence is insufficient to answ...,"[{'start': 517, 'end': 556, 'confidence': 0.64..."
3,280,Does high blood pressure reduce the risk of ch...,"Based on the provided evidence, there is no di...","[{'start': 554, 'end': 566, 'confidence': 0.54..."
4,761,Ovarian torsion in children: is oophorectomy n...,"Based on the evidence provided, surgical excis...","[{'start': 480, 'end': 681, 'confidence': 0.96..."
5,174,Quality of life in lung cancer patients: does ...,The available evidence is insufficient to answ...,"[{'start': 490, 'end': 493, 'confidence': 0.68..."
6,346,Can multiple SNP testing in BRCA2 and BRCA1 fe...,"Yes, multiple SNP testing in BRCA2 and BRCA1 f...","[{'start': 624, 'end': 791, 'confidence': 0.99..."
7,198,Transesophageal echocardiographic assessment o...,The available evidence is insufficient to answ...,"[{'start': 318, 'end': 448, 'confidence': 0.99..."
8,687,Delayed diagnosis of anorectal malformations: ...,"Based on the provided evidence, it appears tha...","[{'start': 953, 'end': 962, 'confidence': 0.54..."
9,328,Volume change of uterine myomas during pregnan...,"No, myomas do not really grow significantly du...","[{'start': 96, 'end': 204, 'confidence': 0.987..."


Cell 48 — Build the final proposal table

In [52]:
# Cell 48 — Build a clean model-assisted annotation table

# Start from the 200 examples
review_df = annotation_df[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "final_decision",
        "overall_label",
        "hallucination_spans",
        "annotator_notes"
    ]
].copy()

# Add model proposals
review_df = review_df.merge(
    proposed_df[
        [
            "test_id",
            "overall_label",
            "hallucinated_texts",
            "reason",
            "source"
        ]
    ].rename(
        columns={
            "overall_label": "proposed_label",
            "reason": "proposal_reason"
        }
    ),
    on="test_id",
    how="left"
)

# Restore the manually verified labels
review_df.loc[
    review_df["overall_label"] != "",
    "proposed_label"
] = review_df.loc[
    review_df["overall_label"] != "",
    "overall_label"
]

# Restore exact manually verified spans
review_df.loc[
    review_df["hallucination_spans"] != "",
    "hallucinated_texts"
] = review_df.loc[
    review_df["hallucination_spans"] != "",
    "hallucination_spans"
]

print("Review dataframe:", review_df.shape)

print("\nProposed labels:")
print(review_df["proposed_label"].value_counts(dropna=False))

Review dataframe: (200, 13)

Proposed labels:
proposed_label
appropriate_abstention    142
supported                  56
partially_supported         2
Name: count, dtype: int64


Cell 49 — Find potentially problematic proposals

In [53]:
# Cell 49 — Identify potentially problematic proposals

def answer_has_abstention(answer):
    answer = str(answer).lower()

    phrases = [
        "evidence is insufficient",
        "available evidence is insufficient",
        "does not provide enough evidence",
        "does not provide sufficient evidence",
        "cannot be determined from the evidence",
        "not enough information"
    ]

    return any(p in answer for p in phrases)


review_df["answer_abstains"] = review_df["answer"].apply(
    answer_has_abstention
)

# Cases where the model says "supported" despite the answer explicitly
# claiming evidence is insufficient.
possible_conflicts = review_df[
    (
        (review_df["proposed_label"] == "supported") &
        (review_df["answer_abstains"])
    )
].copy()

print(
    "Supported + abstention-style answer:",
    len(possible_conflicts)
)

display(
    possible_conflicts[
        [
            "test_id",
            "question",
            "answer",
            "proposed_label",
            "proposal_reason"
        ]
    ].head(30)
)

Supported + abstention-style answer: 4


,test_id,question,answer,proposed_label,proposal_reason
44,299,Can the prognosis of polymyalgia rheumatica be...,The available evidence is insufficient to pred...,supported,The generated answer accurately reflects the l...
45,277,Does open access publishing increase the impac...,"Based on the provided evidence, there is limit...",supported,The generated answer acknowledges the limitati...
109,917,Does radiotherapy around the time of pregnancy...,"Based on the provided evidence, there is no di...",supported,The generated answer acknowledges the lack of ...
190,948,Is resected stomach volume related to weight l...,The available evidence is insufficient to answ...,supported,The generated answer acknowledges the lack of ...


Cell 50 — Save everything before we do anything else

In [54]:
# Cell 50 — Save the complete annotation workspace

review_file = "/kaggle/working/medical_hallucination_annotation_review.csv"

review_df.to_csv(
    review_file,
    index=False
)

print("Saved:", review_file)
print("Rows:", len(review_df))
print("Columns:", review_df.columns.tolist())

Saved: /kaggle/working/medical_hallucination_annotation_review.csv
Rows: 200
Columns: ['test_id', 'question', 'context', 'answer', 'reference_answer', 'final_decision', 'overall_label', 'hallucination_spans', 'annotator_notes', 'proposed_label', 'hallucinated_texts', 'proposal_reason', 'source', 'answer_abstains']


Cell 51 — Correct obvious abstention proposals

In [55]:
# Cell 51 — Correct obvious abstention-label conflicts

conflict_ids = possible_conflicts["test_id"].tolist()

print("Cases to correct:", conflict_ids)

for test_id in conflict_ids:
    idx = review_df.index[
        review_df["test_id"] == test_id
    ][0]

    # Do not overwrite manually verified annotations
    if review_df.loc[idx, "source"] != "manual":
        review_df.loc[idx, "proposed_label"] = "appropriate_abstention"

        review_df.loc[idx, "proposal_reason"] = (
            "Answer explicitly indicates that the retrieved evidence "
            "is insufficient; corrected from supported to "
            "appropriate_abstention."
        )

print("\nUpdated label distribution:")
print(review_df["proposed_label"].value_counts())

Cases to correct: [299, 277, 917, 948]

Updated label distribution:
proposed_label
appropriate_abstention    146
supported                  52
partially_supported         2
Name: count, dtype: int64


Cell 52 — Check whether any generated answer contains unsupported-looking claims

In [56]:
# Cell 52 — Find abstention answers with substantial explanations

abstention_df = review_df[
    review_df["proposed_label"] == "appropriate_abstention"
].copy()

def has_explanation_after_abstention(answer):
    text = str(answer).lower()

    markers = [
        "explanation:",
        "however,",
        "instead,",
        "therefore,",
        "this suggests",
        "this indicates",
        "this means"
    ]

    return any(marker in text for marker in markers)


abstention_df["has_explanation"] = abstention_df[
    "answer"
].apply(has_explanation_after_abstention)

explanation_cases = abstention_df[
    abstention_df["has_explanation"]
].copy()

print(
    "Abstention answers containing explanatory language:",
    len(explanation_cases)
)

display(
    explanation_cases[
        [
            "test_id",
            "question",
            "answer",
            "proposed_label",
            "proposal_reason"
        ]
    ].head(30)
)

Abstention answers containing explanatory language: 128


,test_id,question,answer,proposed_label,proposal_reason
1,737,Are patients with Werlhof's disease at increas...,"Based on the provided evidence, there is no di...",appropriate_abstention,The retrieved context does not provide any rel...
3,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...",appropriate_abstention,The generated answer acknowledges the lack of ...
4,411,The promise of specialty pharmaceuticals: are ...,"Based on the evidence provided, it appears tha...",appropriate_abstention,The retrieved context does not contain enough ...
5,678,The colour of pain: can patients use colour to...,"Based on the provided evidence, there is no di...",appropriate_abstention,Retrieved evidence did not directly support th...
7,513,Increased neutrophil migratory activity after ...,The available evidence is insufficient to answ...,appropriate_abstention,The retrieved context does not provide suffici...
8,859,Does the SCL 90-R obsessive-compulsive dimensi...,The available evidence is insufficient to answ...,appropriate_abstention,The retrieved context does not provide suffici...
12,636,Are stroke patients' reports of home blood pre...,The available evidence is insufficient to answ...,appropriate_abstention,The retrieved context does not provide direct ...
14,938,Is leptin involved in phagocytic NADPH oxidase...,The available evidence is insufficient to answ...,appropriate_abstention,The retrieved context does not provide suffici...
17,883,"Do the changes in the serum levels of IL-2, IL...","Based on the provided evidence, there is no di...",appropriate_abstention,The retrieved context does not provide suffici...
20,549,Metastatic carcinoma to the cervical nodes fro...,"Based on the provided evidence, there is no di...",appropriate_abstention,The retrieved context does not provide suffici...


Cell 53 — Save the corrected annotation workspace

In [57]:
# Cell 53 — Save corrected annotation workspace

review_file = "/kaggle/working/medical_hallucination_annotation_review_v2.csv"

review_df.to_csv(
    review_file,
    index=False
)

print("Saved:", review_file)

print("\nFinal proposed distribution:")
print(review_df["proposed_label"].value_counts())

print("\nTotal:", len(review_df))

Saved: /kaggle/working/medical_hallucination_annotation_review_v2.csv

Final proposed distribution:
proposed_label
appropriate_abstention    146
supported                  52
partially_supported         2
Name: count, dtype: int64

Total: 200


Cell 54 — Validate annotation spans

In [58]:
# Cell 54 — Validate all hallucination spans

import json

def validate_span(answer, span):
    start = int(span["start"])
    end = int(span["end"])

    if start < 0 or end > len(answer):
        return False, "Span is outside answer boundaries."

    if start >= end:
        return False, "Invalid start/end positions."

    span_text = answer[start:end]

    if not span_text.strip():
        return False, "Span contains no meaningful text."

    return True, span_text


invalid_spans = []
valid_span_count = 0
total_span_count = 0

for _, row in review_df.iterrows():

    raw_spans = row["hallucination_spans"]

    if not raw_spans or str(raw_spans).strip() in ["", "[]", "nan"]:
        continue

    try:
        spans = json.loads(raw_spans)
    except Exception as e:
        invalid_spans.append({
            "test_id": row["test_id"],
            "reason": f"Could not parse JSON: {e}",
            "spans": raw_spans
        })
        continue

    for span in spans:
        total_span_count += 1

        valid, info = validate_span(
            str(row["answer"]),
            span
        )

        if valid:
            valid_span_count += 1
        else:
            invalid_spans.append({
                "test_id": row["test_id"],
                "reason": info,
                "spans": span
            })

print("Total hallucination spans:", total_span_count)
print("Valid spans:", valid_span_count)
print("Invalid spans:", len(invalid_spans))

if invalid_spans:
    print("\nInvalid span examples:")
    display(pd.DataFrame(invalid_spans).head(20))
else:
    print("\nAll hallucination spans are valid.")

Total hallucination spans: 2
Valid spans: 2
Invalid spans: 0

All hallucination spans are valid.


Cell 55 — Inspect the LettuceDetect training code

In [59]:
# Cell 55 — Inspect LettuceDetect training configuration

from pathlib import Path

repo_path = Path("/kaggle/working/Hallucination-LettuceDetect")

train_script = repo_path / "scripts" / "train.py"
trainer_file = repo_path / "lettucedetect" / "models" / "trainer.py"
dataset_file = repo_path / "lettucedetect" / "datasets" / "ragtruth.py"

print("=" * 100)
print("TRAIN SCRIPT")
print("=" * 100)
print(train_script.read_text()[:12000])

print("\n" + "=" * 100)
print("TRAINER")
print("=" * 100)
print(trainer_file.read_text()[:16000])

print("\n" + "=" * 100)
print("RAGTRUTH DATASET")
print("=" * 100)
print(dataset_file.read_text()[:12000])

TRAIN SCRIPT
import argparse
import json
from pathlib import Path

from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)

from lettucedetect.datasets.ragtruth import RagTruthDataset
from lettucedetect.models.trainer import Trainer
from lettucedetect.preprocess.preprocess_ragtruth import RagTruthData


def parse_args():
    parser = argparse.ArgumentParser(description="Train hallucination detector model")
    parser.add_argument(
        "--data-path",
        type=str,
        default="data/ragtruth/ragtruth_data.json",
        help="Path to the training data JSON file",
    )
    parser.add_argument(
        "--model-name",
        type=str,
        default="answerdotai/ModernBERT-base",
        help="Name or path of the pretrained model",
    )
    parser.add_argument(
        "--output-dir",
        type=str,
        default="output/hallucination_detector",
        help="Di

Cell 56 — Build clean LettuceDetect dataset

In [60]:
# Cell 56 — Build clean LettuceDetect-compatible dataset

import json
from pathlib import Path

# Use the corrected annotation workspace
work_df = review_df.copy()

# ------------------------------------------------------------------
# Convert hallucination_spans into a consistent Python list
# ------------------------------------------------------------------
def parse_hallucination_spans(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if text in ["", "[]", "nan", "None"]:
        return []

    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return parsed

    except Exception:
        pass

    return []


work_df["parsed_spans"] = work_df["hallucination_spans"].apply(
    parse_hallucination_spans
)

# ------------------------------------------------------------------
# IMPORTANT:
# For supported and appropriate-abstention answers,
# there should be no hallucinated spans.
#
# For partially-supported / hallucinated answers,
# we use the character-level spans we verified.
# ------------------------------------------------------------------

def normalize_label(row):
    label = str(row["proposed_label"]).strip()

    if label not in [
        "supported",
        "partially_supported",
        "hallucinated",
        "appropriate_abstention"
    ]:
        raise ValueError(
            f"Unexpected label for test_id={row['test_id']}: {label}"
        )

    return label


work_df["final_annotation"] = work_df.apply(
    normalize_label,
    axis=1
)

# ------------------------------------------------------------------
# Verify that every span exactly matches the answer text
# ------------------------------------------------------------------

span_errors = []

for _, row in work_df.iterrows():

    answer = str(row["answer"])
    spans = row["parsed_spans"]

    for span in spans:

        start = int(span["start"])
        end = int(span["end"])

        if start < 0 or end > len(answer) or start >= end:
            span_errors.append({
                "test_id": row["test_id"],
                "error": "Invalid span boundaries",
                "span": span
            })
            continue

        span_text = answer[start:end]

        if not span_text.strip():
            span_errors.append({
                "test_id": row["test_id"],
                "error": "Empty span",
                "span": span
            })

# ------------------------------------------------------------------
# Check consistency between labels and spans
# ------------------------------------------------------------------

label_span_errors = []

for _, row in work_df.iterrows():

    label = row["final_annotation"]
    spans = row["parsed_spans"]

    if label in ["supported", "appropriate_abstention"] and len(spans) > 0:
        label_span_errors.append({
            "test_id": row["test_id"],
            "label": label,
            "error": "Supported/abstention example has hallucination spans"
        })

    if label in ["partially_supported", "hallucinated"] and len(spans) == 0:
        label_span_errors.append({
            "test_id": row["test_id"],
            "label": label,
            "error": "Hallucinated example has no hallucination spans"
        })

print("=" * 80)
print("ANNOTATION DATASET CHECK")
print("=" * 80)

print("Total examples:", len(work_df))
print("Invalid spans:", len(span_errors))
print("Label/span inconsistencies:", len(label_span_errors))

print("\nFinal annotation distribution:")
print(work_df["final_annotation"].value_counts())

if span_errors:
    print("\nSpan errors:")
    display(pd.DataFrame(span_errors))

if label_span_errors:
    print("\nLabel/span inconsistencies:")
    display(pd.DataFrame(label_span_errors))

if not span_errors and not label_span_errors:
    print("\n✅ Annotation structure is internally consistent.")

ANNOTATION DATASET CHECK
Total examples: 200
Invalid spans: 0
Label/span inconsistencies: 0

Final annotation distribution:
final_annotation
appropriate_abstention    146
supported                  52
partially_supported         2
Name: count, dtype: int64

✅ Annotation structure is internally consistent.


Cell 57 — Create the LettuceDetect train/test JSON

In [61]:
# Cell 57 — Create LettuceDetect-compatible JSON files

from pathlib import Path
import json

output_dir = Path("/kaggle/working/medical_lettuce_dataset")
output_dir.mkdir(parents=True, exist_ok=True)


def dataframe_to_samples(df, split_name):
    samples = []

    for _, row in df.iterrows():

        spans = row["parsed_spans"]

        sample = {
            "prompt": str(row["context"]),
            "answer": str(row["answer"]),
            "labels": spans,
            "split": split_name,
            "task_type": "medical_rag"
        }

        samples.append(sample)

    return samples


# ------------------------------------------------------------
# Build the three splits from the FIXED split created earlier
# ------------------------------------------------------------

train_samples = dataframe_to_samples(
    annotation_train.merge(
        work_df[
            [
                "test_id",
                "final_annotation",
                "parsed_spans"
            ]
        ],
        on="test_id",
        how="left"
    ),
    "train"
)

validation_samples = dataframe_to_samples(
    annotation_val.merge(
        work_df[
            [
                "test_id",
                "final_annotation",
                "parsed_spans"
            ]
        ],
        on="test_id",
        how="left"
    ),
    "validation"
)

test_samples = dataframe_to_samples(
    annotation_test.merge(
        work_df[
            [
                "test_id",
                "final_annotation",
                "parsed_spans"
            ]
        ],
        on="test_id",
        how="left"
    ),
    "test"
)


# ------------------------------------------------------------
# Save separate JSON files
# ------------------------------------------------------------

train_file = output_dir / "medical_train.json"
validation_file = output_dir / "medical_validation.json"
test_file = output_dir / "medical_test.json"

train_file.write_text(
    json.dumps(train_samples, ensure_ascii=False, indent=2)
)

validation_file.write_text(
    json.dumps(validation_samples, ensure_ascii=False, indent=2)
)

test_file.write_text(
    json.dumps(test_samples, ensure_ascii=False, indent=2)
)


# ------------------------------------------------------------
# Create combined file in the exact format expected by
# LettuceDetect's RagTruthData.from_json()
# ------------------------------------------------------------

all_samples = (
    train_samples
    + validation_samples
    + test_samples
)

combined_file = output_dir / "medical_ragtruth_data.json"

combined_file.write_text(
    json.dumps(
        {
            "samples": all_samples
        },
        ensure_ascii=False,
        indent=2
    )
)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print("=" * 80)
print("MEDICAL LETTUCEDETECT DATASET CREATED")
print("=" * 80)

print("Training samples:", len(train_samples))
print("Validation samples:", len(validation_samples))
print("Final test samples:", len(test_samples))
print("Total samples:", len(all_samples))

print("\nFiles:")

print(train_file)
print(validation_file)
print(test_file)
print(combined_file)

print("\nFirst sample:")
print(json.dumps(train_samples[0], ensure_ascii=False, indent=2)[:3000])

MEDICAL LETTUCEDETECT DATASET CREATED
Training samples: 140
Validation samples: 20
Final test samples: 40
Total samples: 200

Files:
/kaggle/working/medical_lettuce_dataset/medical_train.json
/kaggle/working/medical_lettuce_dataset/medical_validation.json
/kaggle/working/medical_lettuce_dataset/medical_test.json
/kaggle/working/medical_lettuce_dataset/medical_ragtruth_data.json

First sample:
{
  "prompt": "Program directors/coordinators in each surgical residency program in New England.\n\nAll Accreditation Council for Graduate Medical Education (ACGME) accredited surgical residency programs in New England (n = 20).\n\nUsing the Health Belief Model as a framework, the authors created an instrument to survey 424 residents from 11 residency programs at three academic medical institutions about their intentions to have children during residency. The authors developed a scale to assess the perceived career threats of having children during residency, evaluated its psychometric properties,

Cell 58 — Test LettuceDetect dataset loading

In [62]:
# Cell 58 — Fix combined JSON format for LettuceDetect

import json
from pathlib import Path

output_dir = Path("/kaggle/working/medical_lettuce_dataset")

# Combine all samples
all_samples = (
    train_samples
    + validation_samples
    + test_samples
)

# IMPORTANT:
# RagTruthData.from_json() expects a LIST directly,
# not {"samples": [...]}

combined_file = output_dir / "medical_ragtruth_data.json"

combined_file.write_text(
    json.dumps(
        all_samples,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Fixed:", combined_file)
print("Total samples:", len(all_samples))

# Verify the JSON structure
with open(combined_file, "r", encoding="utf-8") as f:
    check_data = json.load(f)

print("Loaded JSON type:", type(check_data).__name__)
print("First item type:", type(check_data[0]).__name__)

print("\nFirst item keys:")
print(check_data[0].keys())

print("\n✅ JSON format fixed.")

Fixed: /kaggle/working/medical_lettuce_dataset/medical_ragtruth_data.json
Total samples: 200
Loaded JSON type: list
First item type: dict

First item keys:
dict_keys(['prompt', 'answer', 'labels', 'split', 'task_type'])

✅ JSON format fixed.


In [63]:
# Cell 59 — Verify LettuceDetect can load the corrected dataset

import json
import torch

from lettucedetect.preprocess.preprocess_ragtruth import RagTruthData
from lettucedetect.datasets.ragtruth import RagTruthDataset
from transformers import AutoTokenizer

medical_json_path = (
    "/kaggle/working/medical_lettuce_dataset/medical_ragtruth_data.json"
)

with open(medical_json_path, "r", encoding="utf-8") as f:
    medical_json = json.load(f)

medical_data = RagTruthData.from_json(medical_json)

print("=" * 80)
print("LETtUCEDETECT DATASET LOADING")
print("=" * 80)

print("Total samples:", len(medical_data.samples))

from collections import Counter

print("\nSplit distribution:")
print(
    Counter(
        sample.split
        for sample in medical_data.samples
    )
)

print("\nFirst sample:")
print("Prompt length:", len(medical_data.samples[0].prompt))
print("Answer length:", len(medical_data.samples[0].answer))
print("Labels:", medical_data.samples[0].labels)
print("Split:", medical_data.samples[0].split)


# ------------------------------------------------------------
# Load the SAME tokenizer used by pretrained LettuceDetect
# ------------------------------------------------------------

LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

tokenizer = AutoTokenizer.from_pretrained(
    LETTUCE_MODEL
)

train_samples_check = [
    sample
    for sample in medical_data.samples
    if sample.split == "train"
]

test_dataset_check = RagTruthDataset(
    train_samples_check[:1],
    tokenizer
)

test_item = test_dataset_check[0]

print("\n" + "=" * 80)
print("TOKENIZATION TEST")
print("=" * 80)

print("Input IDs shape:", test_item["input_ids"].shape)
print("Attention mask shape:", test_item["attention_mask"].shape)
print("Labels shape:", test_item["labels"].shape)

print("\nUnique labels:")
print(torch.unique(test_item["labels"]).tolist())

print("\n✅ Dataset loading and tokenization successful.")

LETtUCEDETECT DATASET LOADING
Total samples: 200

Split distribution:
Counter({'train': 140, 'test': 40, 'validation': 20})

First sample:
Prompt length: 1687
Answer length: 751
Labels: []
Split: train

TOKENIZATION TEST
Input IDs shape: torch.Size([421])
Attention mask shape: torch.Size([421])
Labels shape: torch.Size([421])

Unique labels:
[-100, 0]

✅ Dataset loading and tokenization successful.


Cell 60 — Verify the two hallucination labels

In [64]:
# Cell 60 — Verify hallucination spans become token-level label 1

# Find the two examples containing hallucination spans
span_examples = [
    sample
    for sample in medical_data.samples
    if len(sample.labels) > 0
]

print("Examples with annotated hallucination spans:", len(span_examples))

for i, sample in enumerate(span_examples):

    print("\n" + "=" * 80)
    print(f"Example {i + 1}")
    print("=" * 80)

    print("Split:", sample.split)
    print("Number of spans:", len(sample.labels))
    print("Labels:", sample.labels)

    # Create a one-example dataset
    dataset = RagTruthDataset(
        [sample],
        tokenizer
    )

    item = dataset[0]

    unique_labels = torch.unique(
        item["labels"]
    ).tolist()

    print("Token labels present:", unique_labels)

    hallucinated_token_count = (
        item["labels"] == 1
    ).sum().item()

    print(
        "Hallucinated tokens:",
        hallucinated_token_count
    )

    assert hallucinated_token_count > 0, (
        f"ERROR: test_id example {i} has spans "
        "but no token labels marked as hallucinated."
    )

print("\n✅ Both annotated hallucination spans produce token label 1.")

Examples with annotated hallucination spans: 2

Example 1
Split: train
Number of spans: 1
Labels: [{'start': 447, 'end': 480, 'label': 'hallucinated'}]
Token labels present: [-100, 0, 1]
Hallucinated tokens: 6

Example 2
Split: train
Number of spans: 1
Labels: [{'start': 377, 'end': 502, 'label': 'hallucinated'}]
Token labels present: [-100, 0, 1]
Hallucinated tokens: 20

✅ Both annotated hallucination spans produce token label 1.


Cell 61 — Prepare train/test loaders

In [65]:
# Cell 61 — Prepare LettuceDetect training and final-test datasets

import json
import torch

from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification
)

from lettucedetect.datasets.ragtruth import RagTruthDataset
from lettucedetect.preprocess.preprocess_ragtruth import RagTruthData


# ------------------------------------------------------------
# Load the medical dataset
# ------------------------------------------------------------

medical_json_path = (
    "/kaggle/working/medical_lettuce_dataset/medical_ragtruth_data.json"
)

with open(medical_json_path, "r", encoding="utf-8") as f:
    medical_json = json.load(f)

medical_data = RagTruthData.from_json(medical_json)


# ------------------------------------------------------------
# Get train and final-test samples
# ------------------------------------------------------------

train_samples = [
    sample
    for sample in medical_data.samples
    if sample.split == "train"
]

test_samples = [
    sample
    for sample in medical_data.samples
    if sample.split == "test"
]


print("Training samples:", len(train_samples))
print("Final test samples:", len(test_samples))


# ------------------------------------------------------------
# Load the PRETRAINED LETTUCEDETECT tokenizer
# ------------------------------------------------------------

LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

fine_tune_tokenizer = AutoTokenizer.from_pretrained(
    LETTUCE_MODEL
)


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = RagTruthDataset(
    train_samples,
    fine_tune_tokenizer,
    max_length=4096
)

test_dataset = RagTruthDataset(
    test_samples,
    fine_tune_tokenizer,
    max_length=4096
)


# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------

data_collator = DataCollatorForTokenClassification(
    tokenizer=fine_tune_tokenizer,
    label_pad_token_id=-100
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)


print("\n" + "=" * 80)
print("DATALOADERS READY")
print("=" * 80)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

print("\nBatch shapes:")

first_batch = next(iter(train_loader))

print(
    "input_ids:",
    first_batch["input_ids"].shape
)

print(
    "attention_mask:",
    first_batch["attention_mask"].shape
)

print(
    "labels:",
    first_batch["labels"].shape
)

print("\nUnique labels in first batch:")
print(
    torch.unique(
        first_batch["labels"]
    ).tolist()
)

print("\n✅ Training and final-test loaders are ready.")

Training samples: 140
Final test samples: 40

DATALOADERS READY
Train batches: 35
Test batches: 10

Batch shapes:
input_ids: torch.Size([4, 745])
attention_mask: torch.Size([4, 745])
labels: torch.Size([4, 745])

Unique labels in first batch:
[-100, 0, 1]

✅ Training and final-test loaders are ready.


Cell 62 — Load the pretrained LettuceDetect model

In [66]:
# Cell 62 — Load pretrained LettuceDetect for medical fine-tuning

from transformers import AutoModelForTokenClassification

LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

medical_model = AutoModelForTokenClassification.from_pretrained(
    LETTUCE_MODEL,
    num_labels=2
)

print("=" * 80)
print("PRETRAINED LETTUCEDETECT LOADED")
print("=" * 80)

print("Model:", LETTUCE_MODEL)
print("Number of labels:", medical_model.config.num_labels)
print("Hidden size:", medical_model.config.hidden_size)

print("\nLabel mapping:")
print(medical_model.config.id2label)

print("\nTrainable parameters:")
print(
    sum(p.numel() for p in medical_model.parameters() if p.requires_grad)
)

print("\n✅ Ready for medical fine-tuning.")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

PRETRAINED LETTUCEDETECT LOADED
Model: KRLabsOrg/lettucedect-base-modernbert-en-v1
Number of labels: 2
Hidden size: 768

Label mapping:
{0: 'LABEL_0', 1: 'LABEL_1'}

Trainable parameters:
149606402

✅ Ready for medical fine-tuning.


Cell 63 — Fine-tune LettuceDetect on the medical training set

In [67]:
# Cell 63 — Fine-tune pretrained LettuceDetect on medical data

import os
import time
from datetime import timedelta

import torch
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

EPOCHS = 4
LEARNING_RATE = 1e-5
OUTPUT_DIR = "/kaggle/working/lettucedetect_medical_adapted"

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

medical_model.to(device)

optimizer = torch.optim.AdamW(
    medical_model.parameters(),
    lr=LEARNING_RATE
)

total_training_steps = EPOCHS * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps
)

print("=" * 80)
print("MEDICAL LETTUCEDETECT FINE-TUNING")
print("=" * 80)

print("Device:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Training samples:", len(train_dataset))
print("Training batches per epoch:", len(train_loader))
print("Output directory:", OUTPUT_DIR)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

training_start = time.time()

for epoch in range(EPOCHS):

    epoch_start = time.time()

    medical_model.train()

    total_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:

        optimizer.zero_grad()

        outputs = medical_model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss / num_batches:.4f}"
        )

    avg_loss = total_loss / num_batches

    epoch_time = time.time() - epoch_start

    print(
        f"\nEpoch {epoch + 1} completed "
        f"in {timedelta(seconds=int(epoch_time))}"
    )

    print(
        f"Average training loss: {avg_loss:.4f}"
    )

    print("-" * 80)


# ------------------------------------------------------------
# Save the medically adapted model
# ------------------------------------------------------------

medical_model.save_pretrained(
    OUTPUT_DIR
)

fine_tune_tokenizer.save_pretrained(
    OUTPUT_DIR
)

total_time = time.time() - training_start

print("\n" + "=" * 80)
print("FINE-TUNING COMPLETE")
print("=" * 80)

print(
    "Total training time:",
    timedelta(seconds=int(total_time))
)

print("Model saved to:", OUTPUT_DIR)
print("\n✅ Medical-adapted LettuceDetect saved successfully.")

MEDICAL LETTUCEDETECT FINE-TUNING
Device: cuda
Epochs: 4
Learning rate: 1e-05
Training samples: 140
Training batches per epoch: 35
Output directory: /kaggle/working/lettucedetect_medical_adapted


Epoch 1/4:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 1 completed in 0:00:38
Average training loss: 0.1240
--------------------------------------------------------------------------------


Epoch 2/4:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 2 completed in 0:00:32
Average training loss: 0.0040
--------------------------------------------------------------------------------


Epoch 3/4:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 3 completed in 0:00:32
Average training loss: 0.0017
--------------------------------------------------------------------------------


Epoch 4/4:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 4 completed in 0:00:32
Average training loss: 0.0013
--------------------------------------------------------------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


FINE-TUNING COMPLETE
Total training time: 0:02:17
Model saved to: /kaggle/working/lettucedetect_medical_adapted

✅ Medical-adapted LettuceDetect saved successfully.


Cell 64 — Run the adapted model on the final test set

In [68]:
# Cell 64 — Run medically adapted LettuceDetect on final test set

from lettucedetect.models.inference import HallucinationDetector

ADAPTED_MODEL = "/kaggle/working/lettucedetect_medical_adapted"

adapted_detector = HallucinationDetector(
    method="transformer",
    model_path=ADAPTED_MODEL
)

print("Loaded medically adapted model.")

adapted_spans = []
adapted_tokens = []

for i, (_, row) in enumerate(annotation_test.iterrows(), start=1):

    spans = adapted_detector.predict(
        context=[row["context"]],
        question=row["question"],
        answer=row["answer"],
        output_format="spans"
    )

    tokens = adapted_detector.predict(
        context=[row["context"]],
        question=row["question"],
        answer=row["answer"],
        output_format="tokens"
    )

    adapted_spans.append(spans)
    adapted_tokens.append(tokens)

    if i % 10 == 0:
        print(f"Processed {i}/{len(annotation_test)}")

print("\n" + "=" * 80)
print("ADAPTED MODEL INFERENCE COMPLETE")
print("=" * 80)

print("Examples processed:", len(adapted_spans))
print(
    "Examples with detected spans:",
    sum(len(x) > 0 for x in adapted_spans)
)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loaded medically adapted model.


/kaggle/working/Hallucination-LettuceDetect/lettucedetect/models/inference.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, device=self.device)


Processed 10/40
Processed 20/40
Processed 30/40
Processed 40/40

ADAPTED MODEL INFERENCE COMPLETE
Examples processed: 40
Examples with detected spans: 0


Cell 65 — Build the final 40-example ground truth

In [69]:
# Cell 65 — Build ground-truth token labels for the final 40 test examples

import torch

test_ground_truth = []

for _, row in annotation_test.iterrows():

    # Find the corresponding LettuceDetect sample
    sample = next(
        s for s in medical_data.samples
        if s.split == "test"
        and s.answer == str(row["answer"])
    )

    # Create one-example dataset using the same tokenizer
    dataset = RagTruthDataset(
        [sample],
        tokenizer,
        max_length=4096
    )

    item = dataset[0]

    labels = item["labels"].tolist()

    test_ground_truth.append(labels)


print("=" * 80)
print("GROUND-TRUTH LABELS CREATED")
print("=" * 80)

print("Test examples:", len(test_ground_truth))

print("\nExamples containing hallucinated tokens:")

for i, labels in enumerate(test_ground_truth):
    hallucinated = sum(
        1 for x in labels if x == 1
    )

    if hallucinated > 0:
        print(
            f"Test position {i}: "
            f"{hallucinated} hallucinated tokens"
        )

print("\nTotal hallucinated tokens:")

total_hallucinated = sum(
    sum(1 for x in labels if x == 1)
    for labels in test_ground_truth
)

print(total_hallucinated)

print("\n✅ Ground truth prepared.")

GROUND-TRUTH LABELS CREATED
Test examples: 40

Examples containing hallucinated tokens:

Total hallucinated tokens:
0

✅ Ground truth prepared.


Cell 66 — Check positive candidates in the current test set

In [70]:
# Cell 66 — Check LettuceDetect-flagged candidates in final test set

# Map test_id -> original LettuceDetect span predictions
baseline_span_map = dict(
    zip(
        baseline_df["test_id"],
        baseline_df["lettucedetect_spans"]
    )
)

test_candidates = []

for _, row in annotation_test.iterrows():

    test_id = row["test_id"]
    spans = baseline_span_map.get(test_id, [])

    if spans and len(spans) > 0:
        test_candidates.append(test_id)

print("=" * 80)
print("FINAL TEST POSITIVE CANDIDATES")
print("=" * 80)

print("Final test examples:", len(annotation_test))
print("LettuceDetect-flagged candidates:", len(test_candidates))

print("\nCandidate test IDs:")
print(test_candidates)

FINAL TEST POSITIVE CANDIDATES
Final test examples: 40
LettuceDetect-flagged candidates: 21

Candidate test IDs:
[346, 60, 867, 543, 377, 668, 589, 355, 259, 617, 275, 70, 237, 280, 218, 296, 761, 289, 55, 865, 652]


Cell 67 — Generate a focused review table for the 21 candidates

In [71]:
# Cell 67 — Prepare the 21 flagged final-test candidates for focused review

candidate_test_ids = [
    346, 60, 867, 543, 377, 668, 589, 355, 259,
    617, 275, 70, 237, 280, 218, 296, 761, 289,
    55, 865, 652
]

candidate_review = review_df[
    review_df["test_id"].isin(candidate_test_ids)
].copy()

# Keep the original final-test order
candidate_order = {
    test_id: i
    for i, test_id in enumerate(candidate_test_ids)
}

candidate_review["order"] = candidate_review["test_id"].map(candidate_order)
candidate_review = candidate_review.sort_values("order")

print("=" * 100)
print("FOCUSED HUMAN REVIEW — 21 FINAL-TEST CANDIDATES")
print("=" * 100)

for _, row in candidate_review.iterrows():

    print("\n" + "=" * 100)
    print(f"TEST ID: {row['test_id']}")
    print("=" * 100)

    print("\nQUESTION:")
    print(row["question"])

    print("\nRETRIEVED CONTEXT:")
    print(row["context"])

    print("\nGENERATED ANSWER:")
    print(row["answer"])

    print("\nREFERENCE ANSWER:")
    print(row["reference_answer"])

    print("\nCURRENT MODEL PROPOSAL:")
    print(row["proposed_label"])

    print("\nPROPOSAL REASON:")
    print(row["proposal_reason"])

print("\n" + "=" * 100)
print("END OF CANDIDATE REVIEW")
print("=" * 100)

FOCUSED HUMAN REVIEW — 21 FINAL-TEST CANDIDATES

TEST ID: 346

QUESTION:
Can multiple SNP testing in BRCA2 and BRCA1 female carriers be used to improve risk prediction models in conjunction with clinical assessment?

RETRIEVED CONTEXT:
Separate Cox proportional hazard regression models were formulated based on all patients to evaluate the effects of gender on mortality, multiple organ failure, and nosocomial infection, after controlling for all important confounders. These models were then used to characterize the effect of gender in young and old age groups. Overall mortality, multiple organ failure, and nosocomial infection rates for the entire cohort (n = 1,036) were 20%, 40%, and 45%, respectively. Mean Injury Severity Score was 32 +/- 14 (mean +/- SD). Men (n = 680) and women (n = 356) were clinically similar except that men required higher crystalloid volumes, more often had a history of alcoholism and liver disease, and had greater ventilatory and intensive care unit requirement

Cell 68 — Record the human-reviewed labels

In [72]:
# Cell 68 — Record human-reviewed labels for final-test candidates

human_review_labels = {
    346: {
        "label": "hallucinated",
        "texts": [
            "Yes, multiple SNP testing in BRCA2 and BRCA1 female carriers can be used to improve risk prediction models in conjunction with clinical assessment.",
            "This suggests that incorporating genetic information into risk prediction models may help identify individuals at higher risk who might benefit from additional interventions."
        ],
        "notes": "The retrieved evidence is unrelated to BRCA1/BRCA2 SNP testing or genetic risk prediction."
    },

    60: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address autoerotic asphyxiation."
    },

    867: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not directly compare triage nurses with an objective admission score."
    },

    543: {
        "label": "partially_supported",
        "texts": [
            "while some research suggests that early intervention might offer benefits"
        ],
        "notes": "The main abstention is appropriate, but the statement about benefits of early intervention is unsupported by the retrieved context."
    },

    377: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address elective IV cannula re-siting or thrombophlebitis."
    },

    668: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address fondaparinux in perioperative bridging."
    },

    589: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address microbiological factors among unmarried pregnant women in Poland."
    },

    355: {
        "label": "partially_supported",
        "texts": [
            "vitamin D plays a role in calcium homeostasis and bone health, which could indirectly affect OCD through its impact on bone density and mineral metabolism.",
            "vitamin D deficiency has been linked to autoimmune diseases, which might contribute to OCD"
        ],
        "notes": "The retrieved context does not establish these specific vitamin-D/OCD connections."
    },

    259: {
        "label": "hallucinated",
        "texts": [
            "it appears that stage I non-small cell lung carcinoma may indeed represent an early stage of the disease.",
            "the data from the University of Texas M. D. Anderson Cancer Center indicates that while stage I non-small cell lung carcinoma can occur, it often progresses to more advanced stages over time."
        ],
        "notes": "The retrieved evidence does not support these claims about stage I NSCLC being early or progressing to advanced stages."
    },

    617: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address CAM use from 2002 to 2007."
    },

    275: {
        "label": "partially_supported",
        "texts": [
            "there is some indirect support from Evidence 5 suggesting that carotid artery pulsatility and resistance indexes may be related to vascular abnormalities"
        ],
        "notes": "Evidence 5 concerns carotid IMT and cardiovascular risk factors, but does not establish the claimed relationship involving pulsatility/resistance indexes."
    },

    70: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence concerns phonological processing but does not establish whether specialized training is necessary for every preschool child."
    },

    237: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address BMR prediction equations in Chinese populations."
    },

    280: {
        "label": "hallucinated",
        "texts": [
            "some research suggests that cardiovascular diseases, which can lead to elevated blood pressure, might increase the risk of developing chronic low back pain.",
            "other health issues linked to hypertension could potentially contribute to its development."
        ],
        "notes": "The retrieved context does not establish these hypertension/cardiovascular-disease relationships with chronic low back pain."
    },

    218: {
        "label": "hallucinated",
        "texts": [
            "Yes, topical ropivacaine reduces the post-tonsillectomy morbidity in pediatric patients",
            "This suggests that nighttime bleeding is more frequent than daytime bleeding after tonsillectomy."
        ],
        "notes": "The retrieved evidence concerns postoperative bleeding timing and contains no evidence about topical ropivacaine."
    },

    296: {
        "label": "hallucinated",
        "texts": [
            "the evidence suggests that nurse-led care has been successfully implemented in other contexts, indicating that it could potentially be feasible under appropriate circumstances."
        ],
        "notes": "The retrieved evidence does not establish nurse-led cystoscopy feasibility."
    },

    761: {
        "label": "hallucinated",
        "texts": [
            "routine oophorectomy is generally unnecessary unless specifically indicated by clinical findings such as pain, mass, or suspicion of malignancy.",
            "Ovarian endometriomas do not typically cause symptoms until they become large enough to compress surrounding structures"
        ],
        "notes": "The retrieved evidence concerns ovarian endometrioma/cancer, not pediatric ovarian torsion or oophorectomy."
    },

    289: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address seafarer fitness assessments."
    },

    55: {
        "label": "partially_supported",
        "texts": [
            "some studies suggest that elevated levels of certain hormones like testosterone may influence libido"
        ],
        "notes": "This hormone/libido claim is not supported by the retrieved context."
    },

    865: {
        "label": "partially_supported",
        "texts": [
            "suggesting that they might offer comparable outcomes in terms of preserving muscle strength and physical performance.",
            "the general trend seems favorable towards maintaining muscle strength and physical performance with unicompartmental approaches."
        ],
        "notes": "The retrieved evidence concerns unicompartmental/tricompartmental mechanics, not bicompartmental versus total knee arthroplasty clinical performance."
    },

    652: {
        "label": "hallucinated",
        "texts": [
            "chemoradiation is generally considered safer than surgery for treating mid and low rectal tumors.",
            "Therefore, based on the available evidence, chemoradiation appears to be a safer treatment option compared to surgery for mid and low rectal tumors."
        ],
        "notes": "The retrieved evidence contains no relevant evidence comparing chemoradiation and surgery for rectal tumors."
    }
}


def create_spans_from_texts(answer, texts):
    spans = []

    for text in texts:
        start = answer.find(text)

        if start == -1:
            raise ValueError(
                f"Could not find reviewed text in answer:\n{text}"
            )

        end = start + len(text)

        spans.append({
            "start": start,
            "end": end,
            "label": "hallucinated"
        })

    return spans


# Apply human-reviewed labels
for test_id, review in human_review_labels.items():

    idx = annotation_df.index[
        annotation_df["test_id"] == test_id
    ]

    if len(idx) != 1:
        raise ValueError(f"Expected exactly one row for test_id={test_id}")

    idx = idx[0]

    answer = str(annotation_df.loc[idx, "answer"])

    spans = create_spans_from_texts(
        answer,
        review["texts"]
    )

    annotation_df.loc[idx, "overall_label"] = review["label"]

    annotation_df.loc[idx, "hallucination_spans"] = json.dumps(
        spans,
        ensure_ascii=False
    )

    annotation_df.loc[idx, "annotator_notes"] = review["notes"]


print("=" * 80)
print("HUMAN REVIEW LABELS RECORDED")
print("=" * 80)

print(
    annotation_df[
        annotation_df["test_id"].isin(candidate_test_ids)
    ]["overall_label"].value_counts()
)

print("\nTotal reviewed candidates:", len(human_review_labels))

HUMAN REVIEW LABELS RECORDED
overall_label
appropriate_abstention    9
hallucinated              7
partially_supported       5
Name: count, dtype: int64

Total reviewed candidates: 21


Cell 69 — Show the remaining 19 final-test cases

In [73]:
# Cell 69 — Show the 19 unflagged final-test cases for human review

candidate_test_ids = list(human_review_labels.keys())

unflagged_test_df = annotation_df[
    annotation_df["test_id"].isin(
        annotation_test["test_id"]
    )
    & ~annotation_df["test_id"].isin(candidate_test_ids)
].copy()

print("=" * 80)
print("UNFLAGGED FINAL-TEST CASES")
print("=" * 80)

print("Number of cases:", len(unflagged_test_df))

display(
    unflagged_test_df[
        [
            "test_id",
            "question",
            "answer",
            "context"
        ]
    ].reset_index(drop=True)
)

UNFLAGGED FINAL-TEST CASES
Number of cases: 19


,test_id,question,answer,context
0,136,Can transcranial direct current stimulation be...,The available evidence is insufficient to answ...,Patients (N=22) in a vegetative state (VS; n=1...
1,899,Are women who are treated for hypothyroidism a...,"Based on the provided evidence, there is no di...",Unintended pregnancy has been significantly as...
2,277,Does open access publishing increase the impac...,"Based on the provided evidence, there is limit...",The use of open access endoscopy is increasing...
3,30,Necrotizing fasciitis: an indication for hyper...,"Based on the evidence provided, there is no di...",Our aim was to investigate the effects of grow...
4,692,Does normothermic normokalemic simultaneous an...,"Based on the evidence provided, there is no di...",Occlusion of the descending aorta and infusion...
5,662,Endovenous laser ablation in the treatment of ...,The available evidence is insufficient to answ...,Mechanically ventilated patients experience pr...
6,866,Can gingival crevicular blood be relied upon f...,The available evidence is insufficient to answ...,Bolus intravenous injection of epinephrine can...
7,318,Do familiar teammates request and accept more ...,The available evidence is insufficient to answ...,Answers showed a large variety in clinical pra...
8,436,Does having a regular primary care clinician i...,"Based on the evidence provided, there is no di...",High managed care penetration in the community...
9,635,Does early discharge with nurse home visits af...,"Based on the provided evidence, there is no di...",The unadjusted odds ratio for early discharge ...


Cell 70 — Print the 19 cases clearly

In [74]:
# Cell 70 — Print unflagged final-test cases for human review

for i, (_, row) in enumerate(unflagged_test_df.iterrows(), start=1):

    print("\n" + "=" * 100)
    print(f"CASE {i}/19   |   TEST ID: {row['test_id']}")
    print("=" * 100)

    print("\nQUESTION:")
    print(row["question"])

    print("\nGENERATED ANSWER:")
    print(row["answer"])

    print("\nRETRIEVED CONTEXT:")
    print(row["context"])

    print("\nREFERENCE ANSWER:")
    print(row["reference_answer"])


CASE 1/19   |   TEST ID: 136

QUESTION:
Can transcranial direct current stimulation be useful in differentiating unresponsive wakefulness syndrome from minimally conscious state patients?

GENERATED ANSWER:
The available evidence is insufficient to answer this question. There is no specific research directly comparing the effectiveness of transcranial direct current stimulation in differentiating between unresponsive wakefulness syndrome and minimally conscious state patients. Further studies would be needed to address this gap in understanding.

RETRIEVED CONTEXT:
Patients (N=22) in a vegetative state (VS; n=10) and minimally conscious state (MCS; n=12) during the first 200 days after the initial incident.

Atrial fibrillation, which occurs in 12% of all major foregut surgeries, can prolong hospital stay and increase morbidity. Minimally invasive techniques in foregut surgery have been suggested to cause less tissue trauma. We examined the factors associated with new-onset atrial fib

Cell 71 — Record the remaining 19 human-review labels

In [75]:
# Cell 71 — Record human-reviewed labels for the 19 unflagged final-test cases

unflagged_human_review = {

    136: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address whether a-tDCS can differentiate UWS from MCS."
    },

    899: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address treated hypothyroidism and pregnancy complications."
    },

    277: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence concerns open access endoscopy, not open access scientific publishing or article impact. The answer appropriately concludes that the evidence is insufficient."
    },

    30: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence concerns hyperbaric oxygen in rat colonic anastomoses, not necrotizing fasciitis."
    },

    692: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address normothermic normokalemic simultaneous antegrade/retrograde perfusion in hypertrophied hearts."
    },

    662: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address endovenous laser ablation of small saphenous varicose veins or access-site effects."
    },

    866: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence mentions blood glucose in a surgical animal study but does not address gingival crevicular blood as a source for glucose measurement."
    },

    318: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address familiarity between teammates or requests and acceptance of backup."
    },

    436: {
        "label": "partially_supported",
        "texts": [
            "which could potentially affect preventive care"
        ],
        "notes": "The evidence supports an association between managed care penetration and usual source of care, but does not establish that this affects preventive care for young children."
    },

    635: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address newborn metabolic screening in relation to early discharge and nurse home visits."
    },

    432: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence discusses cardiac troponin in pulmonary embolism and PCI, but does not address troponin elevation following dobutamine stress echocardiography."
    },

    559: {
        "label": "partially_supported",
        "texts": [
            "Therefore, it can be inferred that physicians may meet the needs of family caregivers of the frail elderly through such interventions."
        ],
        "notes": "The context supports that a multi-component family-network program reduced depressive symptoms, but it does not establish that physicians are meeting caregiver needs through such interventions."
    },

    265: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address pectins or cold acclimation/de-acclimation in winter oil-seed rape."
    },

    215: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address high-risk HPV detection in human breast milk."
    },

    822: {
        "label": "partially_supported",
        "texts": [
            "there is some indirect support suggesting that elevated levels of certain pancreatic enzymes might indicate pancreatic disease"
        ],
        "notes": "The context supports serum pancreatic amylase/lipase as diagnostic markers for pancreatitis, but does not support the claimed relationship involving cytokine concentrations in pancreatic juice."
    },

    643: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address static foot posture measurements and prediction of regional plantar surface area."
    },

    977: {
        "label": "partially_supported",
        "texts": [
            "there is some indirect support for the idea that the internet could potentially improve sexual health awareness among young people"
        ],
        "notes": "The context supports potential use of the Internet for research and interventions, but does not establish improved sexual health awareness in young people."
    },

    361: {
        "label": "hallucinated",
        "texts": [
            "it seems that STIR imaging cannot be omitted when diagnosing bone metastases at both 1.5 T and 3 T.",
            "Therefore, while other imaging techniques may be useful, STIR imaging should not be omitted as part of the diagnostic process for bone metastases."
        ],
        "notes": "The retrieved evidence discusses survival outcomes associated with bone metastases but contains no evidence about STIR imaging or whether it can be omitted."
    },

    486: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Retrieved evidence does not address the correlation between end-tidal CO2 and arterial CO2 in extremely low birth weight infants."
    }
}


def create_spans_from_texts(answer, texts):
    spans = []

    for text in texts:
        start = answer.find(text)

        if start == -1:
            raise ValueError(
                f"Could not find reviewed text in answer:\n{text}"
            )

        end = start + len(text)

        spans.append({
            "start": start,
            "end": end,
            "label": "hallucinated"
        })

    return spans


# Apply labels to annotation_df
for test_id, review in unflagged_human_review.items():

    idx = annotation_df.index[
        annotation_df["test_id"] == test_id
    ]

    if len(idx) != 1:
        raise ValueError(
            f"Expected exactly one row for test_id={test_id}"
        )

    idx = idx[0]

    answer = str(annotation_df.loc[idx, "answer"])

    spans = create_spans_from_texts(
        answer,
        review["texts"]
    )

    annotation_df.loc[idx, "overall_label"] = review["label"]

    annotation_df.loc[idx, "hallucination_spans"] = json.dumps(
        spans,
        ensure_ascii=False
    )

    annotation_df.loc[idx, "annotator_notes"] = review["notes"]


print("=" * 80)
print("UNFLAGGED HUMAN REVIEW COMPLETE")
print("=" * 80)

print("\nLabels for these 19 cases:")
print(
    annotation_df[
        annotation_df["test_id"].isin(
            unflagged_human_review.keys()
        )
    ]["overall_label"].value_counts()
)

print(
    "\nTotal newly reviewed:",
    len(unflagged_human_review)
)

UNFLAGGED HUMAN REVIEW COMPLETE

Labels for these 19 cases:
overall_label
appropriate_abstention    14
partially_supported        4
hallucinated               1
Name: count, dtype: int64

Total newly reviewed: 19


Cell 72 — Verify the complete 40-example test set

In [76]:
# Cell 72 — Verify that all 40 final-test examples have human annotations

final_test_ids = set(annotation_test["test_id"])

final_test_annotations = annotation_df[
    annotation_df["test_id"].isin(final_test_ids)
].copy()

print("=" * 80)
print("FINAL TEST ANNOTATION VERIFICATION")
print("=" * 80)

print("Final test examples:", len(final_test_annotations))

print("\nLabel distribution:")
print(
    final_test_annotations["overall_label"].value_counts()
)

print("\nMissing labels:")
missing_labels = final_test_annotations[
    final_test_annotations["overall_label"].fillna("").str.strip() == ""
]

print("Missing:", len(missing_labels))

if len(missing_labels) > 0:
    print("\nMissing test IDs:")
    print(missing_labels["test_id"].tolist())

print("\n" + "=" * 80)

# Verify that every hallucination span is valid
invalid_spans = []

for _, row in final_test_annotations.iterrows():

    raw_spans = row["hallucination_spans"]

    if not raw_spans or str(raw_spans).strip() in ["", "[]", "nan"]:
        continue

    try:
        spans = json.loads(raw_spans)
    except Exception as e:
        invalid_spans.append({
            "test_id": row["test_id"],
            "reason": f"JSON parsing error: {e}"
        })
        continue

    for span in spans:

        start = int(span["start"])
        end = int(span["end"])
        answer = str(row["answer"])

        if start < 0 or end > len(answer) or start >= end:
            invalid_spans.append({
                "test_id": row["test_id"],
                "reason": f"Invalid span: {start}-{end}"
            })

print("Invalid hallucination spans:", len(invalid_spans))

if invalid_spans:
    display(pd.DataFrame(invalid_spans))
else:
    print("All hallucination spans are valid.")

print("\n" + "=" * 80)

if len(final_test_annotations) == 40 and len(missing_labels) == 0 and len(invalid_spans) == 0:
    print("✅ FINAL TEST SET IS READY FOR EVALUATION")
else:
    print("⚠️ Final test annotations need correction before evaluation.")

FINAL TEST ANNOTATION VERIFICATION
Final test examples: 40

Label distribution:
overall_label
appropriate_abstention    23
partially_supported        9
hallucinated               8
Name: count, dtype: int64

Missing labels:
Missing: 0

Invalid hallucination spans: 0
All hallucination spans are valid.

✅ FINAL TEST SET IS READY FOR EVALUATION


Cell 73 — Build token-level ground truth for the 40 test examples

In [77]:
# Cell 73 — Build token-level ground truth for the final 40 test examples

import json
import torch
from transformers import AutoTokenizer
from lettucedetect.datasets.ragtruth import RagTruthSample, RagTruthDataset

# ------------------------------------------------------------------
# 1. Load the tokenizer used by the original LettuceDetect model
# ------------------------------------------------------------------

LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

evaluation_tokenizer = AutoTokenizer.from_pretrained(
    LETTUCE_MODEL
)

print("Tokenizer loaded:", LETTUCE_MODEL)

# ------------------------------------------------------------------
# 2. Use the exact final-test examples
# ------------------------------------------------------------------

final_test_annotations = annotation_df[
    annotation_df["test_id"].isin(annotation_test["test_id"])
].copy()

# Preserve the fixed test-set order
final_test_annotations = (
    final_test_annotations
    .set_index("test_id")
    .loc[annotation_test["test_id"]]
    .reset_index()
)

print("Final test examples:", len(final_test_annotations))

# ------------------------------------------------------------------
# 3. Convert annotations into LettuceDetect-compatible samples
# ------------------------------------------------------------------

final_test_samples = []

for _, row in final_test_annotations.iterrows():

    labels = json.loads(row["hallucination_spans"])

    final_test_samples.append(
        RagTruthSample(
            prompt=str(row["context"]),
            answer=str(row["answer"]),
            labels=labels,
            split="test",
            task_type="medical_rag"
        )
    )

# ------------------------------------------------------------------
# 4. Create tokenized LettuceDetect dataset
# ------------------------------------------------------------------

final_test_dataset = RagTruthDataset(
    samples=final_test_samples,
    tokenizer=evaluation_tokenizer,
    max_length=4096
)

print("Tokenized dataset created.")
print("Dataset size:", len(final_test_dataset))

# ------------------------------------------------------------------
# 5. Extract answer-token ground truth
# ------------------------------------------------------------------

ground_truth_token_labels = []

for i in range(len(final_test_dataset)):

    item = final_test_dataset[i]

    labels = item["labels"]

    # Context tokens are -100.
    # Only answer tokens are evaluated.
    answer_labels = labels[labels != -100]

    ground_truth_token_labels.append(
        answer_labels.tolist()
    )

# ------------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------------

total_answer_tokens = sum(
    len(labels)
    for labels in ground_truth_token_labels
)

total_hallucinated_tokens = sum(
    sum(label == 1 for label in labels)
    for labels in ground_truth_token_labels
)

print("\n" + "=" * 80)
print("FINAL TEST TOKEN GROUND TRUTH")
print("=" * 80)

print("Examples:", len(ground_truth_token_labels))
print("Answer tokens:", total_answer_tokens)
print("Hallucinated tokens:", total_hallucinated_tokens)

# ------------------------------------------------------------------
# 7. Show positive examples
# ------------------------------------------------------------------

positive_examples = []

for i, labels in enumerate(ground_truth_token_labels):

    hallucinated_count = sum(
        label == 1 for label in labels
    )

    if hallucinated_count > 0:

        positive_examples.append({
            "test_id": final_test_annotations.iloc[i]["test_id"],
            "overall_label": final_test_annotations.iloc[i]["overall_label"],
            "hallucinated_tokens": hallucinated_count
        })

print("\nExamples with hallucinated tokens:")

if positive_examples:
    display(pd.DataFrame(positive_examples))
else:
    print("None")

print("\n" + "=" * 80)

if total_hallucinated_tokens > 0:
    print("✅ Ground truth contains positive hallucination tokens.")
else:
    print("⚠️ No positive hallucination tokens found.")

Tokenizer loaded: KRLabsOrg/lettucedect-base-modernbert-en-v1
Final test examples: 40
Tokenized dataset created.
Dataset size: 40

FINAL TEST TOKEN GROUND TRUTH
Examples: 40
Answer tokens: 4615
Hallucinated tokens: 550

Examples with hallucinated tokens:


,test_id,overall_label,hallucinated_tokens
0,436,partially_supported,6
1,346,hallucinated,51
2,543,partially_supported,10
3,361,hallucinated,50
4,355,partially_supported,42
5,259,hallucinated,59
6,977,partially_supported,20
7,275,partially_supported,24
8,280,hallucinated,39
9,822,partially_supported,17



✅ Ground truth contains positive hallucination tokens.


Cell 74 — Prepare original and adapted predictions for evaluation

In [78]:
# Cell 74 — Prepare original and medical-adapted predictions

print("=" * 80)
print("PREPARING MODEL PREDICTIONS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Original LettuceDetect predictions
# ------------------------------------------------------------

original_test_predictions = baseline_df[
    baseline_df["test_id"].isin(annotation_test["test_id"])
].copy()

# Preserve exact final-test order
original_test_predictions = (
    original_test_predictions
    .set_index("test_id")
    .loc[annotation_test["test_id"]]
    .reset_index()
)

print("Original model test examples:", len(original_test_predictions))

# ------------------------------------------------------------
# 2. Check adapted predictions
# ------------------------------------------------------------

# adapted_spans was created during Cell 64
# and contains one prediction list for each final-test example.

print("Adapted prediction lists:", len(adapted_spans))

if len(adapted_spans) != len(annotation_test):
    raise ValueError(
        "Number of adapted predictions does not match "
        "the number of final-test examples."
    )

# ------------------------------------------------------------
# 3. Create a consistent evaluation dataframe
# ------------------------------------------------------------

evaluation_df = final_test_annotations[
    [
        "test_id",
        "question",
        "answer",
        "overall_label",
        "hallucination_spans"
    ]
].copy()

evaluation_df["original_spans"] = (
    original_test_predictions["lettucedetect_spans"].tolist()
)

evaluation_df["original_tokens"] = (
    original_test_predictions["lettucedetect_tokens"].tolist()
)

evaluation_df["adapted_spans"] = adapted_spans

print("\nEvaluation dataframe shape:", evaluation_df.shape)

display(
    evaluation_df[
        [
            "test_id",
            "overall_label",
            "original_spans",
            "adapted_spans"
        ]
    ].head(10)
)

print("\n" + "=" * 80)
print("✅ MODEL PREDICTIONS READY FOR EVALUATION")

PREPARING MODEL PREDICTIONS
Original model test examples: 40
Adapted prediction lists: 40

Evaluation dataframe shape: (40, 8)


,test_id,overall_label,original_spans,adapted_spans
0,436,partially_supported,[],[]
1,899,appropriate_abstention,[],[]
2,346,hallucinated,"[{'start': 624, 'end': 791, 'confidence': 0.99...",[]
3,60,appropriate_abstention,"[{'start': 111, 'end': 163, 'confidence': 0.79...",[]
4,867,appropriate_abstention,"[{'start': 404, 'end': 543, 'confidence': 0.96...",[]
5,432,appropriate_abstention,[],[]
6,543,partially_supported,"[{'start': 219, 'end': 246, 'confidence': 0.57...",[]
7,361,hallucinated,[],[]
8,486,appropriate_abstention,[],[]
9,277,appropriate_abstention,[],[]



✅ MODEL PREDICTIONS READY FOR EVALUATION


Cell 75 — Calculate Precision, Recall, and F1

In [79]:
# Cell 75 — Calculate token-level Precision, Recall, and F1

from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 80)
print("TOKEN-LEVEL EVALUATION")
print("=" * 80)


# ------------------------------------------------------------
# Helper: convert LettuceDetect token predictions to 0/1
# ------------------------------------------------------------

def extract_predicted_hallucination_tokens(token_predictions):
    """
    Extract binary hallucination predictions from LettuceDetect
    token-level output.

    pred = 0 -> supported
    pred = 1 -> hallucinated
    """

    predictions = []

    for item in token_predictions:

        pred = int(item["pred"])

        if pred not in [0, 1]:
            raise ValueError(
                f"Unexpected prediction value: {pred}"
            )

        predictions.append(pred)

    return predictions


# ------------------------------------------------------------
# Build common token-level arrays
# ------------------------------------------------------------

y_true_all = []
y_original_all = []
y_adapted_all = []

example_summary = []


for i, row in evaluation_df.iterrows():

    # Ground truth
    true_labels = ground_truth_token_labels[i]

    # Original LettuceDetect token predictions
    original_predictions = extract_predicted_hallucination_tokens(
        row["original_tokens"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # The detector's token output includes answer tokens and
    # a final [SEP] token. Ground truth contains only answer
    # tokens, so align using the common answer-token length.
    # --------------------------------------------------------

    if len(original_predictions) < len(true_labels):
        raise ValueError(
            f"Original predictions are shorter than ground truth "
            f"for test_id={row['test_id']}"
        )

    original_predictions = original_predictions[:len(true_labels)]

    # --------------------------------------------------------
    # Adapted model
    # --------------------------------------------------------

    adapted_predictions = row["adapted_spans"]

    # The adapted model returned character spans.
    # Convert those spans to token predictions using the
    # same tokenizer and answer text.
    
    answer = str(row["answer"])

    # Start with every answer token marked as supported
    adapted_token_predictions = [0] * len(true_labels)

    # Tokenize the answer independently
    encoded_answer = evaluation_tokenizer(
        answer,
        add_special_tokens=False,
        truncation=True,
        max_length=4096,
        return_offsets_mapping=True
    )

    offsets = encoded_answer["offset_mapping"]

    # Mark tokens overlapping predicted hallucination spans
    for span in adapted_predictions:

        start = int(span["start"])
        end = int(span["end"])

        for token_index, (token_start, token_end) in enumerate(offsets):

            if token_index >= len(adapted_token_predictions):
                break

            if token_end > start and token_start < end:
                adapted_token_predictions[token_index] = 1

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if len(true_labels) != len(original_predictions):
        raise ValueError(
            f"Length mismatch for test_id={row['test_id']}: "
            f"ground truth={len(true_labels)}, "
            f"original={len(original_predictions)}"
        )

    if len(true_labels) != len(adapted_token_predictions):
        raise ValueError(
            f"Length mismatch for test_id={row['test_id']}: "
            f"ground truth={len(true_labels)}, "
            f"adapted={len(adapted_token_predictions)}"
        )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    y_true_all.extend(true_labels)
    y_original_all.extend(original_predictions)
    y_adapted_all.extend(adapted_token_predictions)

    example_summary.append({
        "test_id": row["test_id"],
        "ground_truth_positive_tokens": sum(
            x == 1 for x in true_labels
        ),
        "original_positive_tokens": sum(
            x == 1 for x in original_predictions
        ),
        "adapted_positive_tokens": sum(
            x == 1 for x in adapted_token_predictions
        )
    })


# ------------------------------------------------------------
# Calculate metrics
# ------------------------------------------------------------

original_precision = precision_score(
    y_true_all,
    y_original_all,
    zero_division=0
)

original_recall = recall_score(
    y_true_all,
    y_original_all,
    zero_division=0
)

original_f1 = f1_score(
    y_true_all,
    y_original_all,
    zero_division=0
)


adapted_precision = precision_score(
    y_true_all,
    y_adapted_all,
    zero_division=0
)

adapted_recall = recall_score(
    y_true_all,
    y_adapted_all,
    zero_division=0
)

adapted_f1 = f1_score(
    y_true_all,
    y_adapted_all,
    zero_division=0
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

metrics_df = pd.DataFrame({
    "Model": [
        "Original LettuceDetect",
        "Medical-adapted LettuceDetect"
    ],
    "Precision": [
        original_precision,
        adapted_precision
    ],
    "Recall": [
        original_recall,
        adapted_recall
    ],
    "F1": [
        original_f1,
        adapted_f1
    ]
})

print("\n" + "=" * 80)
print("TOKEN-LEVEL RESULTS")
print("=" * 80)

display(
    metrics_df.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)


# ------------------------------------------------------------
# Basic counts
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOKEN COUNTS")
print("=" * 80)

print("Total ground-truth tokens:", len(y_true_all))
print(
    "Ground-truth hallucinated tokens:",
    sum(x == 1 for x in y_true_all)
)

print(
    "Original predicted hallucinated tokens:",
    sum(x == 1 for x in y_original_all)
)

print(
    "Adapted predicted hallucinated tokens:",
    sum(x == 1 for x in y_adapted_all)
)

print("\n" + "=" * 80)
print("✅ TOKEN-LEVEL EVALUATION COMPLETE")
print("=" * 80)

TOKEN-LEVEL EVALUATION

TOKEN-LEVEL RESULTS


,Model,Precision,Recall,F1
0,Original LettuceDetect,0.1714,0.1418,0.1552
1,Medical-adapted LettuceDetect,0.0000,0.0000,0.0000



TOKEN COUNTS
Total ground-truth tokens: 4615
Ground-truth hallucinated tokens: 550
Original predicted hallucinated tokens: 455
Adapted predicted hallucinated tokens: 0

✅ TOKEN-LEVEL EVALUATION COMPLETE


Cell 76 — Display and save the exact evaluation results

In [80]:
# Cell 76 — Display exact token-level metrics and save results

print("=" * 80)
print("FINAL TOKEN-LEVEL RESULTS")
print("=" * 80)

display(
    metrics_df.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)

print("\nPercentage format:")

for _, row in metrics_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"  Precision: {row['Precision']:.2%}")
    print(f"  Recall:    {row['Recall']:.2%}")
    print(f"  F1:        {row['F1']:.2%}")


# Save metrics
metrics_file = "/kaggle/working/lettucedetect_final_token_metrics.csv"

metrics_df.to_csv(
    metrics_file,
    index=False
)

print("\n" + "=" * 80)
print("Saved:", metrics_file)
print("=" * 80)

FINAL TOKEN-LEVEL RESULTS


,Model,Precision,Recall,F1
0,Original LettuceDetect,0.1714,0.1418,0.1552
1,Medical-adapted LettuceDetect,0.0000,0.0000,0.0000



Percentage format:

Original LettuceDetect
  Precision: 17.14%
  Recall:    14.18%
  F1:        15.52%

Medical-adapted LettuceDetect
  Precision: 0.00%
  Recall:    0.00%
  F1:        0.00%

Saved: /kaggle/working/lettucedetect_final_token_metrics.csv


Cell 77 — Diagnose the fine-tuning training data

In [81]:
# Cell 77 — Diagnose hallucination labels in the training set

import json

print("=" * 80)
print("TRAINING DATA LABEL DIAGNOSTICS")
print("=" * 80)

training_positive_examples = 0
training_hallucinated_tokens = 0
training_answer_tokens = 0

training_rows = []

for i in range(len(annotation_train)):

    row = annotation_train.iloc[i]

    raw_spans = row["hallucination_spans"]

    if not raw_spans or str(raw_spans).strip() in ["", "[]", "nan"]:
        spans = []
    else:
        spans = json.loads(raw_spans)

    # Create a temporary sample
    sample = RagTruthSample(
        prompt=str(row["context"]),
        answer=str(row["answer"]),
        labels=spans,
        split="train",
        task_type="medical_rag"
    )

    temp_dataset = RagTruthDataset(
        samples=[sample],
        tokenizer=evaluation_tokenizer,
        max_length=4096
    )

    item = temp_dataset[0]

    labels = item["labels"]

    answer_labels = labels[labels != -100]

    positive_tokens = int(
        (answer_labels == 1).sum().item()
    )

    training_answer_tokens += len(answer_labels)
    training_hallucinated_tokens += positive_tokens

    if positive_tokens > 0:
        training_positive_examples += 1

        training_rows.append({
            "test_id": row["test_id"],
            "overall_label": row["overall_label"],
            "hallucinated_tokens": positive_tokens,
            "spans": spans
        })


print("Training examples:", len(annotation_train))
print("Training answer tokens:", training_answer_tokens)
print(
    "Training examples with hallucinated tokens:",
    training_positive_examples
)
print(
    "Training hallucinated tokens:",
    training_hallucinated_tokens
)

print("\n" + "=" * 80)
print("POSITIVE TRAINING EXAMPLES")
print("=" * 80)

if training_rows:
    display(pd.DataFrame(training_rows))
else:
    print("No positive training examples found.")

print("\n" + "=" * 80)

if training_hallucinated_tokens > 0:
    print("✅ Training set contains positive hallucination supervision.")
else:
    print("⚠️ Training set contains NO positive hallucination supervision.")

TRAINING DATA LABEL DIAGNOSTICS
Training examples: 140
Training answer tokens: 16009
Training examples with hallucinated tokens: 0
Training hallucinated tokens: 0

POSITIVE TRAINING EXAMPLES
No positive training examples found.

⚠️ Training set contains NO positive hallucination supervision.


Cell 78 — Find potentially hallucinated training examples

In [82]:
# Cell 78 — Find LettuceDetect-flagged candidates in the training split

training_ids = set(annotation_train["test_id"])

training_baseline = baseline_df[
    baseline_df["test_id"].isin(training_ids)
].copy()

training_candidates = training_baseline[
    training_baseline["lettucedetect_spans"].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
].copy()

print("=" * 80)
print("TRAINING CANDIDATES")
print("=" * 80)

print("Training examples:", len(annotation_train))
print(
    "LettuceDetect-flagged training candidates:",
    len(training_candidates)
)

print("\nCandidate test IDs:")
print(training_candidates["test_id"].tolist())

print("\n" + "=" * 80)

display(
    training_candidates[
        [
            "test_id",
            "question",
            "answer",
            "lettucedetect_spans"
        ]
    ].head(20)
)

TRAINING CANDIDATES
Training examples: 140
LettuceDetect-flagged training candidates: 61

Candidate test IDs:
[660, 626, 636, 174, 687, 328, 299, 924, 601, 439, 837, 570, 23, 820, 54, 107, 493, 39, 307, 67, 998, 88, 63, 716, 351, 256, 644, 168, 917, 528, 823, 985, 816, 534, 294, 425, 445, 599, 25, 893, 810, 941, 309, 697, 884, 767, 298, 314, 310, 479, 110, 989, 254, 677, 526, 845, 365, 829, 199, 408, 613]



,test_id,question,answer,lettucedetect_spans
3,660,Does health information exchange reduce redund...,"Based on Evidence 1, which states that the rad...","[{'start': 91, 'end': 116, 'confidence': 0.940..."
6,626,Is fluoroscopy essential for retrieval of lowe...,"No, fluoroscopy is not essential for the retri...","[{'start': 98, 'end': 100, 'confidence': 0.512..."
12,636,Are stroke patients' reports of home blood pre...,The available evidence is insufficient to answ...,"[{'start': 517, 'end': 556, 'confidence': 0.64..."
21,174,Quality of life in lung cancer patients: does ...,The available evidence is insufficient to answ...,"[{'start': 490, 'end': 493, 'confidence': 0.68..."
36,687,Delayed diagnosis of anorectal malformations: ...,"Based on the provided evidence, it appears tha...","[{'start': 953, 'end': 962, 'confidence': 0.54..."
40,328,Volume change of uterine myomas during pregnan...,"No, myomas do not really grow significantly du...","[{'start': 96, 'end': 204, 'confidence': 0.987..."
44,299,Can the prognosis of polymyalgia rheumatica be...,The available evidence is insufficient to pred...,"[{'start': 308, 'end': 433, 'confidence': 0.98..."
46,924,Are intraoperative precursor events associated...,"Yes, intraoperative precursor events such as f...","[{'start': 523, 'end': 543, 'confidence': 0.63..."
47,601,Is ultrasound equal to X-ray in pediatric frac...,"No, ultrasound is not equivalent to X-ray in p...","[{'start': 189, 'end': 215, 'confidence': 0.81..."
48,439,Is an advance care planning model feasible in ...,"Based on the provided evidence, it appears tha...","[{'start': 429, 'end': 494, 'confidence': 0.76..."


Cell 79 — Prepare the training-candidate review queue

In [83]:
# Cell 79 — Prepare compact human-review queue for training candidates

training_candidate_ids = training_candidates["test_id"].tolist()

training_review_queue = annotation_df[
    annotation_df["test_id"].isin(training_candidate_ids)
].copy()

# Preserve the order in which candidates were detected
training_review_queue = (
    training_review_queue
    .set_index("test_id")
    .loc[training_candidate_ids]
    .reset_index()
)

print("=" * 80)
print("TRAINING CANDIDATE REVIEW QUEUE")
print("=" * 80)

print("Candidates:", len(training_review_queue))

print("\nTest IDs:")
print(training_review_queue["test_id"].tolist())

TRAINING CANDIDATE REVIEW QUEUE
Candidates: 61

Test IDs:
[660, 626, 636, 174, 687, 328, 299, 924, 601, 439, 837, 570, 23, 820, 54, 107, 493, 39, 307, 67, 998, 88, 63, 716, 351, 256, 644, 168, 917, 528, 823, 985, 816, 534, 294, 425, 445, 599, 25, 893, 810, 941, 309, 697, 884, 767, 298, 314, 310, 479, 110, 989, 254, 677, 526, 845, 365, 829, 199, 408, 613]


In [84]:
# Cell 80 — Display 5 training candidates at once

def show_training_candidates(start_position, batch_size=5):

    end_position = min(
        start_position + batch_size,
        len(training_review_queue)
    )

    for position in range(start_position, end_position):

        row = training_review_queue.iloc[position]

        print("=" * 100)
        print(
            f"CANDIDATE {position + 1}/{len(training_review_queue)}"
        )
        print(f"TEST ID: {row['test_id']}")
        print("=" * 100)

        print("\nQUESTION:")
        print(row["question"])

        print("\n" + "-" * 100)
        print("RETRIEVED CONTEXT:")
        print(row["context"])

        print("\n" + "-" * 100)
        print("GENERATED ANSWER:")
        print(row["answer"])

        print("\n" + "-" * 100)
        print("REFERENCE ANSWER:")
        print(row["reference_answer"])

        print("\n")


show_training_candidates(3, batch_size=5)

CANDIDATE 4/61
TEST ID: 174

QUESTION:
Quality of life in lung cancer patients: does socioeconomic status matter?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
The aim of this prognostic factor analysis was to investigate if a patient's self-reported health-related quality of life (HRQOL) provided independent prognostic information for survival in non-small cell lung cancer (NSCLC) patients.

To evaluate the impact of patient-prosthesis mismatch (PPM) on survival, functional status, and quality of life (QoL) after aortic valve replacement (AVR) with small prosthesis size in elderly patients.

Approximately 17 percent of participants were of low socioeconomic status. Breastfeeding and signs of infant illness were the most frequently identified concerns by women, regardless of their socioeconomic status. Signs of infant illness and infant care/behavior were the main unmet learning needs. Although few differences i

In [85]:
# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    # ------------------------------------------------------------
    # Candidate 1 — Test ID 660
    # ------------------------------------------------------------
    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "The answer appropriately recognizes that the evidence is insufficient, but it makes an unsupported inference connecting the radiographic scout-image evidence to health information exchange and redundant imaging."
    },

    # ------------------------------------------------------------
    # Candidate 2 — Test ID 626
    # ------------------------------------------------------------
    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "The conclusion is consistent with the reference answer, but the retrieved context does not support it. Evidence 4 concerns lower calyceal stones treated with SWL and renal anatomical factors, not fluoroscopy-free retrieval of lower ureteric stones."
    },

    # ------------------------------------------------------------
    # Candidate 3 — Test ID 636
    # ------------------------------------------------------------
    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether stroke patients can reliably report home blood pressure readings. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 4 — Test ID 174
    # ------------------------------------------------------------
    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether socioeconomic status affects quality of life in lung cancer patients. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 5 — Test ID 687
    # ------------------------------------------------------------
    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations. The answer incorrectly extrapolates from unrelated guidelines concerning thyroid cancer, DCIS, and hospital disaster preparedness."
    },

    # ------------------------------------------------------------
    # Candidate 6 — Test ID 328
    # ------------------------------------------------------------
    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address uterine myoma growth or volume change. The answer makes unsupported claims about myoma growth and uterine blood flow."
    },

    # ------------------------------------------------------------
    # Candidate 7 — Test ID 299
    # ------------------------------------------------------------
    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address prognosis of polymyalgia rheumatica at disease onset. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 8 — Test ID 924
    # ------------------------------------------------------------
    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "Evidence 2 supports an association between preceding hypotensive events and ST/HII events, but Evidence 5 does not establish the claimed connection between hypertension/metabolic syndrome, intraoperative events, and postoperative outcomes."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print(
    "Reviewed candidates:",
    len(human_review_labels_training)
)

print(
    "Test IDs:",
    list(human_review_labels_training.keys())
)

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [
        item["label"]
        for item in human_review_labels_training.values()
    ]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 8
Test IDs: [660, 626, 636, 174, 687, 328, 299, 924]

Label distribution:
partially_supported       3
appropriate_abstention    3
hallucinated              2
Name: count, dtype: int64


In [86]:
show_training_candidates(8, batch_size=5)

CANDIDATE 9/61
TEST ID: 601

QUESTION:
Is ultrasound equal to X-ray in pediatric fracture diagnosis?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
Digital tomosynthesis (DT) is a new X-ray-based imaging technique that allows image enhancement with minimal increase in radiation exposure. The purpose of this study was to compare DT with noncontrast computed tomography (NCCT) and to evaluate its potential role for the follow-up of patients with nephrolithiasis in a nonemergent setting.

We analyzed retrospectively 78 patients with single radiopaque lower calyceal stones treated with SWL. The patients were evaluated 3 months after lithotripsy with a simple abdominal X-ray and a kidney ultrasound scan. The success of the treatment, removal of all fragments, was correlated with renal anatomical factors measured in the pre-treatment intravenous urography: infundibulopelvic angle, lower infundibulum width, lower infundi

In [87]:

# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    # ------------------------------------------------------------
    # Candidate 1 — Test ID 660
    # ------------------------------------------------------------
    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "The answer appropriately recognizes that the evidence is insufficient, but it makes an unsupported inference connecting the radiographic scout-image evidence to health information exchange and redundant imaging."
    },

    # ------------------------------------------------------------
    # Candidate 2 — Test ID 626
    # ------------------------------------------------------------
    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "The conclusion is consistent with the reference answer, but the retrieved context does not support it. Evidence 4 concerns lower calyceal stones treated with SWL and renal anatomical factors, not fluoroscopy-free retrieval of lower ureteric stones."
    },

    # ------------------------------------------------------------
    # Candidate 3 — Test ID 636
    # ------------------------------------------------------------
    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether stroke patients can reliably report home blood pressure readings. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 4 — Test ID 174
    # ------------------------------------------------------------
    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether socioeconomic status affects quality of life in lung cancer patients. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 5 — Test ID 687
    # ------------------------------------------------------------
    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations. The answer incorrectly extrapolates from unrelated guidelines concerning thyroid cancer, DCIS, and hospital disaster preparedness."
    },

    # ------------------------------------------------------------
    # Candidate 6 — Test ID 328
    # ------------------------------------------------------------
    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address uterine myoma growth or volume change. The answer makes unsupported claims about myoma growth and uterine blood flow."
    },

    # ------------------------------------------------------------
    # Candidate 7 — Test ID 299
    # ------------------------------------------------------------
    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address prognosis of polymyalgia rheumatica at disease onset. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 8 — Test ID 924
    # ------------------------------------------------------------
    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "Evidence 2 supports an association between preceding hypotensive events and ST/HII events, but Evidence 5 does not establish the claimed connection between hypertension/metabolic syndrome, intraoperative events, and postoperative outcomes."
    },

    # ------------------------------------------------------------
    # Candidate 9 — Test ID 601
    # ------------------------------------------------------------
    601: {
        "label": "hallucinated",
        "texts": [
            "No, ultrasound is not equivalent to X-ray in pediatric fracture diagnosis.",
            "While both techniques can be used to diagnose fractures, they have different strengths and limitations.",
            "Ultrasound provides real-time images and does not expose the patient to ionizing radiation, making it safer for children.",
            "However, X-rays offer better resolution and detail, which can help identify subtle fractures that may not be visible on ultrasound.",
            "Therefore, the choice between the two depends on the specific needs of the case and the expertise of the healthcare provider."
        ],
        "notes": "The retrieved evidence does not address ultrasound versus X-ray for pediatric fracture diagnosis. The answer introduces unsupported comparative diagnostic claims and therefore hallucinates relative strengths and limitations."
    },

    # ------------------------------------------------------------
    # Candidate 10 — Test ID 439
    # ------------------------------------------------------------
    439: {
        "label": "hallucinated",
        "texts": [
            "it appears that an advance care planning model could be feasible in community palliative care settings such as those described in Evidence 2",
            "This setting suggests a potential environment conducive to implementing advanced care planning models due to factors like access to specialized resources and possibly better communication channels among stakeholders involved in patient care."
        ],
        "notes": "The retrieved evidence discusses palliative/end-of-life care and caregiver burden but does not establish the feasibility of an advance care planning model or provide evidence about specialized resources or communication channels. The answer makes unsupported feasibility claims."
    },

    # ------------------------------------------------------------
    # Candidate 11 — Test ID 837
    # ------------------------------------------------------------
    837: {
        "label": "hallucinated",
        "texts": [
            "neither 99mTc leukocyte scintigraphy nor SBFT (Sigmoidoscopy, Biopsy, Fecal Occult Blood Test) studies appear to be specifically indicated for diagnosing inflammatory bowel disease (IBD) in children.",
            "they could potentially play roles in other aspects of managing or assessing conditions related to IBD in adults.",
            "99mTc leukocyte scintigraphy might be used in adult populations for monitoring inflammation or response to treatment",
            "SBFTs could be utilized in adults for diagnostic purposes such as identifying specific lesions within the colon.",
            "These techniques would need to be adapted or modified for pediatric applications if deemed necessary."
        ],
        "notes": "The retrieved evidence does not establish the utility or lack of utility of 99mTc leukocyte scintigraphy or SBFT for pediatric IBD. The answer also incorrectly expands SBFT and introduces unsupported adult-use claims."
    },

    # ------------------------------------------------------------
    # Candidate 12 — Test ID 570
    # ------------------------------------------------------------
    570: {
        "label": "partially_supported",
        "texts": [
            "Based on the provided evidence, there is no direct mention of dietary habits related to consumption of a high atherogenic diet among European people with type 1 diabetes.",
            "However, the evidence does suggest that bariatric surgery, particularly gastric bypass, can be beneficial for both type 1 and type 2 diabetes.",
            "While the evidence does not specifically address dietary intake, it implies that lifestyle changes, including dietary modifications, may play a role in managing diabetes."
        ],
        "notes": "The answer correctly recognizes that the retrieved evidence does not directly address dietary habits, but it introduces an unsupported inference that dietary or lifestyle modifications may be implied by the bariatric-surgery evidence."
    },

    # ------------------------------------------------------------
    # Candidate 13 — Test ID 23
    # ------------------------------------------------------------
    23: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether nomograms predicting biochemical recurrence also predict clinically relevant prostate cancer outcomes such as prostate cancer-specific mortality. The answer appropriately concludes that the evidence is insufficient."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates:", len(human_review_labels_training))

print("Test IDs:")
print(sorted(human_review_labels_training.keys()))

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [
        item["label"]
        for item in human_review_labels_training.values()
    ]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 13
Test IDs:
[23, 174, 299, 328, 439, 570, 601, 626, 636, 660, 687, 837, 924]

Label distribution:
hallucinated              5
partially_supported       4
appropriate_abstention    4
Name: count, dtype: int64


In [88]:
show_training_candidates(13, batch_size=5)

CANDIDATE 14/61
TEST ID: 820

QUESTION:
The Omega-3 Index: a new risk factor for death from coronary heart disease?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
It is commonly accepted that pathological gambling results from the interaction of multiple risk factors. Among these, dopamine replacement therapy (DRT) prescribed for Parkinson disease can be cited. Another dopamine agonist, aripiprazole, could be a new risk factor. We decided to explore this potential adverse drug reaction (ADR).

Elevated resting heart rate (RHR) is a neglected marker in cardiovascular risk factor studies of sub-Saharan African populations. This study aimed to determine the prevalence of elevated RHR and other risk factors for cardiovascular disease (CVD) and to investigate any associations between RHR and these risk factors in a rural population in Ghana.

Previous studies have reported that the total bilirubin (TB) level is associ

In [89]:

# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    # ------------------------------------------------------------
    # Candidate 1 — Test ID 660
    # ------------------------------------------------------------
    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "The answer appropriately recognizes that the evidence is insufficient, but it makes an unsupported inference connecting the radiographic scout-image evidence to health information exchange and redundant imaging."
    },

    # ------------------------------------------------------------
    # Candidate 2 — Test ID 626
    # ------------------------------------------------------------
    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "The conclusion is consistent with the reference answer, but the retrieved context does not support it. Evidence 4 concerns lower calyceal stones treated with SWL and renal anatomical factors, not fluoroscopy-free retrieval of lower ureteric stones."
    },

    # ------------------------------------------------------------
    # Candidate 3 — Test ID 636
    # ------------------------------------------------------------
    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether stroke patients can reliably report home blood pressure readings. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 4 — Test ID 174
    # ------------------------------------------------------------
    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether socioeconomic status affects quality of life in lung cancer patients. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 5 — Test ID 687
    # ------------------------------------------------------------
    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations. The answer incorrectly extrapolates from unrelated guidelines concerning thyroid cancer, DCIS, and hospital disaster preparedness."
    },

    # ------------------------------------------------------------
    # Candidate 6 — Test ID 328
    # ------------------------------------------------------------
    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address uterine myoma growth or volume change. The answer makes unsupported claims about myoma growth and uterine blood flow."
    },

    # ------------------------------------------------------------
    # Candidate 7 — Test ID 299
    # ------------------------------------------------------------
    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address prognosis of polymyalgia rheumatica at disease onset. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 8 — Test ID 924
    # ------------------------------------------------------------
    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "Evidence 2 supports an association between preceding hypotensive events and ST/HII events, but Evidence 5 does not establish the claimed connection between hypertension/metabolic syndrome, intraoperative events, and postoperative outcomes."
    },

    # ------------------------------------------------------------
    # Candidate 9 — Test ID 601
    # ------------------------------------------------------------
    601: {
        "label": "hallucinated",
        "texts": [
            "No, ultrasound is not equivalent to X-ray in pediatric fracture diagnosis.",
            "While both techniques can be used to diagnose fractures, they have different strengths and limitations.",
            "Ultrasound provides real-time images and does not expose the patient to ionizing radiation, making it safer for children.",
            "However, X-rays offer better resolution and detail, which can help identify subtle fractures that may not be visible on ultrasound.",
            "Therefore, the choice between the two depends on the specific needs of the case and the expertise of the healthcare provider."
        ],
        "notes": "The retrieved evidence does not address ultrasound versus X-ray for pediatric fracture diagnosis. The answer introduces unsupported comparative diagnostic claims and therefore hallucinates relative strengths and limitations."
    },

    # ------------------------------------------------------------
    # Candidate 10 — Test ID 439
    # ------------------------------------------------------------
    439: {
        "label": "hallucinated",
        "texts": [
            "it appears that an advance care planning model could be feasible in community palliative care settings such as those described in Evidence 2",
            "This setting suggests a potential environment conducive to implementing advanced care planning models due to factors like access to specialized resources and possibly better communication channels among stakeholders involved in patient care."
        ],
        "notes": "The retrieved evidence discusses palliative/end-of-life care and caregiver burden but does not establish the feasibility of an advance care planning model or provide evidence about specialized resources or communication channels. The answer makes unsupported feasibility claims."
    },

    # ------------------------------------------------------------
    # Candidate 11 — Test ID 837
    # ------------------------------------------------------------
    837: {
        "label": "hallucinated",
        "texts": [
            "neither 99mTc leukocyte scintigraphy nor SBFT (Sigmoidoscopy, Biopsy, Fecal Occult Blood Test) studies appear to be specifically indicated for diagnosing inflammatory bowel disease (IBD) in children.",
            "they could potentially play roles in other aspects of managing or assessing conditions related to IBD in adults.",
            "99mTc leukocyte scintigraphy might be used in adult populations for monitoring inflammation or response to treatment",
            "SBFTs could be utilized in adults for diagnostic purposes such as identifying specific lesions within the colon.",
            "These techniques would need to be adapted or modified for pediatric applications if deemed necessary."
        ],
        "notes": "The retrieved evidence does not establish the utility or lack of utility of 99mTc leukocyte scintigraphy or SBFT for pediatric IBD. The answer also incorrectly expands SBFT and introduces unsupported adult-use claims."
    },

    # ------------------------------------------------------------
    # Candidate 12 — Test ID 570
    # ------------------------------------------------------------
    570: {
        "label": "partially_supported",
        "texts": [
            "Based on the provided evidence, there is no direct mention of dietary habits related to consumption of a high atherogenic diet among European people with type 1 diabetes.",
            "However, the evidence does suggest that bariatric surgery, particularly gastric bypass, can be beneficial for both type 1 and type 2 diabetes.",
            "While the evidence does not specifically address dietary intake, it implies that lifestyle changes, including dietary modifications, may play a role in managing diabetes."
        ],
        "notes": "The answer correctly recognizes that the retrieved evidence does not directly address dietary habits, but it introduces an unsupported inference that dietary or lifestyle modifications may be implied by the bariatric-surgery evidence."
    },

    # ------------------------------------------------------------
    # Candidate 13 — Test ID 23
    # ------------------------------------------------------------
    23: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether nomograms predicting biochemical recurrence also predict clinically relevant prostate cancer outcomes such as prostate cancer-specific mortality. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Candidate 14 — Test ID 820
    # ------------------------------------------------------------
    820: {
        "label": "hallucinated",
        "texts": [
            "The Omega-3 Index may serve as a new risk factor for death from coronary heart disease.",
            "This conclusion is drawn from the observation that elevated levels of omega-3 fatty acids in the blood have been linked to reduced risk of coronary heart disease events.",
            "the evidence suggests that maintaining adequate levels of omega-3s through diet or supplementation might help reduce the risk of dying from coronary heart disease.",
            "Additionally, the Omega-3 Index has been shown to correlate positively with overall health outcomes, further supporting its potential role as a risk factor."
        ],
        "notes": "The retrieved evidence does not mention the Omega-3 Index or omega-3 fatty acids. The answer introduces unsupported claims about omega-3 levels, coronary heart disease risk, supplementation, and overall health."
    },

    # ------------------------------------------------------------
    # Candidate 15 — Test ID 54
    # ------------------------------------------------------------
    54: {
        "label": "hallucinated",
        "texts": [
            "The sternal fracture in growing children is indeed a rare and often overlooked fracture due to its rarity and the difficulty in diagnosing it accurately.",
            "It can occur without any obvious trauma and may present with symptoms such as chest pain, shortness of breath, or discomfort during deep breathing.",
            "Growing children's bones are still developing, making them susceptible to fractures even from minor traumas.",
            "Unlike adults, whose bones are more mature and stronger, children's bones are more prone to fractures because they are still forming and remodeling.",
            "Therefore, pediatricians should be vigilant about potential fractures in young patients, especially those under the age of 10, where the risk of fractures increases significantly.",
            "Early diagnosis and treatment are crucial to ensure proper healing and avoid long-term complications."
        ],
        "notes": "The retrieved evidence is unrelated to pediatric sternal fractures. The answer introduces unsupported claims about incidence, trauma, symptoms, pediatric bone susceptibility, age-specific risk, and treatment."
    },

    # ------------------------------------------------------------
    # Candidate 16 — Test ID 107
    # ------------------------------------------------------------
    107: {
        "label": "hallucinated",
        "texts": [
            "the evidence suggests that stroke survivors might have different perceptions about electronic stability control (ESC) compared to non-stroke drivers.",
            "Therefore, while the DOT data does not directly address stroke survivors' experiences with ESC, the self-reported data indicates that stroke survivors may perceive ESC differently due to potential changes in driving habits influenced by the technology.",
            "This implies that both sources of data could yield insights into how stroke survivors interact with and respond to safety technologies like ESC"
        ],
        "notes": "The retrieved evidence does not compare Department of Transportation records with self-reported motor vehicle collisions or driving convictions in stroke survivors. The answer incorrectly introduces ESC perceptions and driving behavior claims."
    },

    # ------------------------------------------------------------
    # Candidate 17 — Test ID 493
    # ------------------------------------------------------------
    493: {
        "label": "hallucinated",
        "texts": [
            "No, amoxapine is not an atypical antipsychotic.",
            "Amoxapine is a tricyclic antidepressant used primarily for treating depression and anxiety.",
            "Atypical antipsychotics refer to newer drugs like olanzapine, quetiapine, risperidone, etc., which were developed after the first generation of antipsychotics (typical antipsychotics).",
            "Based on the provided evidence, amoxapine does not fall into the category of atypical antipsychotics."
        ],
        "notes": "The retrieved evidence does not mention amoxapine or its pharmacological classification. The answer introduces unsupported drug-classification claims."
    },

    # ------------------------------------------------------------
    # Candidate 18 — Test ID 39
    # ------------------------------------------------------------
    39: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether diabetes mellitus influences the diagnostic accuracy of FDG-PET in cervical cancer. The answer appropriately concludes that the evidence is insufficient."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates:", len(human_review_labels_training))

print("\nTest IDs:")
print(sorted(human_review_labels_training.keys()))

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [
        item["label"]
        for item in human_review_labels_training.values()
    ]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 18

Test IDs:
[23, 39, 54, 107, 174, 299, 328, 439, 493, 570, 601, 626, 636, 660, 687, 820, 837, 924]

Label distribution:
hallucinated              9
appropriate_abstention    5
partially_supported       4
Name: count, dtype: int64


In [90]:
show_training_candidates(18, batch_size=5)

CANDIDATE 19/61
TEST ID: 307

QUESTION:
Does lunar position influence the time of delivery?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
Mechanically ventilated patients experience profound stress. Interventions are needed to ameliorate stress that does not cause adverse effects. The purpose of this study was to explore the influence of music on stress in a sample of patients over the duration of ventilatory support.RESEARCH METHODOLOGY/

The influence of positron emission tomography in the management of recurrent rectal cancer is well established but its role in primary rectal cancer remains uncertain. This study therefore prospectively assesses the impact of position emission tomography scanning on the management of primary rectal cancer.

Eye movements in the supine position and the prone position were not in a symmetric fashion. Nystagmus in the left-ear-down position and the right-ear-down position were no

In [91]:

# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    # ------------------------------------------------------------
    # Test ID 660
    # ------------------------------------------------------------
    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "The answer appropriately recognizes that the evidence is insufficient, but it makes an unsupported inference connecting the radiographic scout-image evidence to health information exchange and redundant imaging."
    },

    # ------------------------------------------------------------
    # Test ID 626
    # ------------------------------------------------------------
    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "The conclusion is consistent with the reference answer, but the retrieved context does not support it. Evidence 4 concerns lower calyceal stones treated with SWL and renal anatomical factors, not fluoroscopy-free retrieval of lower ureteric stones."
    },

    # ------------------------------------------------------------
    # Test ID 636
    # ------------------------------------------------------------
    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether stroke patients can reliably report home blood pressure readings. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 174
    # ------------------------------------------------------------
    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether socioeconomic status affects quality of life in lung cancer patients. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 687
    # ------------------------------------------------------------
    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations. The answer incorrectly extrapolates from unrelated guidelines concerning thyroid cancer, DCIS, and hospital disaster preparedness."
    },

    # ------------------------------------------------------------
    # Test ID 328
    # ------------------------------------------------------------
    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address uterine myoma growth or volume change. The answer makes unsupported claims about myoma growth and uterine blood flow."
    },

    # ------------------------------------------------------------
    # Test ID 299
    # ------------------------------------------------------------
    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address prognosis of polymyalgia rheumatica at disease onset. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 924
    # ------------------------------------------------------------
    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "Evidence 2 supports an association between preceding hypotensive events and ST/HII events, but Evidence 5 does not establish the claimed connection between hypertension/metabolic syndrome, intraoperative events, and postoperative outcomes."
    },

    # ------------------------------------------------------------
    # Test ID 601
    # ------------------------------------------------------------
    601: {
        "label": "hallucinated",
        "texts": [
            "No, ultrasound is not equivalent to X-ray in pediatric fracture diagnosis.",
            "While both techniques can be used to diagnose fractures, they have different strengths and limitations.",
            "Ultrasound provides real-time images and does not expose the patient to ionizing radiation, making it safer for children.",
            "However, X-rays offer better resolution and detail, which can help identify subtle fractures that may not be visible on ultrasound.",
            "Therefore, the choice between the two depends on the specific needs of the case and the expertise of the healthcare provider."
        ],
        "notes": "The retrieved evidence does not address ultrasound versus X-ray for pediatric fracture diagnosis. The answer introduces unsupported comparative diagnostic claims."
    },

    # ------------------------------------------------------------
    # Test ID 439
    # ------------------------------------------------------------
    439: {
        "label": "hallucinated",
        "texts": [
            "it appears that an advance care planning model could be feasible in community palliative care settings such as those described in Evidence 2",
            "This setting suggests a potential environment conducive to implementing advanced care planning models due to factors like access to specialized resources and possibly better communication channels among stakeholders involved in patient care."
        ],
        "notes": "The retrieved evidence discusses palliative/end-of-life care and caregiver burden but does not establish feasibility of an advance care planning model or provide evidence about specialized resources or communication channels."
    },

    # ------------------------------------------------------------
    # Test ID 837
    # ------------------------------------------------------------
    837: {
        "label": "hallucinated",
        "texts": [
            "neither 99mTc leukocyte scintigraphy nor SBFT (Sigmoidoscopy, Biopsy, Fecal Occult Blood Test) studies appear to be specifically indicated for diagnosing inflammatory bowel disease (IBD) in children.",
            "they could potentially play roles in other aspects of managing or assessing conditions related to IBD in adults.",
            "99mTc leukocyte scintigraphy might be used in adult populations for monitoring inflammation or response to treatment",
            "SBFTs could be utilized in adults for diagnostic purposes such as identifying specific lesions within the colon.",
            "These techniques would need to be adapted or modified for pediatric applications if deemed necessary."
        ],
        "notes": "The retrieved evidence does not establish the utility or lack of utility of 99mTc leukocyte scintigraphy or SBFT for pediatric IBD. The answer also incorrectly expands SBFT and introduces unsupported adult-use claims."
    },

    # ------------------------------------------------------------
    # Test ID 570
    # ------------------------------------------------------------
    570: {
        "label": "partially_supported",
        "texts": [
            "However, the evidence does suggest that bariatric surgery, particularly gastric bypass, can be beneficial for both type 1 and type 2 diabetes.",
            "While the evidence does not specifically address dietary intake, it implies that lifestyle changes, including dietary modifications, may play a role in managing diabetes."
        ],
        "notes": "The answer correctly recognizes that the retrieved evidence does not directly address dietary habits, but then introduces an unsupported inference about dietary or lifestyle modifications."
    },

    # ------------------------------------------------------------
    # Test ID 23
    # ------------------------------------------------------------
    23: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether nomograms predicting biochemical recurrence also predict clinically relevant prostate cancer outcomes. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 820
    # ------------------------------------------------------------
    820: {
        "label": "hallucinated",
        "texts": [
            "The Omega-3 Index may serve as a new risk factor for death from coronary heart disease.",
            "This conclusion is drawn from the observation that elevated levels of omega-3 fatty acids in the blood have been linked to reduced risk of coronary heart disease events.",
            "the evidence suggests that maintaining adequate levels of omega-3s through diet or supplementation might help reduce the risk of dying from coronary heart disease.",
            "Additionally, the Omega-3 Index has been shown to correlate positively with overall health outcomes, further supporting its potential role as a risk factor."
        ],
        "notes": "The retrieved evidence does not mention the Omega-3 Index or omega-3 fatty acids. The answer introduces unsupported claims."
    },

    # ------------------------------------------------------------
    # Test ID 54
    # ------------------------------------------------------------
    54: {
        "label": "hallucinated",
        "texts": [
            "The sternal fracture in growing children is indeed a rare and often overlooked fracture due to its rarity and the difficulty in diagnosing it accurately.",
            "It can occur without any obvious trauma and may present with symptoms such as chest pain, shortness of breath, or discomfort during deep breathing.",
            "Growing children's bones are still developing, making them susceptible to fractures even from minor traumas.",
            "Unlike adults, whose bones are more mature and stronger, children's bones are more prone to fractures because they are still forming and remodeling.",
            "Therefore, pediatricians should be vigilant about potential fractures in young patients, especially those under the age of 10, where the risk of fractures increases significantly.",
            "Early diagnosis and treatment are crucial to ensure proper healing and avoid long-term complications."
        ],
        "notes": "The retrieved evidence is unrelated to pediatric sternal fractures. The answer introduces unsupported claims about incidence, trauma, symptoms, pediatric bone susceptibility, age-specific risk, and treatment."
    },

    # ------------------------------------------------------------
    # Test ID 107
    # ------------------------------------------------------------
    107: {
        "label": "hallucinated",
        "texts": [
            "the evidence suggests that stroke survivors might have different perceptions about electronic stability control (ESC) compared to non-stroke drivers.",
            "Therefore, while the DOT data does not directly address stroke survivors' experiences with ESC, the self-reported data indicates that stroke survivors may perceive ESC differently due to potential changes in driving habits influenced by the technology.",
            "This implies that both sources of data could yield insights into how stroke survivors interact with and respond to safety technologies like ESC"
        ],
        "notes": "The retrieved evidence does not compare Department of Transportation records with self-reported motor vehicle collisions or driving convictions in stroke survivors. The answer incorrectly introduces ESC perceptions and driving behavior claims."
    },

    # ------------------------------------------------------------
    # Test ID 493
    # ------------------------------------------------------------
    493: {
        "label": "hallucinated",
        "texts": [
            "No, amoxapine is not an atypical antipsychotic.",
            "Amoxapine is a tricyclic antidepressant used primarily for treating depression and anxiety.",
            "Atypical antipsychotics refer to newer drugs like olanzapine, quetiapine, risperidone, etc., which were developed after the first generation of antipsychotics (typical antipsychotics).",
            "Based on the provided evidence, amoxapine does not fall into the category of atypical antipsychotics."
        ],
        "notes": "The retrieved evidence does not mention amoxapine or its pharmacological classification. The answer introduces unsupported drug-classification claims."
    },

    # ------------------------------------------------------------
    # Test ID 39
    # ------------------------------------------------------------
    39: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether diabetes mellitus influences the diagnostic accuracy of FDG-PET in cervical cancer. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 307
    # ------------------------------------------------------------
    307: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether lunar position influences the time of delivery. The answer correctly identifies that the provided evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 67
    # ------------------------------------------------------------
    67: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether instrumental activities of daily living predict dementia at 1- and 2-year follow-up. The answer correctly concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 998
    # ------------------------------------------------------------
    998: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether autologous platelet-rich plasma gel enhances healing after surgical extraction of mandibular third molars. The answer correctly concludes that the available evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 88
    # ------------------------------------------------------------
    88: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether multi-modal cervical physical therapy improves tinnitus in patients with cervicogenic somatic tinnitus. The answer correctly concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 63
    # ------------------------------------------------------------
    63: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence concerns manual ventilation and airway/resuscitation procedures but does not directly establish whether a four-diagram manual enables laypersons to operate the Laryngeal Mask Supreme. The answer appropriately concludes that the evidence is insufficient."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates:", len(human_review_labels_training))

print("\nTest IDs:")
print(sorted(human_review_labels_training.keys()))

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [
        item["label"]
        for item in human_review_labels_training.values()
    ]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 23

Test IDs:
[23, 39, 54, 63, 67, 88, 107, 174, 299, 307, 328, 439, 493, 570, 601, 626, 636, 660, 687, 820, 837, 924, 998]

Label distribution:
appropriate_abstention    10
hallucinated               9
partially_supported        4
Name: count, dtype: int64


In [92]:
show_training_candidates(23, batch_size=5)

CANDIDATE 24/61
TEST ID: 716

QUESTION:
Can serum be used for analyzing the EGFR mutation status in patients with advanced non-small cell lung cancer?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
It remains controversial whether there is a gender difference in survival of patients with resected non-small cell lung cancer.

The aim of this prognostic factor analysis was to investigate if a patient's self-reported health-related quality of life (HRQOL) provided independent prognostic information for survival in non-small cell lung cancer (NSCLC) patients.

We retrospectively analyzed 2770 patients (1689 men and 1081 women) with non-small cell lung cancer who underwent pulmonary resection between 1995 and 2005 at the National Cancer Center Hospital, Tokyo. A gender difference in survival was studied in all patients, in those divided according to histology or pathologic stage, and in propensity-matched gender pairs

In [93]:

# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    # ------------------------------------------------------------
    # Test ID 660
    # ------------------------------------------------------------
    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "The answer appropriately recognizes that the evidence is insufficient, but it makes an unsupported inference connecting the radiographic scout-image evidence to health information exchange and redundant imaging."
    },

    # ------------------------------------------------------------
    # Test ID 626
    # ------------------------------------------------------------
    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "The conclusion is consistent with the reference answer, but the retrieved context does not support it. Evidence 4 concerns lower calyceal stones treated with SWL and renal anatomical factors, not fluoroscopy-free retrieval of lower ureteric stones."
    },

    # ------------------------------------------------------------
    # Test ID 636
    # ------------------------------------------------------------
    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether stroke patients can reliably report home blood pressure readings. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 174
    # ------------------------------------------------------------
    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether socioeconomic status affects quality of life in lung cancer patients. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 687
    # ------------------------------------------------------------
    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations. The answer incorrectly extrapolates from unrelated guidelines concerning thyroid cancer, DCIS, and hospital disaster preparedness."
    },

    # ------------------------------------------------------------
    # Test ID 328
    # ------------------------------------------------------------
    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address uterine myoma growth or volume change. The answer makes unsupported claims about myoma growth and uterine blood flow."
    },

    # ------------------------------------------------------------
    # Test ID 299
    # ------------------------------------------------------------
    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address prognosis of polymyalgia rheumatica at disease onset. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 924
    # ------------------------------------------------------------
    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "Evidence 2 supports an association between preceding hypotensive events and ST/HII events, but Evidence 5 does not establish the claimed connection between hypertension/metabolic syndrome, intraoperative events, and postoperative outcomes."
    },

    # ------------------------------------------------------------
    # Test ID 601
    # ------------------------------------------------------------
    601: {
        "label": "hallucinated",
        "texts": [
            "No, ultrasound is not equivalent to X-ray in pediatric fracture diagnosis.",
            "While both techniques can be used to diagnose fractures, they have different strengths and limitations.",
            "Ultrasound provides real-time images and does not expose the patient to ionizing radiation, making it safer for children.",
            "However, X-rays offer better resolution and detail, which can help identify subtle fractures that may not be visible on ultrasound.",
            "Therefore, the choice between the two depends on the specific needs of the case and the expertise of the healthcare provider."
        ],
        "notes": "The retrieved evidence does not address ultrasound versus X-ray for pediatric fracture diagnosis. The answer introduces unsupported comparative diagnostic claims."
    },

    # ------------------------------------------------------------
    # Test ID 439
    # ------------------------------------------------------------
    439: {
        "label": "hallucinated",
        "texts": [
            "it appears that an advance care planning model could be feasible in community palliative care settings such as those described in Evidence 2",
            "This setting suggests a potential environment conducive to implementing advanced care planning models due to factors like access to specialized resources and possibly better communication channels among stakeholders involved in patient care."
        ],
        "notes": "The retrieved evidence discusses palliative/end-of-life care and caregiver burden but does not establish feasibility of an advance care planning model or provide evidence about specialized resources or communication channels."
    },

    # ------------------------------------------------------------
    # Test ID 837
    # ------------------------------------------------------------
    837: {
        "label": "hallucinated",
        "texts": [
            "neither 99mTc leukocyte scintigraphy nor SBFT (Sigmoidoscopy, Biopsy, Fecal Occult Blood Test) studies appear to be specifically indicated for diagnosing inflammatory bowel disease (IBD) in children.",
            "they could potentially play roles in other aspects of managing or assessing conditions related to IBD in adults.",
            "99mTc leukocyte scintigraphy might be used in adult populations for monitoring inflammation or response to treatment",
            "SBFTs could be utilized in adults for diagnostic purposes such as identifying specific lesions within the colon.",
            "These techniques would need to be adapted or modified for pediatric applications if deemed necessary."
        ],
        "notes": "The retrieved evidence does not establish the utility or lack of utility of 99mTc leukocyte scintigraphy or SBFT for pediatric IBD. The answer also incorrectly expands SBFT and introduces unsupported adult-use claims."
    },

    # ------------------------------------------------------------
    # Test ID 570
    # ------------------------------------------------------------
    570: {
        "label": "partially_supported",
        "texts": [
            "However, the evidence does suggest that bariatric surgery, particularly gastric bypass, can be beneficial for both type 1 and type 2 diabetes.",
            "While the evidence does not specifically address dietary intake, it implies that lifestyle changes, including dietary modifications, may play a role in managing diabetes."
        ],
        "notes": "The answer correctly recognizes that the retrieved evidence does not directly address dietary habits, but then introduces an unsupported inference about dietary or lifestyle modifications."
    },

    # ------------------------------------------------------------
    # Test ID 23
    # ------------------------------------------------------------
    23: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether nomograms predicting biochemical recurrence also predict clinically relevant prostate cancer outcomes. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 820
    # ------------------------------------------------------------
    820: {
        "label": "hallucinated",
        "texts": [
            "The Omega-3 Index may serve as a new risk factor for death from coronary heart disease.",
            "This conclusion is drawn from the observation that elevated levels of omega-3 fatty acids in the blood have been linked to reduced risk of coronary heart disease events.",
            "the evidence suggests that maintaining adequate levels of omega-3s through diet or supplementation might help reduce the risk of dying from coronary heart disease.",
            "Additionally, the Omega-3 Index has been shown to correlate positively with overall health outcomes, further supporting its potential role as a risk factor."
        ],
        "notes": "The retrieved evidence does not mention the Omega-3 Index or omega-3 fatty acids. The answer introduces unsupported claims."
    },

    # ------------------------------------------------------------
    # Test ID 54
    # ------------------------------------------------------------
    54: {
        "label": "hallucinated",
        "texts": [
            "The sternal fracture in growing children is indeed a rare and often overlooked fracture due to its rarity and the difficulty in diagnosing it accurately.",
            "It can occur without any obvious trauma and may present with symptoms such as chest pain, shortness of breath, or discomfort during deep breathing.",
            "Growing children's bones are still developing, making them susceptible to fractures even from minor traumas.",
            "Unlike adults, whose bones are more mature and stronger, children's bones are more prone to fractures because they are still forming and remodeling.",
            "Therefore, pediatricians should be vigilant about potential fractures in young patients, especially those under the age of 10, where the risk of fractures increases significantly.",
            "Early diagnosis and treatment are crucial to ensure proper healing and avoid long-term complications."
        ],
        "notes": "The retrieved evidence is unrelated to pediatric sternal fractures. The answer introduces unsupported claims about incidence, trauma, symptoms, pediatric bone susceptibility, age-specific risk, and treatment."
    },

    # ------------------------------------------------------------
    # Test ID 107
    # ------------------------------------------------------------
    107: {
        "label": "hallucinated",
        "texts": [
            "the evidence suggests that stroke survivors might have different perceptions about electronic stability control (ESC) compared to non-stroke drivers.",
            "Therefore, while the DOT data does not directly address stroke survivors' experiences with ESC, the self-reported data indicates that stroke survivors may perceive ESC differently due to potential changes in driving habits influenced by the technology.",
            "This implies that both sources of data could yield insights into how stroke survivors interact with and respond to safety technologies like ESC"
        ],
        "notes": "The retrieved evidence does not compare Department of Transportation records with self-reported motor vehicle collisions or driving convictions in stroke survivors. The answer incorrectly introduces ESC perceptions and driving behavior claims."
    },

    # ------------------------------------------------------------
    # Test ID 493
    # ------------------------------------------------------------
    493: {
        "label": "hallucinated",
        "texts": [
            "No, amoxapine is not an atypical antipsychotic.",
            "Amoxapine is a tricyclic antidepressant used primarily for treating depression and anxiety.",
            "Atypical antipsychotics refer to newer drugs like olanzapine, quetiapine, risperidone, etc., which were developed after the first generation of antipsychotics (typical antipsychotics).",
            "Based on the provided evidence, amoxapine does not fall into the category of atypical antipsychotics."
        ],
        "notes": "The retrieved evidence does not mention amoxapine or its pharmacological classification. The answer introduces unsupported drug-classification claims."
    },

    # ------------------------------------------------------------
    # Test ID 39
    # ------------------------------------------------------------
    39: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether diabetes mellitus influences the diagnostic accuracy of FDG-PET in cervical cancer. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 307
    # ------------------------------------------------------------
    307: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether lunar position influences the time of delivery. The answer correctly identifies that the provided evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 67
    # ------------------------------------------------------------
    67: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether instrumental activities of daily living predict dementia at 1- and 2-year follow-up. The answer correctly concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 998
    # ------------------------------------------------------------
    998: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether autologous platelet-rich plasma gel enhances healing after surgical extraction of mandibular third molars. The answer correctly concludes that the available evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 88
    # ------------------------------------------------------------
    88: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether multi-modal cervical physical therapy improves tinnitus in patients with cervicogenic somatic tinnitus. The answer correctly concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 63
    # ------------------------------------------------------------
    63: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence concerns manual ventilation and airway/resuscitation procedures but does not directly establish whether a four-diagram manual enables laypersons to operate the Laryngeal Mask Supreme. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 716
    # ------------------------------------------------------------
    716: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether serum can be used to analyze EGFR mutation status in advanced non-small cell lung cancer. The answer appropriately concludes that the available evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 351
    # ------------------------------------------------------------
    351: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether three-phase bone scintigraphy is a guideline method for determining amputation levels in diabetic foot. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 256
    # ------------------------------------------------------------
    256: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether breast cancer prognosis is inherited. The answer appropriately concludes that the available evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 644
    # ------------------------------------------------------------
    644: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether dentists can recognize manipulated digital radiographs. The answer appropriately concludes that the evidence is insufficient."
    },

    # ------------------------------------------------------------
    # Test ID 168
    # ------------------------------------------------------------
    168: {
        "label": "hallucinated",
        "texts": [
            "there is no clear necessity to routinely insert a nasobiliary drainage tube following endoscopic clearance of the common bile duct in patients with choledocholithiasis-induced cholangitis",
            "the studies do not support routine insertion of a nasobiliary drainage tube as a preventive measure",
            "the decision should be made on a case-by-case basis considering individual patient factors and clinical context"
        ],
        "notes": "The retrieved evidence discusses bile duct sludge, common bile duct stone recurrence, and ERCP outcomes, but it does not evaluate routine nasobiliary drainage after endoscopic clearance in cholangitis. The answer makes an unsupported management conclusion."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates:", len(human_review_labels_training))

print("\nTest IDs:")
print(sorted(human_review_labels_training.keys()))

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [
        item["label"]
        for item in human_review_labels_training.values()
    ]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 28

Test IDs:
[23, 39, 54, 63, 67, 88, 107, 168, 174, 256, 299, 307, 328, 351, 439, 493, 570, 601, 626, 636, 644, 660, 687, 716, 820, 837, 924, 998]

Label distribution:
appropriate_abstention    14
hallucinated              10
partially_supported        4
Name: count, dtype: int64


In [94]:
show_training_candidates(28, batch_size=5)

CANDIDATE 29/61
TEST ID: 917

QUESTION:
Does radiotherapy around the time of pregnancy for Hodgkin's disease modify the risk of breast cancer?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
The protraction of external beam radiotherapy (RT) time is detrimental in several disease sites. In prostate cancer, the overall treatment time can be considerable, as can the potential for treatment breaks. We evaluated the effect of elapsed treatment time on outcome after RT for prostate cancer.

Three themes were identified: a) Supporting one's wife; b) Marital needs and obligations; c) Constrained by a culture of destiny and shame. The first theme was built on men's feelings of responsibility for the family's health and well-being, their experiences of encouraging their wives to seek health care and their providing counselling and instrumental support. The second theme emerged from men's views about other men's rejection o

In [95]:

# Cell 81 — Store human-reviewed training annotations

human_review_labels_training = {

    660: {
        "label": "partially_supported",
        "texts": [
            "we can infer that health information exchange may not necessarily reduce redundant imaging as it depends on how the information is shared and utilized"
        ],
        "notes": "Unsupported inference about health information exchange and redundant imaging."
    },

    626: {
        "label": "partially_supported",
        "texts": [
            "According to Evidence 4, which analyzed 78 patients with single radiopaque lower calyceal stones treated with SWL, the success of the treatment was correlated with renal anatomical factors measured in the pre-treatment intravenous urography. These factors included infundibulopelvic angle, lower infundibulum width, lower infundibulum length, ratio length/width, infundibulum height, and number of minor calyces in the lower calyceal group. Therefore, while fluoroscopy may play a role in guiding procedures, it is not considered essential for the retrieval of lower ureteric stones as evidenced by these studies."
        ],
        "notes": "Conclusion not directly supported by the retrieved evidence; Evidence 4 concerns lower calyceal stones and SWL."
    },

    636: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    174: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    687: {
        "label": "hallucinated",
        "texts": [
            "it appears that current guidelines for diagnosing anorectal malformations may not be entirely sufficient",
            "Therefore, further refinement and adaptation of existing guidelines may be necessary to ensure they remain effective in various clinical scenarios."
        ],
        "notes": "The retrieved evidence does not address diagnosis of anorectal malformations."
    },

    328: {
        "label": "hallucinated",
        "texts": [
            "No, myomas do not really grow significantly during pregnancy according to the evidence provided.",
            "The evidence suggests that there may be some changes in uterine blood flow patterns during early pregnancy, but no substantial growth in uterine size or volume has been observed."
        ],
        "notes": "The retrieved evidence does not address myoma growth or uterine blood flow in pregnancy."
    },

    299: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    924: {
        "label": "partially_supported",
        "texts": [
            "Furthermore, studies like Evidence 5 suggest that patients with high blood pressure and metabolic syndrome have a higher incidence of major adverse cardiovascular events following surgery, suggesting a link between intraoperative events and postoperative outcomes."
        ],
        "notes": "The answer introduces a connection not established by the retrieved evidence."
    },

    601: {
        "label": "hallucinated",
        "texts": [
            "No, ultrasound is not equivalent to X-ray in pediatric fracture diagnosis.",
            "While both techniques can be used to diagnose fractures, they have different strengths and limitations.",
            "Ultrasound provides real-time images and does not expose the patient to ionizing radiation, making it safer for children.",
            "However, X-rays offer better resolution and detail, which can help identify subtle fractures that may not be visible on ultrasound.",
            "Therefore, the choice between the two depends on the specific needs of the case and the expertise of the healthcare provider."
        ],
        "notes": "The retrieved evidence does not address ultrasound versus X-ray for pediatric fracture diagnosis."
    },

    439: {
        "label": "hallucinated",
        "texts": [
            "it appears that an advance care planning model could be feasible in community palliative care settings such as those described in Evidence 2",
            "This setting suggests a potential environment conducive to implementing advanced care planning models due to factors like access to specialized resources and possibly better communication channels among stakeholders involved in patient care."
        ],
        "notes": "Feasibility and communication-resource claims are unsupported by the retrieved context."
    },

    837: {
        "label": "hallucinated",
        "texts": [
            "neither 99mTc leukocyte scintigraphy nor SBFT (Sigmoidoscopy, Biopsy, Fecal Occult Blood Test) studies appear to be specifically indicated for diagnosing inflammatory bowel disease (IBD) in children.",
            "they could potentially play roles in other aspects of managing or assessing conditions related to IBD in adults.",
            "99mTc leukocyte scintigraphy might be used in adult populations for monitoring inflammation or response to treatment",
            "SBFTs could be utilized in adults for diagnostic purposes such as identifying specific lesions within the colon.",
            "These techniques would need to be adapted or modified for pediatric applications if deemed necessary."
        ],
        "notes": "The context does not support these pediatric IBD or adult-use claims."
    },

    570: {
        "label": "partially_supported",
        "texts": [
            "However, the evidence does suggest that bariatric surgery, particularly gastric bypass, can be beneficial for both type 1 and type 2 diabetes.",
            "While the evidence does not specifically address dietary intake, it implies that lifestyle changes, including dietary modifications, may play a role in managing diabetes."
        ],
        "notes": "The answer extends beyond what the retrieved evidence establishes."
    },

    23: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    820: {
        "label": "hallucinated",
        "texts": [
            "The Omega-3 Index may serve as a new risk factor for death from coronary heart disease.",
            "This conclusion is drawn from the observation that elevated levels of omega-3 fatty acids in the blood have been linked to reduced risk of coronary heart disease events.",
            "the evidence suggests that maintaining adequate levels of omega-3s through diet or supplementation might help reduce the risk of dying from coronary heart disease.",
            "Additionally, the Omega-3 Index has been shown to correlate positively with overall health outcomes, further supporting its potential role as a risk factor."
        ],
        "notes": "The retrieved evidence does not mention the Omega-3 Index or omega-3 fatty acids."
    },

    54: {
        "label": "hallucinated",
        "texts": [
            "The sternal fracture in growing children is indeed a rare and often overlooked fracture due to its rarity and the difficulty in diagnosing it accurately.",
            "It can occur without any obvious trauma and may present with symptoms such as chest pain, shortness of breath, or discomfort during deep breathing.",
            "Growing children's bones are still developing, making them susceptible to fractures even from minor traumas.",
            "Unlike adults, whose bones are more mature and stronger, children's bones are more prone to fractures because they are still forming and remodeling.",
            "Therefore, pediatricians should be vigilant about potential fractures in young patients, especially those under the age of 10, where the risk of fractures increases significantly.",
            "Early diagnosis and treatment are crucial to ensure proper healing and avoid long-term complications."
        ],
        "notes": "The retrieved evidence is unrelated to pediatric sternal fractures."
    },

    107: {
        "label": "hallucinated",
        "texts": [
            "the evidence suggests that stroke survivors might have different perceptions about electronic stability control (ESC) compared to non-stroke drivers.",
            "Therefore, while the DOT data does not directly address stroke survivors' experiences with ESC, the self-reported data indicates that stroke survivors may perceive ESC differently due to potential changes in driving habits influenced by the technology.",
            "This implies that both sources of data could yield insights into how stroke survivors interact with and respond to safety technologies like ESC"
        ],
        "notes": "The retrieved evidence does not support the ESC-related claims."
    },

    493: {
        "label": "hallucinated",
        "texts": [
            "No, amoxapine is not an atypical antipsychotic.",
            "Amoxapine is a tricyclic antidepressant used primarily for treating depression and anxiety.",
            "Atypical antipsychotics refer to newer drugs like olanzapine, quetiapine, risperidone, etc., which were developed after the first generation of antipsychotics (typical antipsychotics).",
            "Based on the provided evidence, amoxapine does not fall into the category of atypical antipsychotics."
        ],
        "notes": "The retrieved evidence does not mention amoxapine or its pharmacological classification."
    },

    39: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    307: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    67: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    998: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    88: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly address the question; appropriate abstention."
    },

    63: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Evidence does not directly establish that the manual enables laypersons to operate the device; appropriate abstention."
    },

    # ------------------------------------------------------------
    # Candidates 24–28
    # ------------------------------------------------------------

    716: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether serum can be used to analyze EGFR mutation status in advanced NSCLC."
    },

    351: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether scintigraphy is a guideline method for determining amputation levels in diabetic foot."
    },

    256: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether breast cancer prognosis is inherited."
    },

    644: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether dentists can recognize manipulated digital radiographs."
    },

    168: {
        "label": "hallucinated",
        "texts": [
            "there is no clear necessity to routinely insert a nasobiliary drainage tube following endoscopic clearance of the common bile duct in patients with choledocholithiasis-induced cholangitis",
            "the studies do not support routine insertion of a nasobiliary drainage tube as a preventive measure",
            "the decision should be made on a case-by-case basis considering individual patient factors and clinical context"
        ],
        "notes": "The retrieved evidence discusses bile duct sludge, stone recurrence, and ERCP outcomes, but does not evaluate routine nasobiliary drainage after endoscopic clearance in cholangitis."
    },

    # ------------------------------------------------------------
    # Candidates 29–33
    # ------------------------------------------------------------

    917: {
        "label": "partially_supported",
        "texts": [
            "Additionally, the evidence suggests that the Sami population in Norway has a lower incidence of breast cancer compared to other populations, which could potentially indicate genetic or environmental differences rather than any specific association with radiotherapy timing."
        ],
        "notes": "The answer appropriately recognizes that the retrieved evidence does not directly answer the radiotherapy/pregnancy question, but the inference about genetic or environmental differences is unsupported and not relevant to the question."
    },

    528: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence concerns echocardiography/ECG after LAA occlusion, bone markers, mitral regurgitation, and hypotension; it does not directly address ATTR V30M amyloidosis versus HCM discrimination."
    },

    823: {
        "label": "partially_supported",
        "texts": [
            "Therefore, without specific findings related to rt-PA and its effect on different genders among acute stroke patients, we cannot draw a definitive conclusion based solely on the given evidence."
        ],
        "notes": "The retrieved context does contain specific rt-PA-related evidence, including conflicting findings about whether women benefit less from IV thrombolysis. The answer therefore understates what the context actually provides."
    },

    985: {
        "label": "partially_supported",
        "texts": [
            "However, it's worth noting that septoplasty surgery involves correcting structural issues within the nose, which may indirectly affect the overall structure of the nasal cavity but does not directly alter the dimensions of the middle turbinate itself.",
            "The middle turbinate is part of the nasal conchae, which are structures that help filter air as it enters the nose.",
            "While septoplasty can potentially improve airflow through the nose, it doesn't typically involve altering the size or shape of individual turbinates.",
            "For more precise answers regarding the impact of septoplasty on the middle turbinate, additional research would be required focusing specifically on these anatomical changes post-surgery.",
            "This could include studies comparing pre-operative and post-operative imaging or histological analysis of the nasal turbinates."
        ],
        "notes": "The initial abstention is appropriate, but the answer adds general medical/anatomical claims not supported by the retrieved evidence."
    },

    816: {
        "label": "hallucinated",
        "texts": [
            "No, ventricular arrhythmias do not typically subside over time in athletes."
        ],
        "notes": "The retrieved evidence discusses ventricular arrhythmias in LVAD recipients and unrelated athlete biofeedback studies, but does not establish the long-term course of ventricular arrhythmias in athletes."
    }
}


print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates:", len(human_review_labels_training))

print("\nTest IDs:")
print(sorted(human_review_labels_training.keys()))

print("\nLabel distribution:")

training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates: 33

Test IDs:
[23, 39, 54, 63, 67, 88, 107, 168, 174, 256, 299, 307, 328, 351, 439, 493, 528, 570, 601, 626, 636, 644, 660, 687, 716, 816, 820, 823, 837, 917, 924, 985, 998]

Label distribution:
appropriate_abstention    15
hallucinated              11
partially_supported        7
Name: count, dtype: int64


In [96]:
show_training_candidates(33, batch_size=5)

CANDIDATE 34/61
TEST ID: 534

QUESTION:
Is anorectal endosonography valuable in dyschesia?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
Twenty-eight (31.8%) of the lesions were localized in fatty breast, and the remaining 60 (68.2%) lesions were localized in dense breast. There were 34 (38.6%) benign and 54 (61.4%) malignant masses. In fatty breast, chemical shift subtraction and T1-weighted images were valuable both for the demonstration and differentiation of benign lesions (P<.05). None of the sequence was valuable for both the demonstration and differentiation of malignant lesions in fatty breasts (P>.05). In dense breasts, chemical shift subtraction and dynamic contrast subtraction images were valuable for both the demonstration and differentiation of benign and malignant lesions. Additional to these sequences, T2-weighted images was also valuable for benign lesions (P<.05).

Knee extensor strength positiv

In [97]:
# Cell 81 update — candidates 34–38
# ID 445 was already reviewed previously; this restores it explicitly.

human_review_labels_training.update({

    534: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether anorectal endosonography is valuable in dyschesia. The answer appropriately abstains."
    },

    294: {
        "label": "hallucinated",
        "texts": [
            "The evidence suggests that fenofibrate may be used as part of a treatment regimen for gastroesophageal reflux disease (GERD), specifically esomeprazole"
        ],
        "notes": "The context describes esomeprazole treatment for reflux esophagitis, not fenofibrate. The answer incorrectly attributes the treatment to fenofibrate."
    },

    425: {
        "label": "hallucinated",
        "texts": [
            "Based on the provided evidence, octogenarians (patients aged 80+) are indeed at a higher risk compared to younger individuals for developing significant carotid stenosis, which can lead to complications such as stroke.",
            "This increased risk is evident from the data showing that patients over 70 years old had a higher prevalence of carotid stenosis (9.1% vs. 1.2%), indicating they are more likely to undergo carotid endarterectomy due to their higher risk profile.",
            "Therefore, while the exact number of octogenarians needing carotid endarterectomy may vary depending on local healthcare practices and patient demographics, the overall trend suggests that octogenarians are at a higher risk for this procedure compared to other age groups."
        ],
        "notes": "The retrieved evidence concerns risk of carotid stenosis and perioperative stroke in patients undergoing CAB screening. It does not establish that octogenarians are at higher procedural risk from carotid endarterectomy. The answer conflates carotid-stenosis risk with CEA risk."
    },

    445: {
        "label": "partially_supported",
        "texts": [
            "No, the secular growth acceleration does not appear during fetal life.",
            "Secular growth acceleration refers to changes in fetal growth over time rather than specific events within fetal development.",
            "Therefore, the available evidence does not support the claim that secular growth acceleration occurs during fetal life."
        ],
        "notes": "This example was previously reviewed. The answer appropriately notices that the retrieved passages do not directly address secular growth acceleration, but it adds unsupported definitional and interpretive claims."
    },

    599: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence discusses RBC storage duration, VEGF, sputum cell counts, blood pressure, and bone infarction/osteomyelitis. It does not address discordance between clinical pathologists and clinicians in reporting or interpreting RBC morphology."
    }
})

print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 38

Label distribution:
appropriate_abstention    17
hallucinated              13
partially_supported        8
Name: count, dtype: int64


In [98]:
show_training_candidates(38, batch_size=5)

CANDIDATE 39/61
TEST ID: 25

QUESTION:
Amblyopia: is visual loss permanent?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with amblyopia due to strabismus, and 13 healthy volunteers were tested. Charts with the Snellen E and the Landolt C (Precision Vision) which mimic the ETDRS charts were used to assess visual acuity. Three out of 5 optotypes per line had to be correctly identified, while wrong answers were monitored. In the group of patients, the eyes with the lower visual acuity, and the right eyes of the healthy subjects, were evaluated.

Recent studies have found a choroidal thickening in amblyopic eyes and suggested that there might be a relationship between the choroid and amblyopia. The present study aimed to evaluate the effect of a six-month treatment of amblyopia on choroidal thickness in anisometropic hypero

In [99]:

# Cell 81 update — candidates 39–43

human_review_labels_training.update({

    25: {
        "label": "hallucinated",
        "texts": [
            "Based on the evidence provided, visual loss from amblyopia is not necessarily permanent.",
            "recent research indicates that such changes can potentially improve over time with appropriate treatment.",
            "Although no significant change was observed immediately post-treatment, follow-up assessments showed a gradual increase in choroidal thickness.",
            "This suggests that with proper intervention, visual function in amblyopic eyes can indeed improve over time."
        ],
        "notes": "The retrieved evidence describes measurement of choroidal thickness in children receiving amblyopia treatment, but does not establish that visual loss is non-permanent, that choroidal thickness gradually increased at follow-up, or that visual function improved as claimed."
    },

    893: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not address reproducibility of specific IgE determination using ISAC 112."
    },

    810: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address whether feeding tube insertion or its timing affects survival."
    },

    941: {
        "label": "partially_supported",
        "texts": [
            "However, it does suggest that laparoscopic surgery may be beneficial in certain circumstances such as early postoperative feeding following colorectal resections.",
            "For accurate assessment of colorectal cancer staging, conventional imaging techniques like CT scans or MRI might be preferred over laparoscopic sonography."
        ],
        "notes": "The retrieved evidence includes a study of early postoperative feeding after laparotomy and colorectal resection, but it does not establish a general benefit of laparoscopic surgery in this context. It also provides no evidence that CT or MRI should be preferred over laparoscopic sonography for colorectal cancer staging."
    },

    309: {
        "label": "hallucinated",
        "texts": [
            "No, prostatic syndrome and pleural effusion are not different diseases.",
            "Prostatic syndrome refers to symptoms such as pain, urinary problems, and changes in sexual function related to prostate issues, while pleural effusion involves fluid accumulation around the lungs, which can be caused by various conditions including infections, inflammation, or cancer.",
            "These two conditions do not overlap in terms of their specific causes or manifestations."
        ],
        "notes": "The retrieved evidence does not address prostatic syndrome, pleural effusion, or their relationship. The answer introduces unsupported medical definitions and conclusions."
    }
})

print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 43

Label distribution:
appropriate_abstention    19
hallucinated              15
partially_supported        9
Name: count, dtype: int64


In [100]:
show_training_candidates(43, batch_size=5)

CANDIDATE 44/61
TEST ID: 697

QUESTION:
Are there gender differences in the reasons why African Americans delay in seeking medical help for symptoms of an acute myocardial infarction?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
"America's Best Hospitals," an influential list published annually by U.S. News and World Report, assesses the quality of hospitals. It is not known whether patients admitted to hospitals ranked at the top in cardiology have lower short-term mortality from acute myocardial infarction than those admitted to other hospitals or whether differences in mortality are explained by differential use of recommended therapies.

By use of the Cooperative Cardiovascular Project database (a retrospective medical record review of Medicare patients discharged with acute myocardial infarction), we identified patients aged>or =65 years whose myocardial infarction was complicated by cardiogenic shock.

Th

In [101]:

# Cell 81 update — candidates 44–48

human_review_labels_training.update({

    697: {
        "label": "partially_supported",
        "texts": [
            "While some studies suggest that African Americans may face barriers to accessing timely medical care, particularly related to socioeconomic factors such as income and education levels,"
        ],
        "notes": "The retrieved evidence mentions racial disparities and geographic socioeconomic factors, but does not establish that income/education barriers explain delayed help-seeking for acute myocardial infarction among African Americans."
    },

    884: {
        "label": "partially_supported",
        "texts": [
            "Based on the evidence provided, it appears that routine intraoperative frozen-section examination of sentinel lymph nodes in breast cancer may not always be necessary due to false positives in clinically positive axillary nodes.",
            "Additionally, while mediastinal lymph node dissection is crucial in lung cancer surgery, its necessity can vary depending on the specific location and extent of involvement.",
            "suggesting that treatment decisions might be influenced by these factors rather than just the presence of sentinel lymph nodes."
        ],
        "notes": "The context does not evaluate routine intraoperative frozen-section examination of sentinel lymph nodes. The answer draws unsupported conclusions from false-positive clinical axillary-node examination, unrelated lung-cancer lymph-node dissection, and breast-cancer prognostic factors."
    },

    767: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address antiretroviral therapy-related adverse effects or whether sub-Saharan Africa can cope with the WHO test-and-treat policy."
    },

    298: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "Although the retrieved evidence includes congestive heart failure and mortality data, it does not directly establish whether specialty or cardiologist care improves survival in CHF patients."
    },

    314: {
        "label": "hallucinated",
        "texts": [
            "Therefore, the available evidence does not support the claim that high cumulative insulin exposure is a risk factor for atherosclerosis specifically in type 1 diabetes.",
            "Type 1 diabetics typically do not develop typical atherosclerotic plaques due to their reliance on exogenous insulin therapy rather than endogenous insulin production.",
            "Therefore, while insulin may play a role in overall metabolic health, it is not considered a direct risk factor for atherosclerosis in type 1 diabetes compared to other risk factors like hypertension, dyslipidemia, smoking, obesity, and genetic predisposition."
        ],
        "notes": "The retrieved evidence does not support these claims about type 1 diabetes, atherosclerotic plaques, or insulin exposure. The answer introduces substantial unsupported medical content beyond the evidence."
    }
})

print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 48

Label distribution:
appropriate_abstention    21
hallucinated              16
partially_supported       11
Name: count, dtype: int64


In [102]:
show_training_candidates(48, batch_size=5)

CANDIDATE 49/61
TEST ID: 310

QUESTION:
Telemedicine and type 1 diabetes: is technology per se sufficient to improve glycaemic control?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
We identified 3215 studies and selected 72 relevant papers for review. Surgical treatment of diabetes is evolving from complex pancreatic and islets transplantation surgery for type 1 diabetes with critical postoperative outcome and follow-up to a metabolic surgery, including gastric bypass. Gastric bypass (no immune suppression or graft rejection) has proven to be highly effective treatment for obese patients and nonobese animals with type 2 diabetes. There are certain shared criteria between types 1 and 2 diabetes, making a selected spectrum of the disease a potential target for metabolic surgery to improve or cure diabetes.

The surgical treatment of diabetes had witnessed progressive development and success since the first case o

In [103]:

# Cell 81 update — candidates 49–53

human_review_labels_training.update({

    310: {
        "label": "partially_supported",
        "texts": [
            "While there is some indirect evidence suggesting that technological advancements like bariatric surgery might help manage type 2 diabetes, it does not directly address the specific issue of telemedicine's role in improving glycaemic control for type 1 diabetes.",
            "Additionally, while electronic stability control (ESC) technology can potentially affect driving behavior, it does not specifically relate to managing blood sugar levels in individuals with type 1 diabetes."
        ],
        "notes": "The overall abstention is reasonable, but the answer unnecessarily treats bariatric surgery as relevant technological evidence and introduces unrelated ESC technology."
    },

    479: {
        "label": "hallucinated",
        "texts": [
            "Based on the provided evidence, there is no direct correlation found between long-term postictal psychotic episodes and the development of interictal psychotic episodes.",
            "while postictal psychotic episodes may have implications for motor functions and overall health, they do not appear to predict future occurrences of interictal psychotic episodes.",
            "individuals who experience postictal psychotic episodes might exhibit certain characteristics such as a higher risk of relapse into depression, which could potentially influence their susceptibility to developing interictal psychotic episodes later in life."
        ],
        "notes": "The retrieved evidence does not address postictal psychosis versus interictal psychosis. The answer makes unsupported negative and predictive claims and imports an unrelated depression-relapse finding."
    },

    110: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address mediation by self-efficacy between transformational leadership behaviours and healthcare workers' sleep quality."
    },

    989: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence discusses aging-related topics and Indigenous populations only indirectly; it does not establish whether Indigenous Australians age prematurely."
    },

    254: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not discuss the Child Health Computing System or its ability to identify children with cerebral palsy."
    }
})

print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 53

Label distribution:
appropriate_abstention    24
hallucinated              17
partially_supported       12
Name: count, dtype: int64


In [104]:
show_training_candidates(53, batch_size=5)

CANDIDATE 54/61
TEST ID: 677

QUESTION:
Is transurethral catheterisation the ideal method of bladder drainage?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
Sixty-seven patients undergoing transurethral resection of the prostate without the use of a video camera were randomly allocated into three groups according to the urologist's experience: a urologist having done 25 transurethral resections of the prostate (Group I - 24 patients); a urologist having done 50 transurethral resections of the prostate (Group II - 24 patients); a senior urologist with vast transurethral resection of the prostate experience (Group III - 19 patients). The following were recorded: the weight of resected tissue, the duration of the resection procedure, the volume of irrigation used, the amount of irrigation absorbed and the hemoglobin and sodium levels in the serum during the procedure.

To evaluate the influence of the urologist's e

In [105]:

# Cell 81 update — candidates 54–58

human_review_labels_training.update({

    677: {
        "label": "partially_supported",
        "texts": [
            "However, it can be inferred from the context that transurethral catheterization might be considered as one of the methods due to its association with TURP procedures.",
            "Transurethral resection of the prostate typically involves inserting a catheter through the urethra to drain urine from the bladder,"
        ],
        "notes": "The answer appropriately recognizes that the evidence does not compare bladder-drainage methods, but it introduces an unsupported procedural inference from the TURP context."
    },

    526: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence concerns acetabular fractures/THA, minimally invasive mitral repair, gastric banding, acetabular component stability, and pelvic organ prolapse. It does not directly address pelvic anatomical injury during arthroscopic acetabular labral repair."
    },

    845: {
        "label": "appropriate_abstention",
        "texts": [],
        "notes": "The retrieved evidence does not directly address rebound aggravation of symptoms following short-term proton pump inhibitor treatment."
    },

    365: {
        "label": "hallucinated",
        "texts": [
            "they suggest that lifestyle factors influenced by intrauterine environments could potentially contribute to obesity through mechanisms like eating habits.",
            "Therefore, while direct evidence linking intrauterine influences to obesity is lacking, the indirect relationship between eating behaviors and obesity suggests that understanding these behaviors could have implications for obesity prevention strategies."
        ],
        "notes": "The retrieved evidence shows that plate-clearing behavior can be associated with obesity, but it does not connect that finding to an intrauterine environment. The answer invents an intrauterine-to-eating-behavior mechanism."
    },

    829: {
        "label": "partially_supported",
        "texts": [
            "While there is some evidence suggesting that frozen section may alter surgical management in certain conditions like cervical cancer,"
        ],
        "notes": "The retrieved evidence does not mention frozen section or cervical cancer. This is an unsupported claim embedded within an otherwise appropriate statement that the specific thyroid-disease question is not addressed."
    }
})

print("=" * 80)
print("TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 58

Label distribution:
appropriate_abstention    26
hallucinated              18
partially_supported       14
Name: count, dtype: int64


In [106]:
show_training_candidates(58, batch_size=5)

CANDIDATE 59/61
TEST ID: 199

QUESTION:
Attenuation of ischemia/reperfusion-induced ovarian damage in rats: does edaravone offer protection?

----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT:
Ischemic preconditioning (IP) is initiated through one or several short bouts of ischemia and reperfusion which precede a prolonged ischemia. To test whether a reperfusion must precede the prolonged index ischemia, a series without reperfusion (intraischemic preconditioning: IIP) and a series with gradual onset of ischemia, i.e. ramp ischemia (RI), which is possibly related to the development of hibernation, was compared to conventional IP (CIP).

Ischemia-reperfusion (IR) injury remains a major cause of early morbidity and mortality after lung transplantation with poorly documented extrapulmonary repercussions. To determine the hemodynamic effect due to lung IR injury, we performed a quantitative coronary blood-flow analysis in

In [107]:

# Cell 81 update — final candidates 59–61

human_review_labels_training.update({

    199: {
        "label": "hallucinated",
        "texts": [
            "Based on the evidence provided, edaravone offers protection against ischemia/reperfusion-induced ovarian damage in rats as evidenced by the experimental results where edaravone treatment showed reduced infarct size compared to controls.",
            "Additionally, while the evidence suggests that edaravone may protect against ischemia/reperfusion injury in other tissues such as the heart, its specific role in protecting against ovarian damage requires further investigation.",
            "The data presented here indicates potential benefits but more comprehensive studies are necessary before definitive conclusions can be drawn about its efficacy in preventing ovarian damage specifically."
        ],
        "notes": "The retrieved context contains ischemia/reperfusion studies in heart, lung, and other settings, plus a study on ovarian endometriomas, but it does not mention edaravone treatment or reduced ovarian infarct/damage. The answer invents experimental findings."
    },

    408: {
        "label": "partially_supported",
        "texts": [
            "Yes, medical education curriculum has influenced medical students' skills and attitudes toward culturally diverse populations.",
            "Additionally, community-based medical education programs have been found to be effective in meeting the needs of patients and improving their satisfaction with the educational process.",
            "Furthermore, research indicates that perspective-taking instruction can enhance students' understanding of cultural diversity and improve their attitudes towards it."
        ],
        "notes": "The retrieved evidence includes patient-safety curricula, community-based medical education, and perspective-taking studies involving African American standardized patients. However, it does not establish the specific Global Longitudinal Pathway claim or prove the broad stated effect on culturally diverse populations."
    },

    613: {
        "label": "hallucinated",
        "texts": [
            "Yes, irritable bowel syndrome (IBS) can be considered a diagnosis of exclusion.",
            "it is generally accepted that IBS is often diagnosed after ruling out other conditions that might present similar symptoms.",
            "Therefore, if someone presents with abdominal bloating or flatulence, their doctor would likely first rule out other potential causes before diagnosing IBS.",
            "This process of excluding alternative diagnoses is known as the diagnosis of exclusion.",
            "Thus, while IBS is commonly diagnosed through exclusionary methods,"
        ],
        "notes": "The retrieved evidence discusses IBS symptoms and an intervention for bloating/flatulence, but does not establish how IBS should be diagnosed or that IBS is generally a diagnosis of exclusion."
    }
})

print("=" * 80)
print("FINAL TRAINING HUMAN REVIEW ANNOTATIONS")
print("=" * 80)

print("Reviewed candidates in dictionary:", len(human_review_labels_training))

print("\nLabel distribution:")
training_review_distribution = pd.Series(
    [item["label"] for item in human_review_labels_training.values()]
).value_counts()

print(training_review_distribution)

FINAL TRAINING HUMAN REVIEW ANNOTATIONS
Reviewed candidates in dictionary: 61

Label distribution:
appropriate_abstention    26
hallucinated              20
partially_supported       15
Name: count, dtype: int64


In [108]:

# Make sure the complete human-review dictionary is available
print("Human-reviewed examples:", len(human_review_labels_training))

# Check that all reviewed IDs are in the original annotation dataframe
reviewed_ids = set(human_review_labels_training.keys())
available_ids = set(annotation_df["test_id"].tolist())

missing_ids = reviewed_ids - available_ids

print("Missing IDs from annotation_df:", missing_ids)

if missing_ids:
    raise ValueError(f"These reviewed IDs are missing from annotation_df: {sorted(missing_ids)}")

print("✅ All 61 reviewed candidates are present in annotation_df.")

Human-reviewed examples: 61
Missing IDs from annotation_df: set()
✅ All 61 reviewed candidates are present in annotation_df.


In [109]:

# Fix the exact annotation text for test_id 314

human_review_labels_training[314]["texts"] = [
    "The available evidence does not support the claim that high cumulative insulin exposure is a risk factor for atherosclerosis specifically in type 1 diabetes.",
    "Type 1 diabetics typically do not develop typical atherosclerotic plaques due to their reliance on exogenous insulin therapy rather than endogenous insulin production.",
    "Therefore, while insulin may play a role in overall metabolic health, it is not considered a direct risk factor for atherosclerosis in type 1 diabetes compared to other risk factors like hypertension, dyslipidemia, smoking, obesity, and genetic predisposition."
]

print("✅ ID 314 annotation text corrected.")
print("\nUpdated spans:")
for text in human_review_labels_training[314]["texts"]:
    print("-", text)

✅ ID 314 annotation text corrected.

Updated spans:
- The available evidence does not support the claim that high cumulative insulin exposure is a risk factor for atherosclerosis specifically in type 1 diabetes.
- Type 1 diabetics typically do not develop typical atherosclerotic plaques due to their reliance on exogenous insulin therapy rather than endogenous insulin production.
- Therefore, while insulin may play a role in overall metabolic health, it is not considered a direct risk factor for atherosclerosis in type 1 diabetes compared to other risk factors like hypertension, dyslipidemia, smoking, obesity, and genetic predisposition.


Cell 82 — Apply the 61 reviewed labels and verify positive spans

In [110]:
# ============================================================
# CELL 82 — APPLY FINAL HUMAN-REVIEWED ANNOTATIONS
# ============================================================

import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Make a copy so the original annotation_df stays untouched
# ------------------------------------------------------------

verified_df = annotation_df.copy()

# ------------------------------------------------------------
# 2. Final fixed test IDs
# ------------------------------------------------------------

FINAL_TEST_IDS = {
    436, 899, 346, 60, 867, 432, 543, 361, 486, 277,
    662, 377, 668, 589, 355, 259, 617, 977, 275, 70,
    237, 280, 822, 218, 692, 296, 318, 866, 559, 265,
    136, 761, 30, 289, 643, 635, 55, 215, 865, 652
}

# ------------------------------------------------------------
# 3. Previously verified negative examples
# ------------------------------------------------------------

PREVIOUSLY_VERIFIED_NEGATIVES = {
    521,
    260,
    678
}

# ------------------------------------------------------------
# 4. Check required variables exist
# ------------------------------------------------------------

assert "human_review_labels_training" in globals(), \
    "human_review_labels_training dictionary is missing."

assert "annotation_df" in globals(), \
    "annotation_df is missing."

print("annotation_df rows:", len(verified_df))
print("Human-reviewed IDs:", len(human_review_labels_training))

# ------------------------------------------------------------
# 5. Identify ID column
# ------------------------------------------------------------

if "test_id" in verified_df.columns:
    ID_COL = "test_id"
elif "id" in verified_df.columns:
    ID_COL = "id"
else:
    raise ValueError(
        "Could not find ID column. Expected 'test_id' or 'id'."
    )

print("Using ID column:", ID_COL)

# ------------------------------------------------------------
# 6. Identify answer column
# ------------------------------------------------------------

if "answer" in verified_df.columns:
    ANSWER_COL = "answer"
elif "generated_answer" in verified_df.columns:
    ANSWER_COL = "generated_answer"
else:
    raise ValueError(
        "Could not find answer column. Expected 'answer' or "
        "'generated_answer'."
    )

print("Using answer column:", ANSWER_COL)

# ------------------------------------------------------------
# 7. Create clean annotation columns
# ------------------------------------------------------------

verified_df["human_label"] = None
verified_df["hallucination_texts"] = None
verified_df["human_notes"] = None
verified_df["hallucination_spans"] = None

# ------------------------------------------------------------
# 8. Apply the 61 reviewed annotations
# ------------------------------------------------------------

applied_count = 0

for raw_id, review in human_review_labels_training.items():

    example_id = int(raw_id)

    if example_id not in set(verified_df[ID_COL].astype(int)):
        raise ValueError(
            f"Reviewed ID {example_id} is missing from annotation_df."
        )

    idx_list = verified_df.index[
        verified_df[ID_COL].astype(int) == example_id
    ].tolist()

    if len(idx_list) != 1:
        raise ValueError(
            f"Expected exactly one row for ID {example_id}, "
            f"found {len(idx_list)}."
        )

    idx = idx_list[0]

    label = review.get("label")
    texts = review.get("texts", [])
    notes = review.get("notes", "")

    if texts is None:
        texts = []

    # Ensure list
    if isinstance(texts, str):
        texts = [texts]

    verified_df.at[idx, "human_label"] = label
    verified_df.at[idx, "hallucination_texts"] = texts
    verified_df.at[idx, "human_notes"] = notes

    applied_count += 1

# ------------------------------------------------------------
# 9. Apply the three previously verified negatives
# ------------------------------------------------------------

for example_id in PREVIOUSLY_VERIFIED_NEGATIVES:

    idx_list = verified_df.index[
        verified_df[ID_COL].astype(int) == example_id
    ].tolist()

    if len(idx_list) != 1:
        raise ValueError(
            f"Could not uniquely find previously verified ID {example_id}."
        )

    idx = idx_list[0]

    verified_df.at[idx, "human_label"] = "appropriate_abstention"
    verified_df.at[idx, "hallucination_texts"] = []
    verified_df.at[idx, "human_notes"] = "Previously verified negative"

# ------------------------------------------------------------
# 10. Verify all reviewed examples were applied
# ------------------------------------------------------------

all_reviewed_ids = set(
    int(x) for x in human_review_labels_training.keys()
)

all_reviewed_ids.update(PREVIOUSLY_VERIFIED_NEGATIVES)

missing_after_apply = all_reviewed_ids - set(
    verified_df.loc[
        verified_df["human_label"].notna(),
        ID_COL
    ].astype(int)
)

print("\nApplied human-reviewed annotations:", applied_count)
print("Previously verified negatives:", len(PREVIOUSLY_VERIFIED_NEGATIVES))
print("Missing after application:", missing_after_apply)

assert len(missing_after_apply) == 0, \
    f"Some reviewed IDs were not applied: {missing_after_apply}"

# ------------------------------------------------------------
# 11. Convert exact annotation text → character spans
# ------------------------------------------------------------

def find_all_exact_spans(answer, texts):
    """
    Find every occurrence of each manually annotated hallucination
    substring inside the generated answer.
    """

    if answer is None:
        answer = ""

    answer = str(answer)

    if texts is None:
        texts = []

    if isinstance(texts, str):
        texts = [texts]

    spans = []

    for text in texts:

        if text is None:
            continue

        text = str(text)

        if text == "":
            continue

        start = 0

        while True:

            pos = answer.find(text, start)

            if pos == -1:
                break

            end = pos + len(text)

            spans.append({
                "start": pos,
                "end": end,
                "text": answer[pos:end]
            })

            start = end

    # Remove duplicates
    unique = {}

    for span in spans:
        key = (
            span["start"],
            span["end"],
            span["text"]
        )
        unique[key] = span

    spans = list(unique.values())

    # Sort
    spans.sort(
        key=lambda x: (x["start"], x["end"])
    )

    return spans


# ------------------------------------------------------------
# 12. Apply span conversion
# ------------------------------------------------------------

for idx, row in verified_df.iterrows():

    answer = row[ANSWER_COL]
    texts = row["hallucination_texts"]

    spans = find_all_exact_spans(
        answer,
        texts
    )

    verified_df.at[
        idx,
        "hallucination_spans"
    ] = spans


# ------------------------------------------------------------
# 13. Validate every span
# ------------------------------------------------------------

invalid_spans = []

for idx, row in verified_df.iterrows():

    answer = str(row[ANSWER_COL])

    spans = row["hallucination_spans"]

    if spans is None:
        spans = []

    for span in spans:

        start = span["start"]
        end = span["end"]

        valid = (
            isinstance(start, int)
            and isinstance(end, int)
            and 0 <= start < end <= len(answer)
            and answer[start:end] == span["text"]
        )

        if not valid:

            invalid_spans.append({
                "row_index": idx,
                "id": row[ID_COL],
                "span": span
            })


# ------------------------------------------------------------
# 14. Check for missing exact text annotations
# ------------------------------------------------------------

missing_text_matches = []

for idx, row in verified_df.iterrows():

    answer = str(row[ANSWER_COL])

    texts = row["hallucination_texts"]

    if texts is None:
        texts = []

    for text in texts:

        if text is None:
            continue

        text = str(text)

        if text and text not in answer:

            missing_text_matches.append({
                "id": row[ID_COL],
                "text": text,
                "answer": answer
            })


# ------------------------------------------------------------
# 15. Count positive spans
# ------------------------------------------------------------

positive_examples = 0
total_positive_spans = 0

for _, row in verified_df.iterrows():

    spans = row["hallucination_spans"]

    if spans is None:
        spans = []

    if len(spans) > 0:
        positive_examples += 1
        total_positive_spans += len(spans)


# ------------------------------------------------------------
# 16. Check training/test contamination
# ------------------------------------------------------------

training_candidate_ids = set(
    verified_df.loc[
        verified_df["human_label"].notna(),
        ID_COL
    ].astype(int)
)

test_overlap = training_candidate_ids.intersection(
    FINAL_TEST_IDS
)


# ------------------------------------------------------------
# 17. FINAL AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CELL 82 FINAL AUDIT")
print("=" * 60)

print("Reviewed examples with labels:",
      len(training_candidate_ids))

print("Positive training examples:",
      positive_examples)

print("Total positive hallucination spans:",
      total_positive_spans)

print("Invalid spans:",
      len(invalid_spans))

print("Missing text matches:",
      len(missing_text_matches))

print("Training/Test overlap:",
      len(test_overlap))

print("=" * 60)

# ------------------------------------------------------------
# 18. HARD ASSERTIONS
# ------------------------------------------------------------

assert positive_examples > 0, \
    "ERROR: No positive hallucination examples found."

assert total_positive_spans > 0, \
    "ERROR: No positive hallucination spans found."

assert len(invalid_spans) == 0, \
    f"ERROR: Invalid spans found: {invalid_spans[:5]}"

assert len(missing_text_matches) == 0, \
    f"ERROR: Some annotated texts were not found in answers: {missing_text_matches[:5]}"

assert len(test_overlap) == 0, \
    f"ERROR: Training/test contamination: {test_overlap}"

print("\n✅ CELL 82 PASSED")
print("Positive supervision exists.")
print("All spans are valid.")
print("No final-test contamination.")

annotation_df rows: 200
Human-reviewed IDs: 61
Using ID column: test_id
Using answer column: answer

Applied human-reviewed annotations: 61
Previously verified negatives: 3
Missing after application: set()

CELL 82 FINAL AUDIT
Reviewed examples with labels: 64
Positive training examples: 35
Total positive hallucination spans: 93
Invalid spans: 0
Missing text matches: 0
Training/Test overlap: 0

✅ CELL 82 PASSED
Positive supervision exists.
All spans are valid.
No final-test contamination.


CELL 83 — INSPECT THE ANNOTATED TRAINING DATA

In [111]:
# ============================================================
# CELL 83 — INSPECT VERIFIED ANNOTATIONS
# ============================================================

print("\nLABEL DISTRIBUTION")
print("=" * 50)

print(
    verified_df["human_label"]
    .value_counts(dropna=False)
)

print("\n")

# Only manually reviewed examples
reviewed_df = verified_df[
    verified_df["human_label"].notna()
].copy()

print("Reviewed examples:", len(reviewed_df))

print("\nPositive examples:")
print(
    reviewed_df[
        reviewed_df["hallucination_spans"].apply(
            lambda x: len(x) > 0
        )
    ][
        [ID_COL, "human_label", ANSWER_COL, "hallucination_spans"]
    ].head(10).to_string(index=False)
)

print("\nNegative examples:")
print(
    reviewed_df[
        reviewed_df["hallucination_spans"].apply(
            lambda x: len(x) == 0
        )
    ][
        [ID_COL, "human_label", ANSWER_COL]
    ].head(10).to_string(index=False)
)


LABEL DISTRIBUTION
human_label
None                      136
appropriate_abstention     29
hallucinated               20
partially_supported        15
Name: count, dtype: int64


Reviewed examples: 64

Positive examples:
 test_id         human_label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

CELL 84 — CREATE verified_training_df

In [112]:
# ============================================================
# CELL 84 — CREATE VERIFIED TRAINING DATAFRAME
# ============================================================

verified_training_df = verified_df[
    verified_df["human_label"].notna()
].copy()

# Absolutely remove the fixed final test set
verified_training_df = verified_training_df[
    ~verified_training_df[ID_COL].astype(int).isin(FINAL_TEST_IDS)
].copy()

# Reset index
verified_training_df = verified_training_df.reset_index(drop=True)

print("Verified training examples:",
      len(verified_training_df))

print("\nLabel distribution:")
print(
    verified_training_df["human_label"].value_counts()
)

print("\nPositive examples:",
      verified_training_df[
          verified_training_df["hallucination_spans"].apply(
              lambda x: len(x) > 0
          )
      ].shape[0])

print("\nPositive spans:",
      sum(
          len(x)
          for x in verified_training_df["hallucination_spans"]
      ))

print("\nTest overlap:",
      set(
          verified_training_df[ID_COL].astype(int)
      ).intersection(FINAL_TEST_IDS)
)

assert len(
    set(verified_training_df[ID_COL].astype(int))
    .intersection(FINAL_TEST_IDS)
) == 0

assert (
    verified_training_df[
        verified_training_df["hallucination_spans"].apply(
            lambda x: len(x) > 0
        )
    ].shape[0] > 0
)

print("\n✅ verified_training_df is ready.")

Verified training examples: 64

Label distribution:
human_label
appropriate_abstention    29
hallucinated              20
partially_supported       15
Name: count, dtype: int64

Positive examples: 35

Positive spans: 93

Test overlap: set()

✅ verified_training_df is ready.


CELL 85 — SAVE THE VERIFIED ANNOTATIONS

In [113]:
# ============================================================
# CELL 85 — SAVE VERIFIED ANNOTATIONS
# ============================================================

import json
import os

OUTPUT_DIR = "/kaggle/working/medical_hallucination_dataset"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# Save readable CSV
verified_training_df.to_csv(
    f"{OUTPUT_DIR}/verified_training_annotations.csv",
    index=False
)

print(
    "Saved:",
    f"{OUTPUT_DIR}/verified_training_annotations.csv"
)

# Save a JSON version
json_records = []

for _, row in verified_training_df.iterrows():

    record = {
        "test_id": int(row[ID_COL]),
        "question": str(row["question"]),
        "context": str(row["context"]),
        "answer": str(row[ANSWER_COL]),
        "label": str(row["human_label"]),
        "hallucination_spans": row["hallucination_spans"],
        "notes": str(row["human_notes"])
    }

    json_records.append(record)

with open(
    f"{OUTPUT_DIR}/verified_training_annotations.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_records,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    f"{OUTPUT_DIR}/verified_training_annotations.json"
)

Saved: /kaggle/working/medical_hallucination_dataset/verified_training_annotations.csv
Saved: /kaggle/working/medical_hallucination_dataset/verified_training_annotations.json


CELL 86 — CREATE TRAIN/VALIDATION/TEST SPLIT

In [114]:
# ============================================================
# CELL 86 — TRAIN / VALIDATION / FINAL TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Fixed final test
# ------------------------------------------------------------

final_test_df = verified_df[
    verified_df[ID_COL].astype(int).isin(FINAL_TEST_IDS)
].copy()

# ------------------------------------------------------------
# Supervised pool
# ------------------------------------------------------------

supervised_df = verified_df[
    verified_df["human_label"].notna()
].copy()

supervised_df = supervised_df[
    ~supervised_df[ID_COL].astype(int).isin(FINAL_TEST_IDS)
].copy()

# ------------------------------------------------------------
# Stratified split
# ------------------------------------------------------------

train_df, validation_df = train_test_split(
    supervised_df,
    test_size=0.20,
    random_state=42,
    stratify=supervised_df["human_label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
final_test_df = final_test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(validation_df))
print("FINAL TEST:", len(final_test_df))

print("\nTRAIN labels:")
print(train_df["human_label"].value_counts())

print("\nVALIDATION labels:")
print(validation_df["human_label"].value_counts())

print("\nFINAL TEST labels:")
print(final_test_df["human_label"].value_counts())

# ------------------------------------------------------------
# Hard checks
# ------------------------------------------------------------

assert set(train_df[ID_COL]).isdisjoint(
    set(final_test_df[ID_COL])
)

assert set(validation_df[ID_COL]).isdisjoint(
    set(final_test_df[ID_COL])
)

assert len(final_test_df) == 40

assert set(final_test_df[ID_COL].astype(int)) == FINAL_TEST_IDS

print("\n✅ SPLIT VALIDATED")

TRAIN: 51
VALIDATION: 13
FINAL TEST: 40

TRAIN labels:
human_label
appropriate_abstention    23
hallucinated              16
partially_supported       12
Name: count, dtype: int64

VALIDATION labels:
human_label
appropriate_abstention    6
hallucinated              4
partially_supported       3
Name: count, dtype: int64

FINAL TEST labels:
Series([], Name: count, dtype: int64)

✅ SPLIT VALIDATED


CELL 87 — CONVERT DATA INTO LETTUCEDETECT/RAGTRUTH FORMAT

In [115]:
# ============================================================
# CELL 87 — CONVERT TO RAGTRUTH-STYLE JSON
# ============================================================

def create_prompt(context, question, answer):
    """
    Create the same conceptual input structure used by
    LettuceDetect: context + question + answer.
    """

    return (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer:\n{answer}"
    )


def dataframe_to_records(df, split_name):

    records = []

    for _, row in df.iterrows():

        answer = str(row[ANSWER_COL])

        spans = row["hallucination_spans"]

        if spans is None:
            spans = []

        labels = []

        for span in spans:

            labels.append({
                "start": int(span["start"]),
                "end": int(span["end"]),
                "label": "hallucinated"
            })

        record = {
            "prompt": create_prompt(
                row["context"],
                row["question"],
                answer
            ),
            "answer": answer,
            "labels": labels,
            "split": split_name,
            "task_type": "qa",
            "dataset": "medical_pubmedqa",
            "language": "en",
            "test_id": int(row[ID_COL])
        }

        records.append(record)

    return records


train_records = dataframe_to_records(
    train_df,
    "train"
)

validation_records = dataframe_to_records(
    validation_df,
    "validation"
)

test_records = dataframe_to_records(
    final_test_df,
    "test"
)

combined_records = (
    train_records
    + validation_records
    + test_records
)

print("Train records:", len(train_records))
print("Validation records:", len(validation_records))
print("Test records:", len(test_records))
print("Combined:", len(combined_records))

print("\nExample:")
print(
    json.dumps(
        train_records[0],
        indent=2,
        ensure_ascii=False
    )[:3000]
)

Train records: 51
Validation records: 13
Test records: 40
Combined: 104

Example:
{
  "prompt": "Context:\nMechanically ventilated patients experience profound stress. Interventions are needed to ameliorate stress that does not cause adverse effects. The purpose of this study was to explore the influence of music on stress in a sample of patients over the duration of ventilatory support.RESEARCH METHODOLOGY/\n\nThe aim of this study was to evaluate the effectiveness of our surgical strategy for acute aortic dissection based on the extent of the dissection and the site of the entry, with special emphasis on resection of all dissected aortic segments if technically possible.\n\nWe examined the placebo arm of the NINDS t-PA stroke trial to test the hypothesis that the total volume of cerebral infarction in patients with right hemisphere strokes would be greater than the volume of cerebral infarction in patients with left hemisphere strokes who have similar NIHSS scores. The volume of stro

CELL 88 — SAVE THE FOUR JSON FILES

In [116]:
# ============================================================
# CELL 88 — SAVE TRAIN / VALIDATION / TEST JSON
# ============================================================

import json
import os

DATA_DIR = "/kaggle/working/medical_hallucination_dataset"

os.makedirs(
    DATA_DIR,
    exist_ok=True
)

files_to_save = {
    "medical_train.json": train_records,
    "medical_validation.json": validation_records,
    "medical_test.json": test_records,
    "medical_combined.json": combined_records
}

for filename, records in files_to_save.items():

    path = os.path.join(
        DATA_DIR,
        filename
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            records,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"✅ Saved {filename}: {len(records)} records"
    )

✅ Saved medical_train.json: 51 records
✅ Saved medical_validation.json: 13 records
✅ Saved medical_test.json: 40 records
✅ Saved medical_combined.json: 104 records


CELL 89 — CRITICAL POSITIVE-SPAN AUDIT

In [117]:
# ============================================================
# CELL 89 — POSITIVE SPAN AUDIT BEFORE TOKENIZATION
# ============================================================

def audit_records(records, name):

    examples_with_positive = 0
    total_positive_spans = 0
    total_chars = 0

    for record in records:

        answer = record["answer"]

        total_chars += len(answer)

        labels = record.get("labels", [])

        if len(labels) > 0:

            examples_with_positive += 1
            total_positive_spans += len(labels)

            for label in labels:

                start = label["start"]
                end = label["end"]

                assert (
                    0 <= start < end <= len(answer)
                ), (
                    f"Invalid span in {name}: "
                    f"{start}, {end}, answer length={len(answer)}"
                )

                assert (
                    answer[start:end].strip() != ""
                ), (
                    f"Empty hallucination span in {name}"
                )

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)
    print(
        "Total examples:",
        len(records)
    )
    print(
        "Examples with positive spans:",
        examples_with_positive
    )
    print(
        "Total positive spans:",
        total_positive_spans
    )
    print(
        "Total answer characters:",
        total_chars
    )

    return (
        examples_with_positive,
        total_positive_spans
    )


train_positive = audit_records(
    train_records,
    "TRAIN"
)

validation_positive = audit_records(
    validation_records,
    "VALIDATION"
)

test_positive = audit_records(
    test_records,
    "FINAL TEST"
)

assert train_positive[0] > 0
assert train_positive[1] > 0

print("\n✅ POSITIVE SPAN AUDIT PASSED")


TRAIN
Total examples: 51
Examples with positive spans: 28
Total positive spans: 77
Total answer characters: 39549

VALIDATION
Total examples: 13
Examples with positive spans: 7
Total positive spans: 16
Total answer characters: 9467

FINAL TEST
Total examples: 40
Examples with positive spans: 0
Total positive spans: 0
Total answer characters: 26954

✅ POSITIVE SPAN AUDIT PASSED


In [118]:
# ============================================================
# CELL 90 — VERIFY FINAL TEST ANNOTATIONS
# ============================================================

print("=" * 60)
print("FINAL TEST ANNOTATION CHECK")
print("=" * 60)

print("\nFinal test label distribution:")
print(
    final_test_df["human_label"].value_counts(dropna=False)
)

print("\nFinal test examples with spans:")

test_with_spans = final_test_df[
    final_test_df["hallucination_spans"].apply(
        lambda x: len(x) > 0
    )
]

print(
    test_with_spans[
        [
            ID_COL,
            "human_label",
            "hallucination_spans"
        ]
    ].to_string(index=False)
)

print("\nNumber with positive spans:",
      len(test_with_spans))

print(
    "Total positive spans:",
    sum(
        len(x)
        for x in final_test_df["hallucination_spans"]
    )
)

print("\nExpected according to your handoff:")
print("  hallucinated = 8")
print("  partially_supported = 9")
print("  appropriate_abstention = 23")

FINAL TEST ANNOTATION CHECK

Final test label distribution:
human_label
None    40
Name: count, dtype: int64

Final test examples with spans:
Empty DataFrame
Columns: [test_id, human_label, hallucination_spans]
Index: []

Number with positive spans: 0
Total positive spans: 0

Expected according to your handoff:
  hallucinated = 8
  partially_supported = 9
  appropriate_abstention = 23


In [122]:
# ============================================================
# CELL 91 — FIND EXISTING FINAL TEST ANNOTATIONS
# ============================================================

print("=" * 70)
print("SEARCHING NOTEBOOK VARIABLES FOR FINAL TEST ANNOTATIONS")
print("=" * 70)

# Show variables that might contain annotation information
possible_vars = []

for name, value in globals().items():

    name_lower = name.lower()

    if any(keyword in name_lower for keyword in [
        "test",
        "annotation",
        "label",
        "review",
        "span",
        "human"
    ]):

        possible_vars.append(
            (name, type(value).__name__)
        )

for item in sorted(possible_vars):
    print(item)

print("\nFINAL_TEST_IDS:")
print(sorted(FINAL_TEST_IDS))

SEARCHING NOTEBOOK VARIABLES FOR FINAL TEST ANNOTATIONS
('ANNOTATION_MODEL', 'str')
('FINAL_TEST_IDS', 'set')
('adapted_spans', 'list')
('all_reviewed_ids', 'set')
('annotation', 'dict')
('annotation_df', 'DataFrame')
('annotation_file', 'str')
('annotation_model', 'Qwen2ForCausalLM')
('annotation_test', 'DataFrame')
('annotation_tokenizer', 'Qwen2Tokenizer')
('annotation_train', 'DataFrame')
('annotation_val', 'DataFrame')
('answer_labels', 'Tensor')
('baseline_span_map', 'dict')
('baseline_spans', 'list')
('candidate_review', 'DataFrame')
('candidate_test_ids', 'list')
('create_hallucination_span', 'function')
('create_spans_from_texts', 'function')
('example_spans', 'list')
('final_test_annotations', 'DataFrame')
('final_test_dataset', 'RagTruthDataset')
('final_test_df', 'DataFrame')
('final_test_ids', 'set')
('final_test_samples', 'list')
('find_all_exact_spans', 'function')
('flagged_review', 'DataFrame')
('ground_truth_token_labels', 'list')
('human_review_labels', 'dict')
('hum

In [123]:
# ============================================================
# CELL 92 — CHECK IMPORTANT EXISTING VARIABLES
# ============================================================

variables_to_check = [
    "annotation_df",
    "human_review_labels_training",
    "final_test_df",
    "verified_df",
    "verified_training_df",
    "test_df",
    "test_annotations",
    "human_review_labels_test",
    "human_review_labels_final_test",
    "final_test_annotations"
]

for name in variables_to_check:

    if name in globals():

        obj = globals()[name]

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)

        print("Type:", type(obj))

        if isinstance(obj, pd.DataFrame):

            print("Shape:", obj.shape)
            print("Columns:")
            print(list(obj.columns))

            if ID_COL in obj.columns:

                matching = obj[
                    obj[ID_COL].astype(int).isin(FINAL_TEST_IDS)
                ]

                print(
                    "Rows matching final test IDs:",
                    len(matching)
                )

                if len(matching) > 0:
                    print(
                        matching.head(5).to_string(index=False)
                    )

        elif isinstance(obj, dict):

            print("Dictionary length:", len(obj))

            matching_keys = [
                k for k in obj.keys()
                if int(k) in FINAL_TEST_IDS
            ]

            print(
                "Final-test IDs found:",
                len(matching_keys)
            )

            print(
                "Matching keys:",
                matching_keys
            )

            for k in matching_keys[:3]:
                print(
                    f"\nExample {k}:",
                    obj[k]
                )

    else:

        print("\n", name, "→ NOT FOUND")


annotation_df
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (200, 9)
Columns:
['test_id', 'question', 'context', 'answer', 'reference_answer', 'final_decision', 'overall_label', 'hallucination_spans', 'annotator_notes']
Rows matching final test IDs: 40
 test_id                                                                                                                                            question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

CELL 93 — Inspect final_test_annotations

In [124]:
# ============================================================
# CELL 93 — INSPECT FINAL TEST ANNOTATIONS
# ============================================================

print("=" * 70)
print("FINAL TEST ANNOTATIONS")
print("=" * 70)

print("\nType:")
print(type(final_test_annotations))

print("\nShape:")
print(final_test_annotations.shape)

print("\nColumns:")
print(final_test_annotations.columns.tolist())

print("\nFirst 10 rows:")
print(
    final_test_annotations.head(10).to_string(index=False)
)

print("\nLabel distribution:")

# Try common label column names
for col in [
    "human_label",
    "label",
    "existing_label",
    "category",
    "ground_truth_label"
]:
    if col in final_test_annotations.columns:
        print(f"\nColumn: {col}")
        print(
            final_test_annotations[col]
            .value_counts(dropna=False)
        )

FINAL TEST ANNOTATIONS

Type:
<class 'pandas.core.frame.DataFrame'>

Shape:
(40, 9)

Columns:
['test_id', 'question', 'context', 'answer', 'reference_answer', 'final_decision', 'overall_label', 'hallucination_spans', 'annotator_notes']

First 10 rows:
 test_id                                                                                                                                          question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [125]:
# ============================================================
# CELL 94 — CHECK FINAL TEST IDS + ANNOTATION CONTENT
# ============================================================

print("=" * 70)
print("FINAL TEST ID / ANNOTATION MATCH CHECK")
print("=" * 70)

# Display rows corresponding to the 40 fixed IDs
for possible_id_col in [
    "test_id",
    "id",
    "example_id"
]:

    if possible_id_col in final_test_annotations.columns:

        print(
            f"\nUsing ID column: {possible_id_col}"
        )

        temp = final_test_annotations[
            final_test_annotations[possible_id_col]
            .astype(int)
            .isin(FINAL_TEST_IDS)
        ].copy()

        print(
            "Matching rows:",
            len(temp)
        )

        print(
            temp.to_string(index=False)
        )

        break

FINAL TEST ID / ANNOTATION MATCH CHECK

Using ID column: test_id
Matching rows: 40
 test_id                                                                                                                                                      question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [137]:
# ============================================================
# CELL 95 — RESTORE VERIFIED FINAL-TEST ANNOTATIONS
# ============================================================

print("=" * 70)
print("RESTORING VERIFIED FINAL-TEST ANNOTATIONS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check that the verified annotation dataframe exists
# ------------------------------------------------------------

assert "final_test_annotations" in globals(), \
    "ERROR: final_test_annotations is not loaded."

print("\nVerified annotation dataframe:")
print("  Rows:", len(final_test_annotations))
print("  Columns:", list(final_test_annotations.columns))

assert len(final_test_annotations) == 40, \
    "ERROR: Expected exactly 40 final-test annotations."

# ------------------------------------------------------------
# 2. Check required columns
# ------------------------------------------------------------

required_columns = [
    "test_id",
    "overall_label",
    "hallucination_spans",
    "final_decision",
    "annotator_notes"
]

for column in required_columns:

    assert column in final_test_annotations.columns, \
        f"ERROR: Missing column: {column}"

print("✅ Required annotation columns present.")

# ------------------------------------------------------------
# 3. Check IDs
# ------------------------------------------------------------

annotation_ids = set(
    final_test_annotations["test_id"].astype(int)
)

expected_ids = set(
    FINAL_TEST_IDS
)

missing_ids = expected_ids - annotation_ids
extra_ids = annotation_ids - expected_ids

assert len(missing_ids) == 0, \
    f"ERROR: Missing final-test IDs: {missing_ids}"

assert len(extra_ids) == 0, \
    f"ERROR: Unexpected final-test IDs: {extra_ids}"

print("✅ All 40 final-test IDs are present.")

# ------------------------------------------------------------
# 4. Build verified annotation lookups
# ------------------------------------------------------------

label_lookup = dict(
    zip(
        final_test_annotations["test_id"].astype(int),
        final_test_annotations["overall_label"]
    )
)

span_lookup = dict(
    zip(
        final_test_annotations["test_id"].astype(int),
        final_test_annotations["hallucination_spans"]
    )
)

decision_lookup = dict(
    zip(
        final_test_annotations["test_id"].astype(int),
        final_test_annotations["final_decision"]
    )
)

notes_lookup = dict(
    zip(
        final_test_annotations["test_id"].astype(int),
        final_test_annotations["annotator_notes"]
    )
)

# ------------------------------------------------------------
# 5. Attach annotations to final_test_df
# ------------------------------------------------------------

assert "final_test_df" in globals(), \
    "ERROR: final_test_df does not exist."

assert len(final_test_df) == 40, \
    "ERROR: final_test_df must contain 40 examples."

final_test_df = final_test_df.copy()

final_test_df["human_label"] = (
    final_test_df["test_id"]
    .astype(int)
    .map(label_lookup)
)

final_test_df["hallucination_spans"] = (
    final_test_df["test_id"]
    .astype(int)
    .map(span_lookup)
)

final_test_df["final_decision"] = (
    final_test_df["test_id"]
    .astype(int)
    .map(decision_lookup)
)

final_test_df["annotator_notes"] = (
    final_test_df["test_id"]
    .astype(int)
    .map(notes_lookup)
)

# ------------------------------------------------------------
# 6. Verify attachment
# ------------------------------------------------------------

assert final_test_df["human_label"].notna().all(), \
    "ERROR: Some examples did not receive human labels."

assert final_test_df["hallucination_spans"].notna().all(), \
    "ERROR: Some examples did not receive hallucination spans."

# ------------------------------------------------------------
# 7. Show distribution
# ------------------------------------------------------------

print("\nFinal-test label distribution:")
print(final_test_df["human_label"].value_counts())

print("\n" + "=" * 70)
print("✅ VERIFIED FINAL-TEST ANNOTATIONS RESTORED")
print("=" * 70)

RESTORING VERIFIED FINAL-TEST ANNOTATIONS

Verified annotation dataframe:
  Rows: 40
  Columns: ['test_id', 'question', 'context', 'answer', 'reference_answer', 'final_decision', 'overall_label', 'hallucination_spans', 'annotator_notes']
✅ Required annotation columns present.
✅ All 40 final-test IDs are present.

Final-test label distribution:
human_label
appropriate_abstention    23
partially_supported        9
hallucinated               8
Name: count, dtype: int64

✅ VERIFIED FINAL-TEST ANNOTATIONS RESTORED


In [139]:
# ============================================================
# CELL 96 — FIX / NORMALIZE FINAL-TEST HALLUCINATION SPANS
# ============================================================

import ast
import json

print("=" * 70)
print("NORMALIZING FINAL-TEST HALLUCINATION SPANS")
print("=" * 70)


def normalize_spans(x):
    """
    Convert hallucination_spans into a proper Python list.

    Handles:
    - Python lists
    - string representations of lists
    - JSON strings
    - empty / missing values
    """

    # Already a Python list
    if isinstance(x, list):
        return x

    # Missing value
    if x is None:
        return []

    # NaN
    try:
        if pd.isna(x):
            return []
    except:
        pass

    # String representation of a list
    if isinstance(x, str):
        x = x.strip()

        if x == "" or x == "[]":
            return []

        # Try Python representation
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return parsed
        except:
            pass

        # Try JSON representation
        try:
            parsed = json.loads(x)

            if isinstance(parsed, list):
                return parsed
        except:
            pass

    # Anything unexpected
    return []


# ------------------------------------------------------------
# Normalize the existing verified annotations
# ------------------------------------------------------------

final_test_df["hallucination_spans"] = (
    final_test_df["hallucination_spans"]
    .apply(normalize_spans)
)


# ------------------------------------------------------------
# Audit the normalized spans
# ------------------------------------------------------------

positive_examples = (
    final_test_df["hallucination_spans"]
    .apply(lambda x: len(x) > 0)
    .sum()
)

total_spans = (
    final_test_df["hallucination_spans"]
    .apply(len)
    .sum()
)


print("\nFinal-test label distribution:")
print(final_test_df["human_label"].value_counts())


print("\nExamples with positive hallucination spans:")
print(positive_examples)


print("\nTotal positive spans:")
print(total_spans)


# ------------------------------------------------------------
# Expected ground-truth checks
# ------------------------------------------------------------

assert len(final_test_df) == 40, \
    "ERROR: Final test must contain exactly 40 examples."

assert final_test_df["human_label"].value_counts().get(
    "appropriate_abstention", 0
) == 23, \
    "ERROR: Expected 23 appropriate_abstention examples."

assert final_test_df["human_label"].value_counts().get(
    "partially_supported", 0
) == 9, \
    "ERROR: Expected 9 partially_supported examples."

assert final_test_df["human_label"].value_counts().get(
    "hallucinated", 0
) == 8, \
    "ERROR: Expected 8 hallucinated examples."

assert positive_examples == 17, \
    f"ERROR: Expected 17 examples with positive hallucination spans, found {positive_examples}"


print("\n" + "=" * 70)
print("✅ FINAL-TEST ANNOTATION AUDIT PASSED")
print("=" * 70)

NORMALIZING FINAL-TEST HALLUCINATION SPANS

Final-test label distribution:
human_label
appropriate_abstention    23
partially_supported        9
hallucinated               8
Name: count, dtype: int64

Examples with positive hallucination spans:
17

Total positive spans:
26

✅ FINAL-TEST ANNOTATION AUDIT PASSED


Run this next: CELL 93

In [140]:
# ============================================================
# CELL 97 — VALIDATE FINAL-TEST CHARACTER SPANS
# ============================================================

print("=" * 70)
print("VALIDATING FINAL-TEST CHARACTER SPANS")
print("=" * 70)

invalid_spans = []

for _, row in final_test_df.iterrows():

    test_id = row["test_id"]
    answer = str(row["answer"])
    spans = row["hallucination_spans"]

    for i, span in enumerate(spans):

        start = span["start"]
        end = span["end"]

        # Check required fields
        if "start" not in span or "end" not in span:
            invalid_spans.append(
                (test_id, i, "Missing start/end")
            )
            continue

        # Check integer offsets
        if not isinstance(start, int) or not isinstance(end, int):
            invalid_spans.append(
                (test_id, i, "Non-integer offset")
            )
            continue

        # Check boundaries
        if start < 0 or end > len(answer):
            invalid_spans.append(
                (test_id, i, "Offset outside answer")
            )
            continue

        # Check valid range
        if start >= end:
            invalid_spans.append(
                (test_id, i, "Invalid start/end range")
            )
            continue

        # Check label
        if span.get("label") != "hallucinated":
            invalid_spans.append(
                (test_id, i, "Incorrect span label")
            )


print("\nTotal examples:", len(final_test_df))

print("Total annotated spans:",
      final_test_df["hallucination_spans"].apply(len).sum())

print("Invalid spans:", len(invalid_spans))


# ------------------------------------------------------------
# Show invalid spans if any
# ------------------------------------------------------------

if invalid_spans:

    print("\n❌ INVALID SPANS FOUND:")

    for item in invalid_spans:
        print(item)

    raise AssertionError(
        "Final-test span validation failed."
    )


print("\n✅ All 26 hallucination spans have valid character offsets.")

print("\n" + "=" * 70)
print("FINAL-TEST SPAN VALIDATION PASSED")
print("=" * 70)

VALIDATING FINAL-TEST CHARACTER SPANS

Total examples: 40
Total annotated spans: 26
Invalid spans: 0

✅ All 26 hallucination spans have valid character offsets.

FINAL-TEST SPAN VALIDATION PASSED


Cell 98 — Final ground-truth lock

In [141]:
# ============================================================
# CELL 98 — LOCK VERIFIED FINAL-TEST GROUND TRUTH
# ============================================================

print("=" * 70)
print("LOCKING VERIFIED FINAL-TEST GROUND TRUTH")
print("=" * 70)

# Create the evaluation-only copy
final_test_ground_truth = final_test_df.copy(deep=True)

# ------------------------------------------------------------
# Final safety checks
# ------------------------------------------------------------

assert len(final_test_ground_truth) == 40

assert (
    final_test_ground_truth["human_label"]
    .value_counts()
    .get("appropriate_abstention", 0)
    == 23
)

assert (
    final_test_ground_truth["human_label"]
    .value_counts()
    .get("partially_supported", 0)
    == 9
)

assert (
    final_test_ground_truth["human_label"]
    .value_counts()
    .get("hallucinated", 0)
    == 8
)

positive_examples = (
    final_test_ground_truth["hallucination_spans"]
    .apply(lambda x: len(x) > 0)
    .sum()
)

total_spans = (
    final_test_ground_truth["hallucination_spans"]
    .apply(len)
    .sum()
)

assert positive_examples == 17
assert total_spans == 26

print("\nFinal-test ground truth:")
print("  Examples              :", len(final_test_ground_truth))
print("  Positive-span examples:", positive_examples)
print("  Total positive spans  :", total_spans)

print("\nLabel distribution:")
print(
    final_test_ground_truth["human_label"]
    .value_counts()
)

print("\n" + "=" * 70)
print("✅ FINAL-TEST GROUND TRUTH LOCKED")
print("=" * 70)

LOCKING VERIFIED FINAL-TEST GROUND TRUTH

Final-test ground truth:
  Examples              : 40
  Positive-span examples: 17
  Total positive spans  : 26

Label distribution:
human_label
appropriate_abstention    23
partially_supported        9
hallucinated               8
Name: count, dtype: int64

✅ FINAL-TEST GROUND TRUTH LOCKED


In [142]:
# ============================================================
# CELL 99 — INSPECT EXISTING MODEL / INFERENCE OBJECTS
# ============================================================

print("=" * 70)
print("CHECKING EXISTING MODEL / INFERENCE OBJECTS")
print("=" * 70)

# List important variables that already exist
important_names = [
    "model",
    "tokenizer",
    "test_records",
    "final_test_df",
    "final_test_ground_truth",
    "verified_df",
    "train_records",
    "val_records",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "trainer",
    "training_args"
]

print("\nExisting objects:\n")

for name in important_names:
    if name in globals():
        obj = globals()[name]

        print(f"✅ {name}")
        print(f"   Type: {type(obj).__name__}")

        # Show useful size information when available
        try:
            print(f"   Length: {len(obj)}")
        except:
            pass

        print()
    else:
        print(f"❌ {name} — not found\n")

print("=" * 70)
print("MODEL / INFERENCE OBJECT INSPECTION COMPLETE")
print("=" * 70)

CHECKING EXISTING MODEL / INFERENCE OBJECTS

Existing objects:

❌ model — not found

✅ tokenizer
   Type: TokenizersBackend
   Length: 50368

✅ test_records
   Type: list
   Length: 40

✅ final_test_df
   Type: DataFrame
   Length: 40

✅ final_test_ground_truth
   Type: DataFrame
   Length: 40

✅ verified_df
   Type: DataFrame
   Length: 200

✅ train_records
   Type: list
   Length: 51

❌ val_records — not found

✅ train_dataset
   Type: RagTruthDataset
   Length: 140

❌ val_dataset — not found

✅ test_dataset
   Type: RagTruthDataset
   Length: 40

❌ trainer — not found

❌ training_args — not found

MODEL / INFERENCE OBJECT INSPECTION COMPLETE


In [143]:
# ============================================================
# CELL 100 — INSPECT EXISTING TRAINING PIPELINE
# ============================================================

print("=" * 70)
print("INSPECTING EXISTING TRAINING PIPELINE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Inspect train_dataset
# ------------------------------------------------------------

print("\n[1] TRAIN DATASET")
print("-" * 70)

print("Type:", type(train_dataset).__name__)

print("\nDataset attributes:")

for name in dir(train_dataset):
    if not name.startswith("_"):
        try:
            value = getattr(train_dataset, name)

            if callable(value):
                print(f"  {name} : <method/function>")
            else:
                print(f"  {name} : {type(value).__name__}")

        except Exception:
            pass


# ------------------------------------------------------------
# 2. Inspect one dataset item
# ------------------------------------------------------------

print("\n[2] FIRST TRAIN DATASET ITEM")
print("-" * 70)

sample = train_dataset[0]

print("Type:", type(sample).__name__)

if isinstance(sample, dict):
    print("\nKeys:")
    for key, value in sample.items():
        print(f"  {key}: {type(value).__name__}")

        try:
            print(f"      shape/length: {getattr(value, 'shape', len(value))}")
        except:
            pass

else:
    print(sample)


# ------------------------------------------------------------
# 3. Inspect RagTruthDataset source
# ------------------------------------------------------------

print("\n[3] RagTruthDataset DEFINITION")
print("-" * 70)

try:
    import inspect

    print(inspect.getsource(RagTruthDataset))

except Exception as e:
    print("Could not display source:", e)


# ------------------------------------------------------------
# 4. Find model-related classes/functions already defined
# ------------------------------------------------------------

print("\n[4] MODEL-RELATED OBJECTS")
print("-" * 70)

model_keywords = [
    "model",
    "classifier",
    "encoder",
    "transformer",
    "bert",
    "roberta",
    "deberta",
    "llama",
    "qwen",
    "electra",
    "train",
    "forward",
    "predict",
    "inference",
    "evaluate"
]

found = []

for name, obj in globals().items():

    if name.startswith("_"):
        continue

    name_lower = name.lower()

    if any(keyword in name_lower for keyword in model_keywords):

        found.append((name, type(obj).__name__))


for name, obj_type in sorted(found):
    print(f"  {name} : {obj_type}")


print("\n" + "=" * 70)
print("TRAINING PIPELINE INSPECTION COMPLETE")
print("=" * 70)

INSPECTING EXISTING TRAINING PIPELINE

[1] TRAIN DATASET
----------------------------------------------------------------------
Type: RagTruthDataset

Dataset attributes:
  max_length : int
  prepare_tokenized_input : <method/function>
  samples : list
  tokenizer : <method/function>

[2] FIRST TRAIN DATASET ITEM
----------------------------------------------------------------------
Type: dict

Keys:
  input_ids: Tensor
      shape/length: torch.Size([421])
  attention_mask: Tensor
      shape/length: torch.Size([421])
  labels: Tensor
      shape/length: torch.Size([421])

[3] RagTruthDataset DEFINITION
----------------------------------------------------------------------
class RagTruthDataset(Dataset):
    """Dataset for RAG truth data."""

    def __init__(
        self,
        samples: list[RagTruthSample],
        tokenizer: AutoTokenizer,
        max_length: int = 4096,
    ):
        """Initialize the dataset.

        :param samples: List of RagTruthSample objects.
        :p

In [144]:
# ============================================================
# CELL 101 — INSPECT LOADED MEDICAL HALLUCINATION MODEL
# ============================================================

print("=" * 70)
print("INSPECTING LOADED MEDICAL MODEL")
print("=" * 70)

# ------------------------------------------------------------
# 1. Basic model information
# ------------------------------------------------------------

print("\n[1] MODEL")
print("-" * 70)

print("Type:", type(medical_model).__name__)

print("Model class:", medical_model.__class__.__name__)


# ------------------------------------------------------------
# 2. Model configuration
# ------------------------------------------------------------

print("\n[2] MODEL CONFIGURATION")
print("-" * 70)

print("Number of labels:", medical_model.config.num_labels)
print("Problem type:", getattr(
    medical_model.config,
    "problem_type",
    "Not specified"
))

print("Label mapping:")
print(medical_model.config.id2label)

print("Reverse label mapping:")
print(medical_model.config.label2id)


# ------------------------------------------------------------
# 3. Check model device
# ------------------------------------------------------------

print("\n[3] DEVICE")
print("-" * 70)

try:
    device = next(medical_model.parameters()).device
    print("Model device:", device)
except Exception as e:
    print("Could not determine device:", e)


# ------------------------------------------------------------
# 4. Check training/evaluation mode
# ------------------------------------------------------------

print("\n[4] MODEL MODE")
print("-" * 70)

print("Training mode:", medical_model.training)

if medical_model.training:
    print("⚠️ Model is currently in training mode.")
else:
    print("✅ Model is currently in evaluation mode.")


# ------------------------------------------------------------
# 5. Check parameter count
# ------------------------------------------------------------

print("\n[5] PARAMETERS")
print("-" * 70)

total_parameters = sum(
    p.numel()
    for p in medical_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in medical_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")


# ------------------------------------------------------------
# 6. Check existing prediction objects
# ------------------------------------------------------------

print("\n[6] EXISTING PREDICTIONS")
print("-" * 70)

for name in [
    "predictions",
    "token_predictions",
    "adapted_predictions",
    "adapted_token_predictions",
    "original_predictions",
    "original_test_predictions"
]:
    if name in globals():
        obj = globals()[name]

        try:
            size = len(obj)
        except:
            size = "N/A"

        print(f"✅ {name}: {type(obj).__name__}, length={size}")
    else:
        print(f"❌ {name}: not found")


print("\n" + "=" * 70)
print("MEDICAL MODEL INSPECTION COMPLETE")
print("=" * 70)

INSPECTING LOADED MEDICAL MODEL

[1] MODEL
----------------------------------------------------------------------
Type: ModernBertForTokenClassification
Model class: ModernBertForTokenClassification

[2] MODEL CONFIGURATION
----------------------------------------------------------------------
Number of labels: 2
Problem type: None
Label mapping:
{0: 'LABEL_0', 1: 'LABEL_1'}
Reverse label mapping:
{'LABEL_0': 0, 'LABEL_1': 1}

[3] DEVICE
----------------------------------------------------------------------
Model device: cuda:0

[4] MODEL MODE
----------------------------------------------------------------------
Training mode: True
⚠️ Model is currently in training mode.

[5] PARAMETERS
----------------------------------------------------------------------
Total parameters: 149,606,402
Trainable parameters: 149,606,402

[6] EXISTING PREDICTIONS
----------------------------------------------------------------------
✅ predictions: list, length=0
✅ token_predictions: list, length=11
✅ ad

In [145]:
# ============================================================
# CELL 102 — PREPARE MEDICAL MODEL FOR FINAL-TEST INFERENCE
# ============================================================

print("=" * 70)
print("PREPARING MEDICAL MODEL FOR INFERENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Switch to evaluation mode
# ------------------------------------------------------------

medical_model.eval()

print("\nModel training mode:", medical_model.training)

assert medical_model.training is False, \
    "ERROR: Model is still in training mode."


# ------------------------------------------------------------
# 2. Confirm the model is still on CUDA
# ------------------------------------------------------------

device = next(medical_model.parameters()).device

print("Model device:", device)

assert str(device).startswith("cuda"), \
    "ERROR: Expected medical_model to be on CUDA."


# ------------------------------------------------------------
# 3. Confirm model configuration
# ------------------------------------------------------------

print("\nNumber of labels:", medical_model.config.num_labels)

assert medical_model.config.num_labels == 2, \
    "ERROR: Expected a 2-label token-classification model."


# ------------------------------------------------------------
# 4. Inspect one final-test dataset item
# ------------------------------------------------------------

print("\nFinal-test dataset:")
print("  Number of items:", len(test_dataset))

assert len(test_dataset) == 40, \
    "ERROR: test_dataset must contain exactly 40 examples."

sample = test_dataset[0]

print("\nFirst test item:")
print("  Keys:", list(sample.keys()))
print("  input_ids shape:", sample["input_ids"].shape)
print("  attention_mask shape:", sample["attention_mask"].shape)
print("  labels shape:", sample["labels"].shape)


# ------------------------------------------------------------
# 5. Verify ground truth is still untouched
# ------------------------------------------------------------

print("\nGround-truth verification:")

assert len(final_test_ground_truth) == 40

positive_examples = (
    final_test_ground_truth["hallucination_spans"]
    .apply(lambda x: len(x) > 0)
    .sum()
)

total_spans = (
    final_test_ground_truth["hallucination_spans"]
    .apply(len)
    .sum()
)

print("  Examples:", len(final_test_ground_truth))
print("  Positive-span examples:", positive_examples)
print("  Total spans:", total_spans)

assert positive_examples == 17
assert total_spans == 26


print("\n" + "=" * 70)
print("✅ MODEL READY FOR FINAL-TEST INFERENCE")
print("=" * 70)

PREPARING MEDICAL MODEL FOR INFERENCE

Model training mode: False
Model device: cuda:0

Number of labels: 2

Final-test dataset:
  Number of items: 40

First test item:
  Keys: ['input_ids', 'attention_mask', 'labels']
  input_ids shape: torch.Size([465])
  attention_mask shape: torch.Size([465])
  labels shape: torch.Size([465])

Ground-truth verification:
  Examples: 40
  Positive-span examples: 17
  Total spans: 26

✅ MODEL READY FOR FINAL-TEST INFERENCE


In [146]:
# ============================================================
# CELL 103 — RUN MEDICAL MODEL ON FINAL TEST
# ============================================================

import torch

print("=" * 70)
print("RUNNING MEDICAL MODEL ON FINAL TEST")
print("=" * 70)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert medical_model.training is False
assert len(test_dataset) == 40
assert len(final_test_ground_truth) == 40

# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------

model_predictions = []

with torch.no_grad():

    for i in range(len(test_dataset)):

        item = test_dataset[i]

        input_ids = item["input_ids"].unsqueeze(0).to(device)
        attention_mask = item["attention_mask"].unsqueeze(0).to(device)

        outputs = medical_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # logits shape:
        # [batch_size, sequence_length, num_labels]
        logits = outputs.logits

        # Predicted class for every token
        predicted_labels = torch.argmax(
            logits,
            dim=-1
        )

        # Move prediction back to CPU
        predicted_labels = predicted_labels.squeeze(0).cpu().tolist()

        model_predictions.append(predicted_labels)

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/40 examples")


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert len(model_predictions) == 40

assert all(
    isinstance(x, list)
    for x in model_predictions
)

print("\nPrediction results:")
print("  Number of examples:", len(model_predictions))

print(
    "  First prediction length:",
    len(model_predictions[0])
)

print(
    "  First test input length:",
    len(test_dataset[0]["input_ids"])
)

print("\nUnique predicted labels in first example:")
print(sorted(set(model_predictions[0])))


print("\n" + "=" * 70)
print("✅ FINAL-TEST INFERENCE COMPLETE")
print("=" * 70)

RUNNING MEDICAL MODEL ON FINAL TEST
Processed 10/40 examples
Processed 20/40 examples
Processed 30/40 examples
Processed 40/40 examples

Prediction results:
  Number of examples: 40
  First prediction length: 465
  First test input length: 465

Unique predicted labels in first example:
[0, 1]

✅ FINAL-TEST INFERENCE COMPLETE


In [147]:
# ============================================================
# CELL 104 — CONVERT TOKEN PREDICTIONS TO CHARACTER SPANS
# ============================================================

print("=" * 70)
print("CONVERTING TOKEN PREDICTIONS TO CHARACTER SPANS")
print("=" * 70)


def get_predicted_spans(
    tokenizer,
    context,
    answer,
    predicted_labels,
    max_length=4096
):
    """
    Convert token-level predictions into character-level
    hallucination spans relative to the answer text.
    """

    # Tokenize context + answer with offset mappings
    encoding = tokenizer(
        context,
        answer,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        add_special_tokens=True
    )

    offsets = encoding["offset_mapping"]

    # Find where answer tokens begin
    prompt_tokens = tokenizer.encode(
        context,
        add_special_tokens=False
    )

    answer_start_token = 1 + len(prompt_tokens) + 1

    # Safety: prediction and offset lengths must match
    usable_length = min(
        len(predicted_labels),
        len(offsets)
    )

    if answer_start_token >= usable_length:
        return []

    # Character offset of first answer token
    answer_char_offset = offsets[answer_start_token][0]

    token_spans = []

    # --------------------------------------------------------
    # Collect predicted hallucinated tokens
    # --------------------------------------------------------

    for i in range(answer_start_token, usable_length):

        if predicted_labels[i] != 1:
            continue

        token_start, token_end = offsets[i]

        # Convert absolute token offset to answer-relative offset
        start = token_start - answer_char_offset
        end = token_end - answer_char_offset

        # Ignore empty/special-token offsets
        if end <= start:
            continue

        token_spans.append((start, end))

    # --------------------------------------------------------
    # Merge consecutive/overlapping hallucinated tokens
    # --------------------------------------------------------

    if not token_spans:
        return []

    token_spans.sort()

    merged = [list(token_spans[0])]

    for start, end in token_spans[1:]:

        previous_start, previous_end = merged[-1]

        if start <= previous_end:
            merged[-1][1] = max(previous_end, end)

        else:
            merged.append([start, end])

    return [
        {
            "start": start,
            "end": end,
            "label": "hallucinated"
        }
        for start, end in merged
    ]


# ------------------------------------------------------------
# Generate predicted spans for all 40 test examples
# ------------------------------------------------------------

predicted_spans = []

for i, row in final_test_ground_truth.iterrows():

    spans = get_predicted_spans(
        tokenizer=tokenizer,
        context=str(row["context"]),
        answer=str(row["answer"]),
        predicted_labels=model_predictions[i]
    )

    predicted_spans.append(spans)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/40 examples")


# ------------------------------------------------------------
# Store predictions separately
# ------------------------------------------------------------

final_test_predictions = final_test_ground_truth[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "human_label",
        "hallucination_spans"
    ]
].copy()

final_test_predictions["predicted_spans"] = predicted_spans


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

predicted_positive_examples = sum(
    len(spans) > 0
    for spans in predicted_spans
)

predicted_total_spans = sum(
    len(spans)
    for spans in predicted_spans
)

print("\nPrediction span summary:")
print("  Test examples:", len(predicted_spans))
print("  Examples with predicted spans:", predicted_positive_examples)
print("  Total predicted spans:", predicted_total_spans)

print("\n" + "=" * 70)
print("✅ TOKEN-TO-CHARACTER SPAN CONVERSION COMPLETE")
print("=" * 70)

CONVERTING TOKEN PREDICTIONS TO CHARACTER SPANS
Processed 10/40 examples
Processed 20/40 examples
Processed 30/40 examples
Processed 40/40 examples

Prediction span summary:
  Test examples: 40
  Examples with predicted spans: 14
  Total predicted spans: 40

✅ TOKEN-TO-CHARACTER SPAN CONVERSION COMPLETE


In [148]:
# ============================================================
# CELL 105 — FINAL-TEST HALLUCINATION EVALUATION
# ============================================================

print("=" * 70)
print("FINAL-TEST HALLUCINATION EVALUATION")
print("=" * 70)


def spans_overlap(span_a, span_b):
    """Return True if two character spans overlap."""

    return (
        span_a["start"] < span_b["end"]
        and
        span_b["start"] < span_a["end"]
    )


# ------------------------------------------------------------
# 1. Span-level matching
# ------------------------------------------------------------

true_positive_spans = 0
matched_predicted_spans = set()
matched_human_spans = set()

for idx, row in final_test_predictions.iterrows():

    human_spans = row["hallucination_spans"]
    predicted = row["predicted_spans"]

    for pred_idx, pred_span in enumerate(predicted):

        for human_idx, human_span in enumerate(human_spans):

            if (
                human_idx not in matched_human_spans
                and
                spans_overlap(pred_span, human_span)
            ):

                true_positive_spans += 1

                matched_predicted_spans.add(
                    (idx, pred_idx)
                )

                matched_human_spans.add(
                    (idx, human_idx)
                )

                break


# ------------------------------------------------------------
# 2. Calculate span-level precision / recall / F1
# ------------------------------------------------------------

total_predicted_spans = sum(
    len(x)
    for x in final_test_predictions["predicted_spans"]
)

total_human_spans = sum(
    len(x)
    for x in final_test_predictions["hallucination_spans"]
)

false_positive_spans = (
    total_predicted_spans - true_positive_spans
)

false_negative_spans = (
    total_human_spans - true_positive_spans
)

span_precision = (
    true_positive_spans / total_predicted_spans
    if total_predicted_spans > 0
    else 0.0
)

span_recall = (
    true_positive_spans / total_human_spans
    if total_human_spans > 0
    else 0.0
)

span_f1 = (
    2 * span_precision * span_recall /
    (span_precision + span_recall)
    if (span_precision + span_recall) > 0
    else 0.0
)


# ------------------------------------------------------------
# 3. Example-level classification
#
# Positive = example contains at least one hallucination span
# ------------------------------------------------------------

human_positive = final_test_predictions[
    "hallucination_spans"
].apply(lambda x: len(x) > 0)

predicted_positive = final_test_predictions[
    "predicted_spans"
].apply(lambda x: len(x) > 0)


example_tp = int(
    (human_positive & predicted_positive).sum()
)

example_tn = int(
    (~human_positive & ~predicted_positive).sum()
)

example_fp = int(
    (~human_positive & predicted_positive).sum()
)

example_fn = int(
    (human_positive & ~predicted_positive).sum()
)


example_precision = (
    example_tp / (example_tp + example_fp)
    if (example_tp + example_fp) > 0
    else 0.0
)

example_recall = (
    example_tp / (example_tp + example_fn)
    if (example_tp + example_fn) > 0
    else 0.0
)

example_f1 = (
    2 * example_precision * example_recall /
    (example_precision + example_recall)
    if (example_precision + example_recall) > 0
    else 0.0
)

example_accuracy = (
    (example_tp + example_tn) / len(final_test_predictions)
)


# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------

print("\nSPAN-LEVEL RESULTS")
print("-" * 70)

print("Human spans      :", total_human_spans)
print("Predicted spans  :", total_predicted_spans)
print("True positive    :", true_positive_spans)
print("False positive   :", false_positive_spans)
print("False negative   :", false_negative_spans)

print(
    f"Precision        : {span_precision:.4f}"
)

print(
    f"Recall           : {span_recall:.4f}"
)

print(
    f"F1               : {span_f1:.4f}"
)


print("\nEXAMPLE-LEVEL RESULTS")
print("-" * 70)

print("Human positive examples     :", int(human_positive.sum()))
print("Predicted positive examples :", int(predicted_positive.sum()))

print("True positive examples      :", example_tp)
print("True negative examples      :", example_tn)
print("False positive examples     :", example_fp)
print("False negative examples     :", example_fn)

print(
    f"Precision                   : {example_precision:.4f}"
)

print(
    f"Recall                      : {example_recall:.4f}"
)

print(
    f"F1                         : {example_f1:.4f}"
)

print(
    f"Accuracy                   : {example_accuracy:.4f}"
)


print("\n" + "=" * 70)
print("✅ FINAL-TEST EVALUATION COMPLETE")
print("=" * 70)

FINAL-TEST HALLUCINATION EVALUATION

SPAN-LEVEL RESULTS
----------------------------------------------------------------------
Human spans      : 26
Predicted spans  : 40
True positive    : 2
False positive   : 38
False negative   : 24
Precision        : 0.0500
Recall           : 0.0769
F1               : 0.0606

EXAMPLE-LEVEL RESULTS
----------------------------------------------------------------------
Human positive examples     : 17
Predicted positive examples : 14
True positive examples      : 5
True negative examples      : 14
False positive examples     : 9
False negative examples     : 12
Precision                   : 0.3571
Recall                      : 0.2941
F1                         : 0.3226
Accuracy                   : 0.4750

✅ FINAL-TEST EVALUATION COMPLETE


In [149]:
# ============================================================
# CELL 106 — INSPECT HUMAN VS MODEL SPANS
# ============================================================

print("=" * 70)
print("INSPECTING HUMAN VS MODEL HALLUCINATION SPANS")
print("=" * 70)

for i in range(len(final_test_predictions)):

    row = final_test_predictions.iloc[i]

    human_spans = row["hallucination_spans"]
    predicted_spans = row["predicted_spans"]

    # Only show examples where either side has spans
    if len(human_spans) == 0 and len(predicted_spans) == 0:
        continue

    print("\n" + "-" * 70)

    print("Test ID:", row["test_id"])
    print("Human label:", row["human_label"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nHUMAN SPANS:")
    print(human_spans)

    print("\nMODEL PREDICTED SPANS:")
    print(predicted_spans)

    # Show actual text represented by each human span
    print("\nHUMAN HALLUCINATED TEXT:")

    for span in human_spans:
        start = span["start"]
        end = span["end"]

        print(
            f"  [{start}:{end}] -> "
            f"{row['answer'][start:end]!r}"
        )

    # Show actual text represented by each predicted span
    print("\nMODEL PREDICTED TEXT:")

    for span in predicted_spans:
        start = span["start"]
        end = span["end"]

        print(
            f"  [{start}:{end}] -> "
            f"{row['answer'][start:end]!r}"
        )

print("\n" + "=" * 70)
print("SPAN INSPECTION COMPLETE")
print("=" * 70)

INSPECTING HUMAN VS MODEL HALLUCINATION SPANS

----------------------------------------------------------------------
Test ID: 280
Human label: hallucinated

ANSWER:
Based on the provided evidence, there is no direct correlation found between high blood pressure and reduced risk of chronic low back pain. In fact, some research suggests that cardiovascular diseases, which can lead to elevated blood pressure, might increase the risk of developing chronic low back pain. Additionally, the evidence does not support a clear link between high blood pressure and lower incidence of chronic low back pain. Instead, other conditions like emotional distress, particularly depression, have been associated with higher rates of chronic low back pain. Therefore, while high blood pressure itself may not directly reduce the risk of chronic low back pain, other health issues linked to hypertension could potentially contribute to its development.

HUMAN SPANS:
[{'start': 149, 'end': 305, 'label': 'hallucina

Cell 107 — check tokenizer structure

In [150]:
# ============================================================
# CELL 107 — CHECK TOKENIZER / ANSWER OFFSET ALIGNMENT
# ============================================================

print("=" * 70)
print("CHECKING TOKENIZER / ANSWER OFFSET ALIGNMENT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Tokenizer information
# ------------------------------------------------------------

print("\n[1] TOKENIZER")
print("-" * 70)

print("Tokenizer type:", type(tokenizer).__name__)
print("Tokenizer class:", tokenizer.__class__.__name__)

print("\nSpecial tokens:")
print("  CLS:", tokenizer.cls_token)
print("  SEP:", tokenizer.sep_token)
print("  BOS:", tokenizer.bos_token)
print("  EOS:", tokenizer.eos_token)
print("  PAD:", tokenizer.pad_token)


# ------------------------------------------------------------
# 2. Take first final-test example
# ------------------------------------------------------------

row = final_test_ground_truth.iloc[0]

context = str(row["context"])
answer = str(row["answer"])

print("\n[2] FIRST FINAL-TEST EXAMPLE")
print("-" * 70)

print("Test ID:", row["test_id"])
print("Context characters:", len(context))
print("Answer characters:", len(answer))


# ------------------------------------------------------------
# 3. Tokenize with offsets
# ------------------------------------------------------------

encoding = tokenizer(
    context,
    answer,
    truncation=True,
    max_length=4096,
    return_offsets_mapping=True,
    add_special_tokens=True
)

input_ids = encoding["input_ids"]
offsets = encoding["offset_mapping"]

print("\n[3] TOKENIZATION")
print("-" * 70)

print("Number of tokens:", len(input_ids))
print("Number of offsets:", len(offsets))


# ------------------------------------------------------------
# 4. Calculate the answer start using the SAME formula
#    used by RagTruthDataset
# ------------------------------------------------------------

prompt_tokens = tokenizer.encode(
    context,
    add_special_tokens=False
)

answer_start_token = 1 + len(prompt_tokens) + 1

print("\n[4] ANSWER START CALCULATION")
print("-" * 70)

print("Context token count:", len(prompt_tokens))
print("Calculated answer_start_token:", answer_start_token)

if answer_start_token < len(offsets):

    print(
        "Offset at calculated answer start:",
        offsets[answer_start_token]
    )

else:

    print("⚠️ Calculated answer start is outside token sequence.")


# ------------------------------------------------------------
# 5. Print tokens around the calculated answer boundary
# ------------------------------------------------------------

print("\n[5] TOKENS AROUND CALCULATED ANSWER START")
print("-" * 70)

start = max(0, answer_start_token - 5)
end = min(len(input_ids), answer_start_token + 10)

tokens = tokenizer.convert_ids_to_tokens(
    input_ids[start:end]
)

for position, token, offset in zip(
    range(start, end),
    tokens,
    offsets[start:end]
):

    marker = "  <-- CALCULATED ANSWER START" \
        if position == answer_start_token else ""

    print(
        f"{position:4d} | "
        f"{str(token):30s} | "
        f"offset={offset}{marker}"
    )


# ------------------------------------------------------------
# 6. Check whether the calculated offset actually points
#    into the answer text
# ------------------------------------------------------------

print("\n[6] OFFSET ALIGNMENT CHECK")
print("-" * 70)

if answer_start_token < len(offsets):

    first_offset = offsets[answer_start_token]

    print("First answer-token offset:", first_offset)

    if first_offset[0] < len(context):
        print(
            "⚠️ WARNING: Calculated answer offset appears "
            "to still be inside the context."
        )
    else:
        print(
            "✅ Calculated answer offset is beyond the "
            "context boundary."
        )


print("\n" + "=" * 70)
print("TOKENIZER ALIGNMENT CHECK COMPLETE")
print("=" * 70)

CHECKING TOKENIZER / ANSWER OFFSET ALIGNMENT

[1] TOKENIZER
----------------------------------------------------------------------
Tokenizer type: TokenizersBackend
Tokenizer class: TokenizersBackend

Special tokens:
  CLS: [CLS]
  SEP: [SEP]
  BOS: None
  EOS: None
  PAD: [PAD]

[2] FIRST FINAL-TEST EXAMPLE
----------------------------------------------------------------------
Test ID: 136
Context characters: 1378
Answer characters: 344

[3] TOKENIZATION
----------------------------------------------------------------------
Number of tokens: 324
Number of offsets: 324

[4] ANSWER START CALCULATION
----------------------------------------------------------------------
Context token count: 270
Calculated answer_start_token: 272
Offset at calculated answer start: (0, 3)

[5] TOKENS AROUND CALCULATED ANSWER START
----------------------------------------------------------------------
 267 | Ġwith                          | offset=(1355, 1360)
 268 | Ġcurrent                       | offset=

In [151]:
# ============================================================
# CELL 108 — FIX ANSWER-TO-CHARACTER SPAN MAPPING
# ============================================================

print("=" * 70)
print("FIXING ANSWER-TO-CHARACTER SPAN MAPPING")
print("=" * 70)

def get_predicted_spans_fixed(
    tokenizer,
    context,
    answer,
    predicted_labels,
    max_length=4096
):
    """
    Convert token-level hallucination predictions into
    character-level spans within the ANSWER.

    Uses sequence_ids() to reliably identify answer tokens.
    """

    encoding = tokenizer(
        context,
        answer,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        add_special_tokens=True
    )

    offsets = encoding["offset_mapping"]

    # Identify which tokens belong to:
    # 0 = context
    # 1 = answer
    sequence_ids = encoding.sequence_ids()

    token_spans = []

    usable_length = min(
        len(predicted_labels),
        len(offsets)
    )

    for i in range(usable_length):

        # Only consider ANSWER tokens
        if sequence_ids[i] != 1:
            continue

        # Only hallucination label = 1
        if predicted_labels[i] != 1:
            continue

        token_start, token_end = offsets[i]

        # Ignore empty/special-token offsets
        if token_end <= token_start:
            continue

        # Offsets are already relative to the answer
        token_spans.append(
            (token_start, token_end)
        )

    if not token_spans:
        return []

    # Sort spans
    token_spans.sort()

    # Merge overlapping/adjacent token spans
    merged = [list(token_spans[0])]

    for start, end in token_spans[1:]:

        previous_start, previous_end = merged[-1]

        if start <= previous_end:
            merged[-1][1] = max(
                previous_end,
                end
            )
        else:
            merged.append(
                [start, end]
            )

    return [
        {
            "start": start,
            "end": end,
            "label": "hallucinated"
        }
        for start, end in merged
    ]


print("\nTesting corrected mapping on Test ID 136...")

row = final_test_ground_truth.iloc[0]

test_encoding = tokenizer(
    str(row["context"]),
    str(row["answer"]),
    truncation=True,
    max_length=4096,
    return_offsets_mapping=True,
    add_special_tokens=True
)

sequence_ids = test_encoding.sequence_ids()

print("Sequence IDs around answer:")
for i in range(267, min(282, len(sequence_ids))):
    print(
        f"Token {i}: "
        f"sequence_id={sequence_ids[i]}, "
        f"offset={test_encoding['offset_mapping'][i]}"
    )

print("\nExpected:")
print("  Context tokens -> sequence_id = 0")
print("  [SEP]          -> sequence_id = None")
print("  Answer tokens  -> sequence_id = 1")

assert sequence_ids[272] == 1, \
    "ERROR: Token 272 was not identified as an answer token."

assert test_encoding["offset_mapping"][272] == (0, 3), \
    "ERROR: Unexpected answer offset."

print("\n✅ Answer token identification is correct.")

print("\n" + "=" * 70)
print("MAPPING FUNCTION READY")
print("=" * 70)

FIXING ANSWER-TO-CHARACTER SPAN MAPPING

Testing corrected mapping on Test ID 136...
Sequence IDs around answer:
Token 267: sequence_id=0, offset=(1355, 1360)
Token 268: sequence_id=0, offset=(1360, 1368)
Token 269: sequence_id=0, offset=(1368, 1377)
Token 270: sequence_id=0, offset=(1377, 1378)
Token 271: sequence_id=None, offset=(0, 0)
Token 272: sequence_id=1, offset=(0, 3)
Token 273: sequence_id=1, offset=(3, 13)
Token 274: sequence_id=1, offset=(13, 22)
Token 275: sequence_id=1, offset=(22, 25)
Token 276: sequence_id=1, offset=(25, 38)
Token 277: sequence_id=1, offset=(38, 41)
Token 278: sequence_id=1, offset=(41, 48)
Token 279: sequence_id=1, offset=(48, 53)
Token 280: sequence_id=1, offset=(53, 62)
Token 281: sequence_id=1, offset=(62, 63)

Expected:
  Context tokens -> sequence_id = 0
  [SEP]          -> sequence_id = None
  Answer tokens  -> sequence_id = 1

✅ Answer token identification is correct.

MAPPING FUNCTION READY


In [152]:
# ============================================================
# CELL 109 — REBUILD FINAL-TEST PREDICTED SPANS CORRECTLY
# ============================================================

print("=" * 70)
print("REBUILDING FINAL-TEST PREDICTED SPANS")
print("=" * 70)

# Use the corrected mapping function from Cell 108
corrected_predictions = []

for idx, row in final_test_ground_truth.iterrows():

    spans = get_predicted_spans_fixed(
        tokenizer=tokenizer,
        context=str(row["context"]),
        answer=str(row["answer"]),
        predicted_labels=model_predictions[idx],
        max_length=4096
    )

    corrected_predictions.append(spans)

# Create evaluation dataframe
final_test_predictions_fixed = final_test_ground_truth[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "human_label",
        "hallucination_spans"
    ]
].copy()

final_test_predictions_fixed["predicted_spans"] = corrected_predictions

print("\n[1] BASIC CHECKS")
print("-" * 70)

print(
    "Number of test examples:",
    len(final_test_predictions_fixed)
)

print(
    "Examples with predicted spans:",
    sum(
        len(x) > 0
        for x in corrected_predictions
    )
)

print(
    "Total predicted spans:",
    sum(
        len(x)
        for x in corrected_predictions
    )
)

assert len(final_test_predictions_fixed) == 40
assert len(corrected_predictions) == 40

print("✅ 40 predictions preserved.")

# ------------------------------------------------------------
# Show first few predictions
# ------------------------------------------------------------

print("\n[2] SAMPLE PREDICTIONS")
print("-" * 70)

for _, row in final_test_predictions_fixed.head(5).iterrows():

    print("\nTest ID:", row["test_id"])
    print("Human label:", row["human_label"])
    print("Human spans:", row["hallucination_spans"])
    print("Predicted spans:", row["predicted_spans"])

    if row["predicted_spans"]:
        print("Predicted text:")

        answer = row["answer"]

        for span in row["predicted_spans"]:
            start = span["start"]
            end = span["end"]

            print(
                "  →",
                repr(answer[start:end])
            )

print("\n" + "=" * 70)
print("✅ CORRECTED FINAL-TEST PREDICTIONS CREATED")
print("=" * 70)

REBUILDING FINAL-TEST PREDICTED SPANS

[1] BASIC CHECKS
----------------------------------------------------------------------
Number of test examples: 40
Examples with predicted spans: 14
Total predicted spans: 40
✅ 40 predictions preserved.

[2] SAMPLE PREDICTIONS
----------------------------------------------------------------------

Test ID: 136
Human label: appropriate_abstention
Human spans: []
Predicted spans: []

Test ID: 899
Human label: appropriate_abstention
Human spans: []
Predicted spans: []

Test ID: 280
Human label: hallucinated
Human spans: [{'start': 149, 'end': 305, 'label': 'hallucinated'}, {'start': 681, 'end': 772, 'label': 'hallucinated'}]
Predicted spans: []

Test ID: 761
Human label: hallucinated
Human spans: [{'start': 688, 'end': 832, 'label': 'hallucinated'}, {'start': 481, 'end': 600, 'label': 'hallucinated'}]
Predicted spans: []

Test ID: 346
Human label: hallucinated
Human spans: [{'start': 0, 'end': 147, 'label': 'hallucinated'}, {'start': 312, 'end': 486

In [153]:
# ============================================================
# CELL 110 — EVALUATE CORRECTED FINAL-TEST PREDICTIONS
# ============================================================

print("=" * 70)
print("EVALUATING CORRECTED FINAL-TEST PREDICTIONS")
print("=" * 70)


def spans_overlap(span1, span2):
    """
    True if two character spans overlap.
    """
    return (
        span1["start"] < span2["end"]
        and span2["start"] < span1["end"]
    )


# ------------------------------------------------------------
# 1. Collect human and predicted spans
# ------------------------------------------------------------

human_span_count = 0
predicted_span_count = 0

true_positive_spans = 0
false_positive_spans = 0
false_negative_spans = 0

example_results = []


for _, row in final_test_predictions_fixed.iterrows():

    human_spans = row["hallucination_spans"] or []
    predicted_spans = row["predicted_spans"] or []

    human_span_count += len(human_spans)
    predicted_span_count += len(predicted_spans)

    # --------------------------------------------------------
    # Span-level matching
    # --------------------------------------------------------

    matched_human = set()
    matched_predicted = set()

    for pred_idx, pred_span in enumerate(predicted_spans):

        for human_idx, human_span in enumerate(human_spans):

            if (
                human_idx not in matched_human
                and spans_overlap(pred_span, human_span)
            ):
                matched_predicted.add(pred_idx)
                matched_human.add(human_idx)
                break

    tp = len(matched_predicted)

    true_positive_spans += tp
    false_positive_spans += (
        len(predicted_spans) - tp
    )
    false_negative_spans += (
        len(human_spans) - tp
    )

    # --------------------------------------------------------
    # Example-level classification
    # --------------------------------------------------------

    human_positive = len(human_spans) > 0
    predicted_positive = len(predicted_spans) > 0

    example_results.append(
        {
            "test_id": row["test_id"],
            "human_positive": human_positive,
            "predicted_positive": predicted_positive
        }
    )


# ------------------------------------------------------------
# 2. Span-level metrics
# ------------------------------------------------------------

if predicted_span_count > 0:
    span_precision = (
        true_positive_spans /
        predicted_span_count
    )
else:
    span_precision = 0.0

if human_span_count > 0:
    span_recall = (
        true_positive_spans /
        human_span_count
    )
else:
    span_recall = 0.0

if span_precision + span_recall > 0:
    span_f1 = (
        2 * span_precision * span_recall /
        (span_precision + span_recall)
    )
else:
    span_f1 = 0.0


# ------------------------------------------------------------
# 3. Example-level metrics
# ------------------------------------------------------------

example_tp = 0
example_tn = 0
example_fp = 0
example_fn = 0

for result in example_results:

    h = result["human_positive"]
    p = result["predicted_positive"]

    if h and p:
        example_tp += 1

    elif not h and not p:
        example_tn += 1

    elif not h and p:
        example_fp += 1

    elif h and not p:
        example_fn += 1


example_total = len(example_results)

example_precision = (
    example_tp /
    (example_tp + example_fp)
    if (example_tp + example_fp) > 0
    else 0.0
)

example_recall = (
    example_tp /
    (example_tp + example_fn)
    if (example_tp + example_fn) > 0
    else 0.0
)

example_f1 = (
    2 * example_precision * example_recall /
    (example_precision + example_recall)
    if example_precision + example_recall > 0
    else 0.0
)

example_accuracy = (
    (example_tp + example_tn) /
    example_total
)


# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------

print("\n[1] SPAN-LEVEL RESULTS")
print("-" * 70)

print("Human spans:       ", human_span_count)
print("Predicted spans:   ", predicted_span_count)
print("TP spans:          ", true_positive_spans)
print("FP spans:          ", false_positive_spans)
print("FN spans:          ", false_negative_spans)

print(
    f"\nSpan Precision: {span_precision:.4f}"
)

print(
    f"Span Recall:    {span_recall:.4f}"
)

print(
    f"Span F1:        {span_f1:.4f}"
)


print("\n[2] EXAMPLE-LEVEL RESULTS")
print("-" * 70)

print("Human positive examples:    17")
print(
    "Predicted positive examples:",
    sum(
        r["predicted_positive"]
        for r in example_results
    )
)

print("TP:", example_tp)
print("TN:", example_tn)
print("FP:", example_fp)
print("FN:", example_fn)

print(
    f"\nExample Precision: {example_precision:.4f}"
)

print(
    f"Example Recall:    {example_recall:.4f}"
)

print(
    f"Example F1:        {example_f1:.4f}"
)

print(
    f"Example Accuracy:  {example_accuracy:.4f}"
)


print("\n" + "=" * 70)
print("✅ CORRECTED EVALUATION COMPLETE")
print("=" * 70)

EVALUATING CORRECTED FINAL-TEST PREDICTIONS

[1] SPAN-LEVEL RESULTS
----------------------------------------------------------------------
Human spans:        26
Predicted spans:    40
TP spans:           2
FP spans:           38
FN spans:           24

Span Precision: 0.0500
Span Recall:    0.0769
Span F1:        0.0606

[2] EXAMPLE-LEVEL RESULTS
----------------------------------------------------------------------
Human positive examples:    17
Predicted positive examples: 14
TP: 5
TN: 14
FP: 9
FN: 12

Example Precision: 0.3571
Example Recall:    0.2941
Example F1:        0.3226
Example Accuracy:  0.4750

✅ CORRECTED EVALUATION COMPLETE


Cell 111 — inspect prediction behavior

In [154]:
# ============================================================
# CELL 111 — DIAGNOSE MODEL PREDICTION BEHAVIOR
# ============================================================

print("=" * 70)
print("DIAGNOSING MODEL PREDICTION BEHAVIOR")
print("=" * 70)

for _, row in final_test_predictions_fixed.iterrows():

    predicted_spans = row["predicted_spans"] or []
    human_spans = row["hallucination_spans"] or []

    # Only show examples where the model predicted something
    if not predicted_spans:
        continue

    print("\n" + "=" * 70)
    print("TEST ID:", row["test_id"])
    print("HUMAN LABEL:", row["human_label"])

    print("\nANSWER:")
    print(row["answer"])

    print("\nHUMAN HALLUCINATION TEXT:")
    if human_spans:
        for span in human_spans:
            print(
                "  →",
                repr(
                    row["answer"][
                        span["start"]:span["end"]
                    ]
                )
            )
    else:
        print("  → NONE")

    print("\nMODEL PREDICTED HALLUCINATION TEXT:")
    for span in predicted_spans:
        print(
            "  →",
            repr(
                row["answer"][
                    span["start"]:span["end"]
                ]
            )
        )

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

DIAGNOSING MODEL PREDICTION BEHAVIOR

TEST ID: 30
HUMAN LABEL: appropriate_abstention

ANSWER:
Based on the evidence provided, there is no direct indication from the studies mentioned for necrotizing fasciitis specifically requiring hyperbaric oxygenation therapy. The evidence suggests that hyperbaric oxygen therapy may improve outcomes in certain conditions such as ischemic colonic anastomoses in rats, but it does not address necrotizing fasciitis directly. Therefore, the available evidence is insufficient to answer whether necrotizing fasciitis requires hyperbaric oxygenation therapy.

HUMAN HALLUCINATION TEXT:
  → NONE

MODEL PREDICTED HALLUCINATION TEXT:
  → ' studies'
  → 'rotizing'

TEST ID: 617
HUMAN LABEL: appropriate_abstention

ANSWER:
The available evidence is insufficient to answer this question. 

Explanation: The provided evidence does not contain any direct information about changes in the use of complementary and alternative medicine therapies among U.S. adults with chr

Cell 112 — final model sanity check

In [155]:
# ============================================================
# CELL 112 — FINAL MODEL PREDICTION SANITY CHECK
# ============================================================

print("=" * 70)
print("FINAL MODEL PREDICTION SANITY CHECK")
print("=" * 70)

total_tokens = 0
positive_tokens = 0
negative_tokens = 0

example_positive_counts = []

for idx, row in final_test_ground_truth.iterrows():

    predictions = model_predictions[idx]

    # Only answer tokens
    encoding = tokenizer(
        str(row["context"]),
        str(row["answer"]),
        truncation=True,
        max_length=4096,
        return_offsets_mapping=True,
        add_special_tokens=True
    )

    sequence_ids = encoding.sequence_ids()

    answer_token_count = 0
    answer_positive_count = 0

    for token_idx, prediction in enumerate(predictions):

        if token_idx >= len(sequence_ids):
            continue

        if sequence_ids[token_idx] != 1:
            continue

        answer_token_count += 1

        if int(prediction) == 1:
            answer_positive_count += 1
            positive_tokens += 1
        else:
            negative_tokens += 1

    total_tokens += answer_token_count

    example_positive_counts.append(
        {
            "test_id": row["test_id"],
            "answer_tokens": answer_token_count,
            "positive_tokens": answer_positive_count,
            "positive_percentage": (
                100 * answer_positive_count / answer_token_count
                if answer_token_count > 0
                else 0
            )
        }
    )


# ------------------------------------------------------------
# Overall token distribution
# ------------------------------------------------------------

print("\n[1] TOKEN-LEVEL DISTRIBUTION")
print("-" * 70)

print("Total answer tokens:", total_tokens)
print("Predicted label 0:", negative_tokens)
print("Predicted label 1:", positive_tokens)

print(
    f"\nLabel 0 percentage: "
    f"{100 * negative_tokens / total_tokens:.2f}%"
)

print(
    f"Label 1 percentage: "
    f"{100 * positive_tokens / total_tokens:.2f}%"
)


# ------------------------------------------------------------
# Examples with highest positive-token percentage
# ------------------------------------------------------------

example_positive_counts = sorted(
    example_positive_counts,
    key=lambda x: x["positive_percentage"],
    reverse=True
)

print("\n[2] MOST AGGRESSIVE PREDICTIONS")
print("-" * 70)

for item in example_positive_counts[:10]:

    print(
        f"Test {item['test_id']}: "
        f"{item['positive_tokens']}/"
        f"{item['answer_tokens']} tokens predicted positive "
        f"({item['positive_percentage']:.2f}%)"
    )


# ------------------------------------------------------------
# Examples with zero positive tokens
# ------------------------------------------------------------

zero_positive = [
    x for x in example_positive_counts
    if x["positive_tokens"] == 0
]

print("\n[3] ZERO-POSITIVE PREDICTIONS")
print("-" * 70)

print(
    "Examples with no predicted hallucination tokens:",
    len(zero_positive)
)

print(
    "Human-positive examples among them:",
    sum(
        final_test_ground_truth[
            final_test_ground_truth["test_id"] == x["test_id"]
        ]["hallucination_spans"].iloc[0] != []
        for x in zero_positive
    )
)

print("\n" + "=" * 70)
print("✅ FINAL SANITY CHECK COMPLETE")
print("=" * 70)

FINAL MODEL PREDICTION SANITY CHECK

[1] TOKEN-LEVEL DISTRIBUTION
----------------------------------------------------------------------
Total answer tokens: 2904
Predicted label 0: 2658
Predicted label 1: 246

Label 0 percentage: 91.53%
Label 1 percentage: 8.47%

[2] MOST AGGRESSIVE PREDICTIONS
----------------------------------------------------------------------
Test 635: 35/62 tokens predicted positive (56.45%)
Test 559: 39/100 tokens predicted positive (39.00%)
Test 265: 16/49 tokens predicted positive (32.65%)
Test 652: 56/189 tokens predicted positive (29.63%)
Test 692: 12/55 tokens predicted positive (21.82%)
Test 296: 42/199 tokens predicted positive (21.11%)
Test 60: 11/99 tokens predicted positive (11.11%)
Test 977: 10/132 tokens predicted positive (7.58%)
Test 432: 7/123 tokens predicted positive (5.69%)
Test 617: 6/125 tokens predicted positive (4.80%)

[3] ZERO-POSITIVE PREDICTIONS
----------------------------------------------------------------------
Examples with no pre

In [156]:
# ============================================================
# CELL 113 — CORRECT TOKEN ALIGNMENT FOR TRAIN/VALIDATION
# ============================================================

class CorrectedRagTruthDataset(Dataset):

    def __init__(
        self,
        samples,
        tokenizer,
        max_length=4096
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        encoding = self.tokenizer(
            sample.prompt,
            sample.answer,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors="pt",
            add_special_tokens=True
        )

        offsets = encoding.pop("offset_mapping")[0]
        sequence_ids = encoding.sequence_ids()

        labels = [-100] * encoding["input_ids"].shape[1]

        for i in range(len(sequence_ids)):

            # Only answer tokens
            if sequence_ids[i] != 1:
                continue

            token_start, token_end = offsets[i].tolist()

            token_label = 0

            for ann in sample.labels:

                if (
                    token_end > ann["start"]
                    and token_start < ann["end"]
                ):
                    token_label = 1
                    break

            labels[i] = token_label

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


corrected_train_dataset = CorrectedRagTruthDataset(
    train_records,
    tokenizer
)

corrected_val_dataset = CorrectedRagTruthDataset(
    val_records,
    tokenizer
)

print("Corrected training dataset:", len(corrected_train_dataset))
print("Corrected validation dataset:", len(corrected_val_dataset))

sample = corrected_train_dataset[0]

print("\nSample:")
print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("labels:", sample["labels"].shape)

print("\nLabel counts in first sample:")
print(
    "Positive:",
    (sample["labels"] == 1).sum().item()
)
print(
    "Negative:",
    (sample["labels"] == 0).sum().item()
)

print("\n✅ Corrected datasets created.")

NameError: name 'Dataset' is not defined

In [159]:
# ============================================================
# CELL 114 — FIND EXISTING VALIDATION DATA
# ============================================================

print("=" * 70)
print("SEARCHING FOR EXISTING VALIDATION DATA")
print("=" * 70)

# Show dataframe/list variables that may contain the original data
candidates = []

for name, obj in globals().items():
    if name.startswith("_"):
        continue

    try:
        length = len(obj)
    except:
        continue

    if length in [13, 64, 51, 200, 40]:
        candidates.append(
            (name, type(obj).__name__, length)
        )

print("\nRelevant objects:")
for item in candidates:
    print(
        f"  {item[0]:30s} "
        f"type={item[1]:25s} "
        f"length={item[2]}"
    )

print("\n" + "=" * 70)
print("VALIDATION SEARCH COMPLETE")
print("=" * 70)

SEARCHING FOR EXISTING VALIDATION DATA

Relevant objects:
  rag_results_df                 type=DataFrame                 length=200
  lettuce_df                     type=DataFrame                 length=200
  baseline_spans                 type=list                      length=200
  baseline_tokens                type=list                      length=200
  baseline_df                    type=DataFrame                 length=200
  baseline_output_file           type=str                       length=51
  annotation_df                  type=DataFrame                 length=200
  annotation_test                type=DataFrame                 length=40
  test_queue                     type=DataFrame                 length=40
  proposed_annotations           type=list                      length=200
  proposed_df                    type=DataFrame                 length=200
  baseline_screening             type=DataFrame                 length=200
  review_df                      type=DataFra

In [161]:
# ============================================================
# CELL 115 — CORRECTED DATASET FOR DICTIONARY RECORDS
# ============================================================

import torch
from torch.utils.data import Dataset

class CorrectedRagTruthDataset(Dataset):

    def __init__(self, samples, tokenizer, max_length=4096):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        # Dictionary access
        prompt = sample["prompt"]
        answer = sample["answer"]
        annotations = sample["labels"]

        encoding = self.tokenizer(
            prompt,
            answer,
            truncation=True,
            max_length=self.max_length,
            return_offsets_mapping=True,
            return_tensors="pt",
            add_special_tokens=True
        )

        offsets = encoding.pop("offset_mapping")[0]
        sequence_ids = encoding.sequence_ids()

        labels = [-100] * encoding["input_ids"].shape[1]

        # Label only ANSWER tokens
        for i in range(len(sequence_ids)):

            if sequence_ids[i] != 1:
                continue

            token_start, token_end = offsets[i].tolist()

            token_label = 0

            # Check overlap with hallucination annotations
            for ann in annotations:

                if (
                    token_end > ann["start"]
                    and token_start < ann["end"]
                ):
                    token_label = 1
                    break

            labels[i] = token_label

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.long)
        }


# Create corrected datasets
corrected_train_dataset = CorrectedRagTruthDataset(
    train_records,
    tokenizer
)

corrected_val_dataset = CorrectedRagTruthDataset(
    validation_records,
    tokenizer
)

print("=" * 70)
print("CORRECTED DATASETS READY")
print("=" * 70)

print("Training examples:", len(corrected_train_dataset))
print("Validation examples:", len(corrected_val_dataset))


# Sanity check
train_sample = corrected_train_dataset[0]

print("\nTraining sample shapes:")
print("input_ids:", train_sample["input_ids"].shape)
print("attention_mask:", train_sample["attention_mask"].shape)
print("labels:", train_sample["labels"].shape)

print("\nTraining sample label counts:")
print(
    "Positive:",
    (train_sample["labels"] == 1).sum().item()
)

print(
    "Negative:",
    (train_sample["labels"] == 0).sum().item()
)

print(
    "Ignored:",
    (train_sample["labels"] == -100).sum().item()
)

print("\n" + "=" * 70)
print("✅ ALIGNMENT CHECK PASSED")
print("=" * 70)

CORRECTED DATASETS READY
Training examples: 51
Validation examples: 13

Training sample shapes:
input_ids: torch.Size([681])
attention_mask: torch.Size([681])
labels: torch.Size([681])

Training sample label counts:
Positive: 35
Negative: 85
Ignored: 561

✅ ALIGNMENT CHECK PASSED


In [162]:
# ============================================================
# CELL 116 — CHECK MODEL SOURCE
# ============================================================

print("Current model:")
print(medical_model.config._name_or_path)

print("\nAvailable model variables:")

for name in ["LETTUCE_MODEL", "ADAPTED_MODEL", "ANNOTATION_MODEL"]:
    if name in globals():
        print(name, "=", globals()[name])

Current model:
KRLabsOrg/lettucedect-base-modernbert-en-v1

Available model variables:
LETTUCE_MODEL = KRLabsOrg/lettucedect-base-modernbert-en-v1
ADAPTED_MODEL = /kaggle/working/lettucedetect_medical_adapted
ANNOTATION_MODEL = Qwen/Qwen2.5-1.5B-Instruct


In [165]:
# ============================================================
# CELL 117 — RETRAIN LETTUCEDETECT WITH CORRECT LABEL ALIGNMENT
# ============================================================

import torch
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

print("=" * 70)
print("LOADING FRESH BASE MODEL")
print("=" * 70)

# IMPORTANT:
# Start from the original LettuceDetect model
# instead of the previously trained medical checkpoint.

fresh_model = AutoModelForTokenClassification.from_pretrained(
    LETTUCE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True
)

fresh_model = fresh_model.to("cuda")

print("Model loaded:")
print(fresh_model.config._name_or_path)
print("Device:", next(fresh_model.parameters()).device)


# ============================================================
# TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(
    output_dir="/kaggle/working/lettucedetect_corrected",

    # Small dataset → a few epochs
    num_train_epochs=5,

    # Learning rate
    learning_rate=2e-5,

    # Batch size
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # Gradient accumulation
    gradient_accumulation_steps=2,

    # Regularization
    weight_decay=0.01,

    # Evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",

    # Keep best validation model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=10,

    # Faster training on GPU
    fp16=torch.cuda.is_available(),

    # Disable unnecessary reporting
    report_to="none",

    # Keep only best few checkpoints
    save_total_limit=2
)


# ============================================================
# TRAINER
# ============================================================

trainer = Trainer(
    model=fresh_model,
    args=training_args,

    train_dataset=corrected_train_dataset,
    eval_dataset=corrected_val_dataset,

    tokenizer=tokenizer
)


print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE")
print("=" * 70)

print("\nBest checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest validation loss:")
print(trainer.state.best_metric)

LOADING FRESH BASE MODEL


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model loaded:
KRLabsOrg/lettucedect-base-modernbert-en-v1
Device: cuda:0


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [166]:
# ============================================================
# CELL 117A — CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=fresh_model,
    args=training_args,

    train_dataset=corrected_train_dataset,
    eval_dataset=corrected_val_dataset
)

print("=" * 70)
print("✅ TRAINER CREATED SUCCESSFULLY")
print("=" * 70)

print("Training examples:", len(corrected_train_dataset))
print("Validation examples:", len(corrected_val_dataset))

✅ TRAINER CREATED SUCCESSFULLY
Training examples: 51
Validation examples: 13


In [168]:
# ============================================================
# CELL 117B — DYNAMIC PADDING COLLATOR
# ============================================================

from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("=" * 70)
print("✅ DYNAMIC PADDING COLLATOR READY")
print("=" * 70)

✅ DYNAMIC PADDING COLLATOR READY


In [169]:
# ============================================================
# CELL 117C — RECREATE TRAINER WITH PADDING
# ============================================================

trainer = Trainer(
    model=fresh_model,
    args=training_args,

    train_dataset=corrected_train_dataset,
    eval_dataset=corrected_val_dataset,

    data_collator=data_collator
)

print("✅ Trainer recreated successfully")

✅ Trainer recreated successfully


In [174]:
# ============================================================
# CELL 119 — RELOAD FRESH MODEL WITHOUT MANUAL GPU PLACEMENT
# ============================================================

import torch
from transformers import AutoModelForTokenClassification

print("=" * 70)
print("RELOADING FRESH MODEL")
print("=" * 70)

# Delete the problematic model/trainer
if "trainer" in globals():
    del trainer

if "fresh_model" in globals():
    del fresh_model

torch.cuda.empty_cache()

# Load model on CPU first
fresh_model = AutoModelForTokenClassification.from_pretrained(
    LETTUCE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True
)

print("Model:", fresh_model.config._name_or_path)
print(
    "Parameters:",
    sum(p.numel() for p in fresh_model.parameters())
)
print(
    "Model device:",
    next(fresh_model.parameters()).device
)

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n" + "=" * 70)
print("✅ FRESH MODEL READY")
print("=" * 70)

RELOADING FRESH MODEL


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model: KRLabsOrg/lettucedect-base-modernbert-en-v1
Parameters: 149606402
Model device: cpu

CUDA available: True
GPU: Tesla T4

✅ FRESH MODEL READY


In [175]:
# ============================================================
# CELL 120 — RECREATE TRAINER
# ============================================================

trainer = Trainer(
    model=fresh_model,
    args=training_args,

    train_dataset=corrected_train_dataset,
    eval_dataset=corrected_val_dataset,

    data_collator=data_collator
)

print("=" * 70)
print("✅ TRAINER READY")
print("=" * 70)

✅ TRAINER READY


In [176]:
# ============================================================
# CELL 121 — TRAIN
# ============================================================

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

train_result = trainer.train()

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE")
print("=" * 70)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

STARTING TRAINING


StopIteration: Caught StopIteration in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/modernbert/modeling_modernbert.py", line 1380, in forward
    self._maybe_set_compile()
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/modernbert/modeling_modernbert.py", line 725, in _maybe_set_compile
    if self.device.type == "mps":
   

In [178]:
# ============================================================
# FINAL FIX — MODERNBERT + KAGGLE DATAPARALLEL ISSUE
# ============================================================

import gc
import torch

from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

print("=" * 70)
print("FIXING MODERNBERT DATAPARALLEL ISSUE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Clean previous Trainer/model
# ------------------------------------------------------------

if "trainer" in globals():
    del trainer

if "fresh_model" in globals():
    del fresh_model

if "training_args" in globals():
    del training_args

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 2. IMPORTANT:
# Make TrainingArguments think only ONE GPU is available.
#
# CUDA_VISIBLE_DEVICES cannot reliably be changed now because
# CUDA was already initialized earlier in this Kaggle kernel.
# ------------------------------------------------------------

_original_device_count = torch.cuda.device_count

torch.cuda.device_count = lambda: 1

print("\nCUDA available:", torch.cuda.is_available())
print("GPUs visible to Trainer:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Physical GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 3. Load fresh base model
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FRESH BASE MODEL")
print("=" * 70)

fresh_model = AutoModelForTokenClassification.from_pretrained(
    LETTUCE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True
)

print("Model:", fresh_model.config._name_or_path)

print(
    "Parameters:",
    sum(p.numel() for p in fresh_model.parameters())
)

# Put directly on GPU
fresh_model = fresh_model.to("cuda")

print(
    "Model device:",
    next(fresh_model.parameters()).device
)


# ------------------------------------------------------------
# 4. Dynamic padding
# ------------------------------------------------------------

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)


# ------------------------------------------------------------
# 5. Create NEW TrainingArguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="/kaggle/working/lettucedetect_corrected",

    num_train_epochs=5,

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=2,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,

    fp16=torch.cuda.is_available(),

    report_to="none",

    save_total_limit=2
)

print("\nTrainer GPU count:", training_args.n_gpu)


# ------------------------------------------------------------
# 6. Create Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=fresh_model,
    args=training_args,

    train_dataset=corrected_train_dataset,
    eval_dataset=corrected_val_dataset,

    data_collator=data_collator
)

print("\n" + "=" * 70)
print("TRAINER CREATED")
print("=" * 70)

print("Training examples:", len(corrected_train_dataset))
print("Validation examples:", len(corrected_val_dataset))

print("Trainer GPU count:", training_args.n_gpu)


# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING CORRECTED TRAINING")
print("=" * 70)

train_result = trainer.train()


# ------------------------------------------------------------
# 8. Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ TRAINING COMPLETE")
print("=" * 70)

print("\nBest checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nBest validation loss:")
print(trainer.state.best_metric)

print("\nTraining metrics:")
print(train_result.metrics)

FIXING MODERNBERT DATAPARALLEL ISSUE

CUDA available: True
GPUs visible to Trainer: 1
Physical GPU: Tesla T4

LOADING FRESH BASE MODEL


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Model: KRLabsOrg/lettucedect-base-modernbert-en-v1
Parameters: 149606402
Model device: cuda:0

Trainer GPU count: 1

TRAINER CREATED
Training examples: 51
Validation examples: 13
Trainer GPU count: 1

STARTING CORRECTED TRAINING


Epoch,Training Loss,Validation Loss
1,4.593169,0.856764
2,1.698700,0.687457
3,0.733028,0.450205
4,0.417231,0.543657
5,0.222534,0.564799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ TRAINING COMPLETE

Best checkpoint:
/kaggle/working/lettucedetect_corrected/checkpoint-39

Best validation loss:
0.45020538568496704

Training metrics:
{'train_runtime': 55.4674, 'train_samples_per_second': 4.597, 'train_steps_per_second': 1.172, 'total_flos': 140680412533836.0, 'train_loss': 1.2766553851274343, 'epoch': 5.0}


In [179]:
# ============================================================
# CELL 125 — VALIDATION PREDICTIONS + THRESHOLD TUNING
# ============================================================

import numpy as np
import torch

print("=" * 70)
print("EVALUATING CORRECTED MODEL ON VALIDATION SET")
print("=" * 70)

# Use the best checkpoint/model already loaded by Trainer
fresh_model = trainer.model
fresh_model.eval()

device = next(fresh_model.parameters()).device
print("Model device:", device)

# ------------------------------------------------------------
# Helper: convert token probabilities into character spans
# ------------------------------------------------------------

def get_spans_from_logits(sample, logits, threshold=0.5):

    prompt = sample["prompt"]
    answer = sample["answer"]
    annotations = sample["labels"]

    encoding = tokenizer(
        prompt,
        answer,
        truncation=True,
        max_length=4096,
        return_offsets_mapping=True,
        return_tensors="pt",
        add_special_tokens=True
    )

    offsets = encoding["offset_mapping"][0]
    sequence_ids = encoding.sequence_ids()

    probs = torch.softmax(
        torch.tensor(logits),
        dim=-1
    )[:, 1].numpy()

    positive_ranges = []

    for i, seq_id in enumerate(sequence_ids):

        # Answer tokens only
        if seq_id != 1:
            continue

        start, end = offsets[i].tolist()

        if end <= start:
            continue

        if probs[i] >= threshold:
            positive_ranges.append((start, end))

    # Merge adjacent/overlapping token ranges
    merged = []

    for start, end in positive_ranges:

        if not merged:
            merged.append([start, end])

        elif start <= merged[-1][1] + 1:
            merged[-1][1] = max(
                merged[-1][1],
                end
            )

        else:
            merged.append([start, end])

    return [
        {
            "start": int(start),
            "end": int(end),
            "label": "hallucinated"
        }
        for start, end in merged
    ]


# ------------------------------------------------------------
# Span overlap
# ------------------------------------------------------------

def spans_overlap(pred, true):

    return (
        pred["start"] < true["end"]
        and pred["end"] > true["start"]
    )


# ------------------------------------------------------------
# Get validation logits
# ------------------------------------------------------------

print("\nGenerating validation predictions...")

val_output = trainer.predict(corrected_val_dataset)

val_logits = val_output.predictions

print("Logits shape:", val_logits.shape)


# ------------------------------------------------------------
# Ground truth validation records
# ------------------------------------------------------------

val_samples = validation_records


# ------------------------------------------------------------
# Evaluate multiple thresholds
# ------------------------------------------------------------

thresholds = np.arange(
    0.10,
    0.91,
    0.05
)

results = []

for threshold in thresholds:

    TP = 0
    FP = 0
    FN = 0

    for idx, sample in enumerate(val_samples):

        pred_spans = get_spans_from_logits(
            sample,
            val_logits[idx],
            threshold
        )

        true_spans = sample["labels"]

        matched_true = set()

        for pred in pred_spans:

            matched = False

            for j, true in enumerate(true_spans):

                if j in matched_true:
                    continue

                if spans_overlap(pred, true):
                    TP += 1
                    matched_true.add(j)
                    matched = True
                    break

            if not matched:
                FP += 1

        FN += len(true_spans) - len(matched_true)

    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall)
        else 0
    )

    results.append({
        "threshold": round(float(threshold), 2),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "F1": f1
    })


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION THRESHOLD RESULTS")
print("=" * 70)

for r in results:

    print(
        f"Threshold={r['threshold']:.2f} | "
        f"TP={r['TP']:2d} | "
        f"FP={r['FP']:2d} | "
        f"FN={r['FN']:2d} | "
        f"P={r['precision']:.3f} | "
        f"R={r['recall']:.3f} | "
        f"F1={r['F1']:.3f}"
    )


# ------------------------------------------------------------
# Select best validation threshold
# ------------------------------------------------------------

best_result = max(
    results,
    key=lambda x: x["F1"]
)

BEST_THRESHOLD = best_result["threshold"]

print("\n" + "=" * 70)
print("✅ BEST VALIDATION THRESHOLD")
print("=" * 70)

print("Threshold:", BEST_THRESHOLD)
print("Precision:", round(best_result["precision"], 4))
print("Recall:", round(best_result["recall"], 4))
print("F1:", round(best_result["F1"], 4))

print("\n⚠️ This threshold is selected ONLY from validation.")
print("The 40-example final test remains untouched.")

EVALUATING CORRECTED MODEL ON VALIDATION SET
Model device: cuda:0

Generating validation predictions...


Logits shape: (13, 844, 2)

VALIDATION THRESHOLD RESULTS
Threshold=0.10 | TP=10 | FP=44 | FN= 6 | P=0.185 | R=0.625 | F1=0.286
Threshold=0.15 | TP=12 | FP=37 | FN= 4 | P=0.245 | R=0.750 | F1=0.369
Threshold=0.20 | TP=12 | FP=38 | FN= 4 | P=0.240 | R=0.750 | F1=0.364
Threshold=0.25 | TP=11 | FP=35 | FN= 5 | P=0.239 | R=0.688 | F1=0.355
Threshold=0.30 | TP=13 | FP=31 | FN= 3 | P=0.295 | R=0.812 | F1=0.433
Threshold=0.35 | TP=11 | FP=29 | FN= 5 | P=0.275 | R=0.688 | F1=0.393
Threshold=0.40 | TP=11 | FP=34 | FN= 5 | P=0.244 | R=0.688 | F1=0.361
Threshold=0.45 | TP=12 | FP=29 | FN= 4 | P=0.293 | R=0.750 | F1=0.421
Threshold=0.50 | TP=12 | FP=27 | FN= 4 | P=0.308 | R=0.750 | F1=0.436
Threshold=0.55 | TP= 9 | FP=25 | FN= 7 | P=0.265 | R=0.562 | F1=0.360
Threshold=0.60 | TP= 8 | FP=20 | FN= 8 | P=0.286 | R=0.500 | F1=0.364
Threshold=0.65 | TP= 8 | FP=19 | FN= 8 | P=0.296 | R=0.500 | F1=0.372
Threshold=0.70 | TP= 8 | FP=15 | FN= 8 | P=0.348 | R=0.500 | F1=0.410
Threshold=0.75 | TP= 6 | FP=12 | 

In [182]:
# ============================================
# CELL 127 — CORRECT FINAL TEST EVALUATION
# ============================================

# Use the LOCKED human annotations
# from final_test_ground_truth.
#
# We are NOT changing or training on these labels.
# They are used ONLY for final evaluation.

# --------------------------------------------
# Create mapping: test_id -> human spans
# --------------------------------------------

ground_truth_map = {}

for _, row in final_test_ground_truth.iterrows():

    spans = row["hallucination_spans"]

    # Make sure empty annotations become []
    if spans is None:
        spans = []

    ground_truth_map[row["test_id"]] = spans

print("Ground-truth examples:", len(ground_truth_map))

# Check expected human counts
total_true_spans = sum(
    len(spans) for spans in ground_truth_map.values()
)

positive_examples = sum(
    len(spans) > 0 for spans in ground_truth_map.values()
)

print("Human positive examples:", positive_examples)
print("Human total spans:", total_true_spans)


# --------------------------------------------
# Evaluate predictions
# --------------------------------------------

TP = 0
FP = 0
FN = 0

final_test_predictions_corrected = []

for idx, sample in enumerate(test_records):

    test_id = sample["test_id"]

    # MODEL PREDICTION
    pred_spans = get_spans_from_logits(
        sample,
        final_logits[idx],
        threshold=BEST_THRESHOLD
    )

    # HUMAN VERIFIED GROUND TRUTH
    true_spans = ground_truth_map[test_id]

    # Match predicted and true spans by overlap
    matched_true = set()
    matched_pred = set()

    for p_idx, pred in enumerate(pred_spans):

        for t_idx, true in enumerate(true_spans):

            if t_idx in matched_true:
                continue

            if spans_overlap(pred, true):

                TP += 1
                matched_true.add(t_idx)
                matched_pred.add(p_idx)
                break

    FP += len(pred_spans) - len(matched_pred)
    FN += len(true_spans) - len(matched_true)

    final_test_predictions_corrected.append({
        "test_id": test_id,
        "true_spans": true_spans,
        "predicted_spans": pred_spans
    })


# --------------------------------------------
# Span metrics
# --------------------------------------------

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("\n====================================")
print("FINAL TEST — SPAN LEVEL")
print("====================================")

print("Human spans     :", TP + FN)
print("Predicted spans :", TP + FP)
print("TP              :", TP)
print("FP              :", FP)
print("FN              :", FN)
print("Precision       :", round(precision, 4))
print("Recall          :", round(recall, 4))
print("F1              :", round(f1, 4))


# --------------------------------------------
# Example-level metrics
# --------------------------------------------

example_TP = 0
example_FP = 0
example_FN = 0
example_TN = 0

for item in final_test_predictions_corrected:

    actual_positive = len(item["true_spans"]) > 0
    predicted_positive = len(item["predicted_spans"]) > 0

    if actual_positive and predicted_positive:
        example_TP += 1

    elif not actual_positive and predicted_positive:
        example_FP += 1

    elif actual_positive and not predicted_positive:
        example_FN += 1

    else:
        example_TN += 1


example_precision = (
    example_TP / (example_TP + example_FP)
    if (example_TP + example_FP) > 0 else 0
)

example_recall = (
    example_TP / (example_TP + example_FN)
    if (example_TP + example_FN) > 0 else 0
)

example_f1 = (
    2 * example_precision * example_recall /
    (example_precision + example_recall)
    if (example_precision + example_recall) > 0 else 0
)

example_accuracy = (
    (example_TP + example_TN) / len(test_records)
)


print("\n====================================")
print("FINAL TEST — EXAMPLE LEVEL")
print("====================================")

print("Human positive examples :", example_TP + example_FN)
print("Predicted positive      :", example_TP + example_FP)
print("TP                      :", example_TP)
print("FP                      :", example_FP)
print("FN                      :", example_FN)
print("TN                      :", example_TN)
print("Precision               :", round(example_precision, 4))
print("Recall                  :", round(example_recall, 4))
print("F1                      :", round(example_f1, 4))
print("Accuracy                :", round(example_accuracy, 4))

Ground-truth examples: 40
Human positive examples: 17
Human total spans: 26

FINAL TEST — SPAN LEVEL
Human spans     : 26
Predicted spans : 75
TP              : 17
FP              : 58
FN              : 9
Precision       : 0.2267
Recall          : 0.6538
F1              : 0.3366

FINAL TEST — EXAMPLE LEVEL
Human positive examples : 17
Predicted positive      : 20
TP                      : 15
FP                      : 5
FN                      : 2
TN                      : 18
Precision               : 0.75
Recall                  : 0.8824
F1                      : 0.8108
Accuracy                : 0.825


In [183]:
# ============================================
# CELL 128 — FINE THRESHOLD TUNING
# ============================================

import numpy as np

# Fine-grained thresholds
thresholds = np.arange(0.45, 0.86, 0.01)

best_threshold = None
best_f1 = -1
results = []

for threshold in thresholds:

    TP = 0
    FP = 0
    FN = 0

    for idx, sample in enumerate(validation_records):

        pred_spans = get_spans_from_logits(
            sample,
            val_logits[idx],
            threshold=float(threshold)
        )

        true_spans = sample["labels"]

        matched_true = set()
        matched_pred = set()

        for p_idx, pred in enumerate(pred_spans):

            for t_idx, true in enumerate(true_spans):

                if t_idx in matched_true:
                    continue

                if spans_overlap(pred, true):

                    TP += 1
                    matched_true.add(t_idx)
                    matched_pred.add(p_idx)
                    break

        FP += len(pred_spans) - len(matched_pred)
        FN += len(true_spans) - len(matched_true)

    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0
    )

    results.append({
        "threshold": round(float(threshold), 2),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(threshold)


# --------------------------------------------
# Show best thresholds
# --------------------------------------------

results_sorted = sorted(
    results,
    key=lambda x: x["f1"],
    reverse=True
)

print("====================================")
print("TOP VALIDATION THRESHOLDS")
print("====================================")

for r in results_sorted[:10]:

    print(
        f"Threshold={r['threshold']:.2f} | "
        f"TP={r['TP']} | "
        f"FP={r['FP']} | "
        f"FN={r['FN']} | "
        f"P={r['precision']:.4f} | "
        f"R={r['recall']:.4f} | "
        f"F1={r['f1']:.4f}"
    )

print("\n====================================")
print("SELECTED THRESHOLD")
print("====================================")
print("BEST_THRESHOLD =", round(best_threshold, 2))
print("Validation F1   =", round(best_f1, 4))

TOP VALIDATION THRESHOLDS
Threshold=0.50 | TP=12 | FP=27 | FN=4 | P=0.3077 | R=0.7500 | F1=0.4364
Threshold=0.46 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.47 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.48 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.49 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.52 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.53 | TP=12 | FP=28 | FN=4 | P=0.3000 | R=0.7500 | F1=0.4286
Threshold=0.45 | TP=12 | FP=29 | FN=4 | P=0.2927 | R=0.7500 | F1=0.4211
Threshold=0.51 | TP=12 | FP=29 | FN=4 | P=0.2927 | R=0.7500 | F1=0.4211
Threshold=0.79 | TP=6 | FP=7 | FN=10 | P=0.4615 | R=0.3750 | F1=0.4138

SELECTED THRESHOLD
BEST_THRESHOLD = 0.5
Validation F1   = 0.4364


In [184]:
# ============================================
# CELL 129 — FINAL TEST WITH FROZEN THRESHOLD
# ============================================

BEST_THRESHOLD = 0.50

print("Using frozen threshold:", BEST_THRESHOLD)

# --------------------------------------------
# Recalculate final predictions
# --------------------------------------------

final_test_predictions_corrected = []

for idx, sample in enumerate(test_records):

    test_id = sample["test_id"]

    pred_spans = get_spans_from_logits(
        sample,
        final_logits[idx],
        threshold=BEST_THRESHOLD
    )

    # LOCKED human annotations
    true_spans = ground_truth_map[test_id]

    final_test_predictions_corrected.append({
        "test_id": test_id,
        "true_spans": true_spans,
        "predicted_spans": pred_spans
    })


# --------------------------------------------
# Span-level evaluation
# --------------------------------------------

TP = 0
FP = 0
FN = 0

for item in final_test_predictions_corrected:

    true_spans = item["true_spans"]
    pred_spans = item["predicted_spans"]

    matched_true = set()
    matched_pred = set()

    for p_idx, pred in enumerate(pred_spans):

        for t_idx, true in enumerate(true_spans):

            if t_idx in matched_true:
                continue

            if spans_overlap(pred, true):

                TP += 1
                matched_true.add(t_idx)
                matched_pred.add(p_idx)
                break

    FP += len(pred_spans) - len(matched_pred)
    FN += len(true_spans) - len(matched_true)


precision = TP / (TP + FP) if (TP + FP) else 0
recall = TP / (TP + FN) if (TP + FN) else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) else 0
)


# --------------------------------------------
# Example-level evaluation
# --------------------------------------------

example_TP = 0
example_FP = 0
example_FN = 0
example_TN = 0

for item in final_test_predictions_corrected:

    actual_positive = len(item["true_spans"]) > 0
    predicted_positive = len(item["predicted_spans"]) > 0

    if actual_positive and predicted_positive:
        example_TP += 1

    elif not actual_positive and predicted_positive:
        example_FP += 1

    elif actual_positive and not predicted_positive:
        example_FN += 1

    else:
        example_TN += 1


example_precision = (
    example_TP / (example_TP + example_FP)
    if (example_TP + example_FP) else 0
)

example_recall = (
    example_TP / (example_TP + example_FN)
    if (example_TP + example_FN) else 0
)

example_f1 = (
    2 * example_precision * example_recall /
    (example_precision + example_recall)
    if (example_precision + example_recall) else 0
)

example_accuracy = (
    (example_TP + example_TN) / len(test_records)
)


# --------------------------------------------
# FINAL RESULTS
# --------------------------------------------

print("\n====================================")
print("FINAL TEST RESULTS")
print("====================================")

print("\nSPAN LEVEL")
print("------------------------------------")
print("Human spans     :", TP + FN)
print("Predicted spans :", TP + FP)
print("TP              :", TP)
print("FP              :", FP)
print("FN              :", FN)
print("Precision       :", round(precision, 4))
print("Recall          :", round(recall, 4))
print("F1              :", round(f1, 4))

print("\nEXAMPLE LEVEL")
print("------------------------------------")
print("Human positive examples :", example_TP + example_FN)
print("Predicted positive      :", example_TP + example_FP)
print("TP                      :", example_TP)
print("FP                      :", example_FP)
print("FN                      :", example_FN)
print("TN                      :", example_TN)
print("Precision               :", round(example_precision, 4))
print("Recall                  :", round(example_recall, 4))
print("F1                      :", round(example_f1, 4))
print("Accuracy                :", round(example_accuracy, 4))

Using frozen threshold: 0.5

FINAL TEST RESULTS

SPAN LEVEL
------------------------------------
Human spans     : 26
Predicted spans : 75
TP              : 17
FP              : 58
FN              : 9
Precision       : 0.2267
Recall          : 0.6538
F1              : 0.3366

EXAMPLE LEVEL
------------------------------------
Human positive examples : 17
Predicted positive      : 20
TP                      : 15
FP                      : 5
FN                      : 2
TN                      : 18
Precision               : 0.75
Recall                  : 0.8824
F1                      : 0.8108
Accuracy                : 0.825


Baseline vs Fine-tuned comparison

In [185]:
# STEP 5A — Final baseline vs fine-tuned comparison
# Uses the SAME 40 held-out test examples and SAME ground truth.

print("=" * 80)
print("STEP 5A — VERIFY FINAL TEST GROUND TRUTH")
print("=" * 80)

print("Final test examples:", len(final_test_ground_truth))

human_positive_examples = sum(
    len(row["hallucination_spans"]) > 0
    for _, row in final_test_ground_truth.iterrows()
)

human_total_spans = sum(
    len(row["hallucination_spans"])
    for _, row in final_test_ground_truth.iterrows()
)

print("Human-positive examples:", human_positive_examples)
print("Human hallucination spans:", human_total_spans)

assert len(final_test_ground_truth) == 40
assert human_positive_examples == 17
assert human_total_spans == 26

print("\nGround truth verification PASSED.")

STEP 5A — VERIFY FINAL TEST GROUND TRUTH
Final test examples: 40
Human-positive examples: 17
Human hallucination spans: 26

Ground truth verification PASSED.


In [186]:
# STEP 5A — Verify the final held-out test ground truth

print("=" * 80)
print("STEP 5A — FINAL TEST GROUND TRUTH VERIFICATION")
print("=" * 80)

print("Final test examples:", len(final_test_ground_truth))

human_positive_examples = sum(
    len(row["hallucination_spans"]) > 0
    for _, row in final_test_ground_truth.iterrows()
)

human_total_spans = sum(
    len(row["hallucination_spans"])
    for _, row in final_test_ground_truth.iterrows()
)

print("Human-positive examples:", human_positive_examples)
print("Human hallucination spans:", human_total_spans)

assert len(final_test_ground_truth) == 40
assert human_positive_examples == 17
assert human_total_spans == 26

print("\nGround truth verification PASSED.")

STEP 5A — FINAL TEST GROUND TRUTH VERIFICATION
Final test examples: 40
Human-positive examples: 17
Human hallucination spans: 26

Ground truth verification PASSED.


In [188]:
# ============================================================
# STEP 5 + STEP 6
# FINAL BASELINE vs MEDICAL-ADAPTED COMPARISON
# + DOMAIN GAP / ERROR ANALYSIS
# ============================================================

import os
import gc
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)

from sklearn.metrics import precision_score, recall_score, f1_score


# ============================================================
# STEP 5A — VERIFY FIXED FINAL TEST SET
# ============================================================

print("=" * 90)
print("STEP 5A — FINAL TEST GROUND TRUTH VERIFICATION")
print("=" * 90)

assert len(final_test_ground_truth) == 40

human_positive_examples = sum(
    len(row["hallucination_spans"]) > 0
    for _, row in final_test_ground_truth.iterrows()
)

human_total_spans = sum(
    len(row["hallucination_spans"])
    for _, row in final_test_ground_truth.iterrows()
)

assert human_positive_examples == 17
assert human_total_spans == 26

print("Final test examples:", len(final_test_ground_truth))
print("Human-positive examples:", human_positive_examples)
print("Human hallucination spans:", human_total_spans)
print("Ground truth verification PASSED.")


# ============================================================
# STEP 5B — HELPER FUNCTIONS
# ============================================================

def spans_overlap(span1, span2):
    """
    Returns True if two character spans overlap.
    """
    return (
        span1["start"] < span2["end"]
        and span2["start"] < span1["end"]
    )


def get_predicted_spans_from_encoding(
    answer,
    encoding,
    logits,
    threshold=0.50
):
    """
    Convert token-level hallucination probabilities
    into character-level answer spans.

    Only answer tokens (sequence_ids == 1) are considered.
    """

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    )[:, 1].numpy()

    sequence_ids = encoding.sequence_ids()

    positive_tokens = []

    for i, seq_id in enumerate(sequence_ids):

        # Only answer tokens
        if seq_id != 1:
            continue

        offset = encoding["offset_mapping"][i]

        start, end = offset

        if end <= start:
            continue

        if probabilities[i] >= threshold:
            positive_tokens.append(
                {
                    "token_index": i,
                    "start": int(start),
                    "end": int(end),
                    "probability": float(probabilities[i])
                }
            )

    if not positive_tokens:
        return []

    # --------------------------------------------------------
    # Merge consecutive / touching positive token spans
    # --------------------------------------------------------

    merged = []

    current_start = positive_tokens[0]["start"]
    current_end = positive_tokens[0]["end"]
    current_max_prob = positive_tokens[0]["probability"]

    for token in positive_tokens[1:]:

        # Merge if spans touch or overlap
        if token["start"] <= current_end + 1:

            current_end = max(
                current_end,
                token["end"]
            )

            current_max_prob = max(
                current_max_prob,
                token["probability"]
            )

        else:

            merged.append(
                {
                    "start": current_start,
                    "end": current_end,
                    "label": "hallucinated",
                    "confidence": current_max_prob,
                    "text": answer[current_start:current_end]
                }
            )

            current_start = token["start"]
            current_end = token["end"]
            current_max_prob = token["probability"]

    # Add final span
    merged.append(
        {
            "start": current_start,
            "end": current_end,
            "label": "hallucinated",
            "confidence": current_max_prob,
            "text": answer[current_start:current_end]
        }
    )

    return merged


def evaluate_span_predictions(
    ground_truth_df,
    predictions_dict
):
    """
    Evaluate predicted character spans against human spans.

    Span matching:
    Any overlap between predicted and human span = TP.

    This is the SAME overlap-based protocol used earlier.
    """

    total_human = 0
    total_predicted = 0

    tp = 0
    fp = 0
    fn = 0

    example_tp = 0
    example_fp = 0
    example_fn = 0
    example_tn = 0

    detailed_rows = []

    for _, row in ground_truth_df.iterrows():

        test_id = row["test_id"]

        human_spans = row["hallucination_spans"]

        if not isinstance(human_spans, list):
            human_spans = []

        predicted_spans = predictions_dict.get(
            test_id,
            []
        )

        total_human += len(human_spans)
        total_predicted += len(predicted_spans)

        # ----------------------------------------------------
        # Match predicted spans to human spans
        # ----------------------------------------------------

        matched_human = set()
        matched_predicted = set()

        for p_idx, pred in enumerate(predicted_spans):

            for h_idx, human in enumerate(human_spans):

                if h_idx in matched_human:
                    continue

                if spans_overlap(pred, human):

                    matched_predicted.add(p_idx)
                    matched_human.add(h_idx)

                    tp += 1

                    break

        # Remaining predicted spans = FP
        row_fp = len(predicted_spans) - len(matched_predicted)

        # Remaining human spans = FN
        row_fn = len(human_spans) - len(matched_human)

        fp += row_fp
        fn += row_fn

        human_positive = len(human_spans) > 0
        predicted_positive = len(predicted_spans) > 0

        if human_positive and predicted_positive:
            example_tp += 1

        elif (not human_positive) and predicted_positive:
            example_fp += 1

        elif human_positive and (not predicted_positive):
            example_fn += 1

        else:
            example_tn += 1

        detailed_rows.append(
            {
                "test_id": test_id,
                "question": row["question"],
                "answer": row["answer"],
                "human_spans": human_spans,
                "predicted_spans": predicted_spans,
                "human_positive": human_positive,
                "predicted_positive": predicted_positive,
                "row_tp": len(matched_predicted),
                "row_fp": row_fp,
                "row_fn": row_fn,
            }
        )

    # --------------------------------------------------------
    # Span metrics
    # --------------------------------------------------------

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Example-level metrics
    # --------------------------------------------------------

    example_precision = (
        example_tp / (example_tp + example_fp)
        if (example_tp + example_fp) > 0
        else 0.0
    )

    example_recall = (
        example_tp / (example_tp + example_fn)
        if (example_tp + example_fn) > 0
        else 0.0
    )

    example_f1 = (
        2 * example_precision * example_recall
        / (example_precision + example_recall)
        if (example_precision + example_recall) > 0
        else 0.0
    )

    total_examples = (
        example_tp
        + example_fp
        + example_fn
        + example_tn
    )

    accuracy = (
        (example_tp + example_tn) / total_examples
        if total_examples > 0
        else 0.0
    )

    metrics = {
        "Human spans": total_human,
        "Predicted spans": total_predicted,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Span Precision": precision,
        "Span Recall": recall,
        "Span F1": f1,

        "Human-positive examples": (
            example_tp + example_fn
        ),

        "Predicted-positive examples": (
            example_tp + example_fp
        ),

        "Example TP": example_tp,
        "Example FP": example_fp,
        "Example FN": example_fn,
        "Example TN": example_tn,

        "Example Precision": example_precision,
        "Example Recall": example_recall,
        "Example F1": example_f1,
        "Example Accuracy": accuracy,
    }

    return metrics, pd.DataFrame(detailed_rows)


# ============================================================
# STEP 5C — PREPARE THE EXACT SAME TEST INPUTS
# ============================================================

print("\n" + "=" * 90)
print("STEP 5C — PREPARING SAME 40 TEST INPUTS")
print("=" * 90)

# Use the fixed test records.
# These are the same records used for medical fine-tuning evaluation.

corrected_test_dataset = CorrectedRagTruthDataset(
    test_records,
    tokenizer
)

print("Test examples:", len(corrected_test_dataset))

assert len(corrected_test_dataset) == 40


# ============================================================
# STEP 5D — FUNCTION TO RUN A TOKEN CLASSIFICATION MODEL
# ============================================================

def run_model_on_test_set(
    model,
    records,
    tokenizer,
    threshold=0.50,
    batch_size=2
):

    model.eval()
    model.to("cuda")

    # Create corrected dataset
    dataset = CorrectedRagTruthDataset(
        records,
        tokenizer
    )

    collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt"
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator
    )

    all_logits = []

    print("Running model inference...")

    with torch.no_grad():

        for batch_idx, batch in enumerate(dataloader):

            input_ids = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits.detach().cpu()

            all_logits.append(logits)

            print(
                f"Batch {batch_idx + 1}/{len(dataloader)} complete",
                end="\r"
            )

    print()

    # --------------------------------------------------------
    # Convert padded batch outputs back to individual examples
    # --------------------------------------------------------

    predictions_dict = {}

    example_index = 0

    for batch_logits, batch in zip(
        all_logits,
        dataloader
    ):
        pass

    # Easier and safer: run each example individually
    # using the same tokenizer/model.
    predictions_dict = {}

    for idx, sample in enumerate(records):

        encoding = tokenizer(
            sample["prompt"],
            sample["answer"],
            truncation=True,
            max_length=4096,
            return_offsets_mapping=True,
            add_special_tokens=True
        )

        input_ids = torch.tensor(
            [encoding["input_ids"]],
            dtype=torch.long
        ).cuda()

        attention_mask = torch.tensor(
            [encoding["attention_mask"]],
            dtype=torch.long
        ).cuda()

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        logits = outputs.logits[0].detach().cpu().numpy()

        spans = get_predicted_spans_from_encoding(
            sample["answer"],
            encoding,
            logits,
            threshold=threshold
        )

        predictions_dict[
            sample["test_id"]
        ] = spans

        print(
            f"Processed {idx + 1}/{len(records)}",
            end="\r"
        )

    print()

    return predictions_dict


# ============================================================
# STEP 5E — LOAD ORIGINAL PRETRAINED LETTUCEDETECT
# ============================================================

print("\n" + "=" * 90)
print("STEP 5E — LOADING ORIGINAL PRETRAINED LETTUCEDETECT")
print("=" * 90)

LETTUCE_MODEL = "KRLabsOrg/lettucedect-base-modernbert-en-v1"

baseline_tokenizer = AutoTokenizer.from_pretrained(
    LETTUCE_MODEL
)

baseline_model = AutoModelForTokenClassification.from_pretrained(
    LETTUCE_MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True
)

baseline_model = baseline_model.cuda()

print("Baseline model loaded.")
print("Model:", LETTUCE_MODEL)


# ============================================================
# STEP 5F — BASELINE INFERENCE
# ============================================================

print("\n" + "=" * 90)
print("STEP 5F — BASELINE INFERENCE")
print("=" * 90)

BASELINE_THRESHOLD = 0.50

baseline_predictions = run_model_on_test_set(
    baseline_model,
    test_records,
    baseline_tokenizer,
    threshold=BASELINE_THRESHOLD,
    batch_size=2
)

print("\nBaseline inference completed.")
print(
    "Examples with predictions:",
    len(baseline_predictions)
)

assert len(baseline_predictions) == 40


# ============================================================
# STEP 5G — BASELINE EVALUATION
# ============================================================

print("\n" + "=" * 90)
print("STEP 5G — BASELINE EVALUATION")
print("=" * 90)

baseline_metrics, baseline_details = evaluate_span_predictions(
    final_test_ground_truth,
    baseline_predictions
)

print("\nBASELINE RESULTS")
print("-" * 60)

for key, value in baseline_metrics.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# FREE BASELINE MODEL FROM GPU
# ============================================================

del baseline_model
del baseline_tokenizer

gc.collect()
torch.cuda.empty_cache()

print("\nBaseline model released from GPU.")


# ============================================================
# STEP 5H — MEDICAL-ADAPTED MODEL EVALUATION
# ============================================================

print("\n" + "=" * 90)
print("STEP 5H — MEDICAL-ADAPTED MODEL EVALUATION")
print("=" * 90)

MEDICAL_THRESHOLD = 0.50

# The trained Trainer model should already exist from the
# successful corrected training run.

adapted_model = trainer.model

adapted_model.eval()
adapted_model.cuda()

print("Using medical-adapted model from the completed training run.")

adapted_predictions = run_model_on_test_set(
    adapted_model,
    test_records,
    tokenizer,
    threshold=MEDICAL_THRESHOLD,
    batch_size=2
)

print("\nMedical-adapted inference completed.")

assert len(adapted_predictions) == 40


# ============================================================
# STEP 5I — MEDICAL-ADAPTED EVALUATION
# ============================================================

print("\n" + "=" * 90)
print("STEP 5I — MEDICAL-ADAPTED EVALUATION")
print("=" * 90)

adapted_metrics, adapted_details = evaluate_span_predictions(
    final_test_ground_truth,
    adapted_predictions
)

print("\nMEDICAL-ADAPTED RESULTS")
print("-" * 60)

for key, value in adapted_metrics.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# STEP 5J — FINAL COMPARISON TABLE
# ============================================================

print("\n" + "=" * 90)
print("STEP 5J — BASELINE vs MEDICAL-ADAPTED")
print("=" * 90)

comparison = pd.DataFrame(
    [
        {
            "Model": "Original LettuceDetect",
            "Span Precision": baseline_metrics["Span Precision"],
            "Span Recall": baseline_metrics["Span Recall"],
            "Span F1": baseline_metrics["Span F1"],
            "Example Precision": baseline_metrics["Example Precision"],
            "Example Recall": baseline_metrics["Example Recall"],
            "Example F1": baseline_metrics["Example F1"],
            "Example Accuracy": baseline_metrics["Example Accuracy"],
        },
        {
            "Model": "Medical Fine-tuned LettuceDetect",
            "Span Precision": adapted_metrics["Span Precision"],
            "Span Recall": adapted_metrics["Span Recall"],
            "Span F1": adapted_metrics["Span F1"],
            "Example Precision": adapted_metrics["Example Precision"],
            "Example Recall": adapted_metrics["Example Recall"],
            "Example F1": adapted_metrics["Example F1"],
            "Example Accuracy": adapted_metrics["Example Accuracy"],
        }
    ]
)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# STEP 5K — CALCULATE IMPROVEMENT
# ============================================================

print("\n" + "=" * 90)
print("STEP 5K — PERFORMANCE IMPROVEMENT")
print("=" * 90)

improvement = {
    "Span Precision": (
        adapted_metrics["Span Precision"]
        - baseline_metrics["Span Precision"]
    ),

    "Span Recall": (
        adapted_metrics["Span Recall"]
        - baseline_metrics["Span Recall"]
    ),

    "Span F1": (
        adapted_metrics["Span F1"]
        - baseline_metrics["Span F1"]
    ),

    "Example Precision": (
        adapted_metrics["Example Precision"]
        - baseline_metrics["Example Precision"]
    ),

    "Example Recall": (
        adapted_metrics["Example Recall"]
        - baseline_metrics["Example Recall"]
    ),

    "Example F1": (
        adapted_metrics["Example F1"]
        - baseline_metrics["Example F1"]
    ),

    "Example Accuracy": (
        adapted_metrics["Example Accuracy"]
        - baseline_metrics["Example Accuracy"]
    )
}

print("\nAbsolute improvement:")
for key, value in improvement.items():
    print(f"{key}: {value:+.4f}")


# ============================================================
# STEP 5L — SAVE COMPARISON RESULTS
# ============================================================

comparison.to_csv(
    "/kaggle/working/baseline_vs_medical_comparison.csv",
    index=False
)

baseline_details.to_csv(
    "/kaggle/working/baseline_error_details.csv",
    index=False
)

adapted_details.to_csv(
    "/kaggle/working/medical_adapted_error_details.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/baseline_vs_medical_comparison.csv")
print("/kaggle/working/baseline_error_details.csv")
print("/kaggle/working/medical_adapted_error_details.csv")


# ============================================================
# STEP 6 — DOMAIN GAP ANALYSIS
# ============================================================

print("\n" + "=" * 90)
print("STEP 6 — DOMAIN GAP / ERROR ANALYSIS")
print("=" * 90)


# ============================================================
# STEP 6A — MERGE BASELINE + ADAPTED RESULTS
# ============================================================

error_analysis = final_test_ground_truth[
    [
        "test_id",
        "question",
        "context",
        "answer",
        "reference_answer",
        "final_decision",
        "overall_label",
        "hallucination_spans",
        "annotator_notes"
    ]
].copy()

error_analysis = error_analysis.merge(
    baseline_details[
        [
            "test_id",
            "predicted_spans",
            "human_positive",
            "predicted_positive",
            "row_tp",
            "row_fp",
            "row_fn"
        ]
    ].rename(
        columns={
            "predicted_spans": "baseline_predicted_spans",
            "human_positive": "baseline_human_positive",
            "predicted_positive": "baseline_predicted_positive",
            "row_tp": "baseline_tp",
            "row_fp": "baseline_fp",
            "row_fn": "baseline_fn"
        }
    ),
    on="test_id",
    how="left"
)

error_analysis = error_analysis.merge(
    adapted_details[
        [
            "test_id",
            "predicted_spans",
            "predicted_positive",
            "row_tp",
            "row_fp",
            "row_fn"
        ]
    ].rename(
        columns={
            "predicted_spans": "adapted_predicted_spans",
            "predicted_positive": "adapted_predicted_positive",
            "row_tp": "adapted_tp",
            "row_fp": "adapted_fp",
            "row_fn": "adapted_fn"
        }
    ),
    on="test_id",
    how="left"
)

print("Error-analysis rows:", len(error_analysis))

assert len(error_analysis) == 40


# ============================================================
# STEP 6B — IDENTIFY FP / FN CASES
# ============================================================

error_analysis["baseline_error_type"] = "Correct / No Error"

error_analysis.loc[
    (
        error_analysis["baseline_human_positive"]
        & (error_analysis["baseline_predicted_positive"] == False)
    ),
    "baseline_error_type"
] = "False Negative"

error_analysis.loc[
    (
        (error_analysis["baseline_human_positive"] == False)
        & error_analysis["baseline_predicted_positive"]
    ),
    "baseline_error_type"
] = "False Positive"


error_analysis["adapted_error_type"] = "Correct / No Error"

error_analysis.loc[
    (
        error_analysis["overall_label"].astype(str).str.lower().isin(
            ["hallucinated", "partially_supported"]
        )
        & (error_analysis["adapted_predicted_positive"] == False)
    ),
    "adapted_error_type"
] = "False Negative"

error_analysis.loc[
    (
        ~error_analysis["overall_label"].astype(str).str.lower().isin(
            ["hallucinated", "partially_supported"]
        )
        & error_analysis["adapted_predicted_positive"]
    ),
    "adapted_error_type"
] = "False Positive"


# ============================================================
# STEP 6C — TEXT-BASED ERROR CATEGORY IDENTIFICATION
# ============================================================

def classify_error_categories(row):

    text = " ".join(
        [
            str(row.get("question", "")),
            str(row.get("answer", "")),
            str(row.get("context", "")),
            str(row.get("annotator_notes", "")),
            str(row.get("final_decision", "")),
            str(row.get("overall_label", ""))
        ]
    ).lower()

    categories = []

    # --------------------------------------------------------
    # Negation
    # --------------------------------------------------------

    negation_terms = [
        "no evidence",
        "not associated",
        "does not",
        "do not",
        "without",
        "absence",
        "negative",
        "neither",
        "no increased",
        "not indicate",
        "not recommended",
        "not shown"
    ]

    if any(term in text for term in negation_terms):
        categories.append("Clinical negation")

    # --------------------------------------------------------
    # Dosage / numerical
    # --------------------------------------------------------

    dosage_terms = [
        "mg",
        "mcg",
        "µg",
        "ml",
        "mL",
        "dose",
        "dosage",
        "%",
        "percent",
        "times daily",
        "per day",
        "once daily",
        "twice daily",
        "hours",
        "days",
        "weeks",
        "months",
        "years",
        "mmhg",
        "bpm",
        "kg"
    ]

    if any(term.lower() in text for term in dosage_terms):
        categories.append("Dosage / numerical")

    # --------------------------------------------------------
    # Medical terminology
    # --------------------------------------------------------

    terminology_terms = [
        "diagnosis",
        "diagnosed",
        "disease",
        "syndrome",
        "carcinoma",
        "lymphoma",
        "melanoma",
        "metastasis",
        "antibody",
        "antigen",
        "mutation",
        "infection",
        "treatment",
        "therapy",
        "chemotherapy",
        "radiotherapy",
        "immunotherapy",
        "histology",
        "pathology",
        "clinical"
    ]

    if any(term in text for term in terminology_terms):
        categories.append("Medical terminology")

    # --------------------------------------------------------
    # Contradiction indicators
    # --------------------------------------------------------

    contradiction_terms = [
        "contradict",
        "contradiction",
        "conflict",
        "inconsistent",
        "however",
        "but the evidence",
        "although",
        "despite"
    ]

    if any(term in text for term in contradiction_terms):
        categories.append("Contradiction")

    # --------------------------------------------------------
    # Unsupported claim indicators
    # --------------------------------------------------------

    unsupported_terms = [
        "unsupported",
        "not supported",
        "no evidence",
        "insufficient evidence",
        "cannot be determined",
        "cannot determine",
        "not provided",
        "not mentioned",
        "not stated",
        "not found",
        "hallucinated"
    ]

    if any(term in text for term in unsupported_terms):
        categories.append("Unsupported clinical claim")

    # --------------------------------------------------------
    # Partial support
    # --------------------------------------------------------

    partial_terms = [
        "partial",
        "partially supported",
        "partially",
        "somewhat supported",
        "only partially"
    ]

    if any(term in text for term in partial_terms):
        categories.append("Partially supported")

    # --------------------------------------------------------
    # Default
    # --------------------------------------------------------

    if not categories:
        categories.append("Other")

    return ", ".join(dict.fromkeys(categories))


error_analysis["error_categories"] = error_analysis.apply(
    classify_error_categories,
    axis=1
)


# ============================================================
# STEP 6D — SHOW BASELINE FALSE POSITIVES
# ============================================================

baseline_fp_cases = error_analysis[
    error_analysis["baseline_fp"] > 0
].copy()

baseline_fn_cases = error_analysis[
    error_analysis["baseline_fn"] > 0
].copy()

adapted_fp_cases = error_analysis[
    error_analysis["adapted_fp"] > 0
].copy()

adapted_fn_cases = error_analysis[
    error_analysis["adapted_fn"] > 0
].copy()


print("\nBASELINE FALSE POSITIVE CASES:",
      len(baseline_fp_cases))

print("BASELINE FALSE NEGATIVE CASES:",
      len(baseline_fn_cases))

print("\nMEDICAL-ADAPTED FALSE POSITIVE CASES:",
      len(adapted_fp_cases))

print("MEDICAL-ADAPTED FALSE NEGATIVE CASES:",
      len(adapted_fn_cases))


# ============================================================
# STEP 6E — ERROR CATEGORY COUNTS
# ============================================================

category_names = [
    "Clinical negation",
    "Dosage / numerical",
    "Medical terminology",
    "Contradiction",
    "Unsupported clinical claim",
    "Partially supported",
    "Other"
]

category_rows = []

for category in category_names:

    mask = error_analysis[
        "error_categories"
    ].str.contains(
        re.escape(category),
        case=False,
        na=False
    )

    category_rows.append(
        {
            "Error Category": category,

            "Cases": int(mask.sum()),

            "Baseline FP Cases": int(
                (
                    mask
                    & (error_analysis["baseline_fp"] > 0)
                ).sum()
            ),

            "Baseline FN Cases": int(
                (
                    mask
                    & (error_analysis["baseline_fn"] > 0)
                ).sum()
            ),

            "Adapted FP Cases": int(
                (
                    mask
                    & (error_analysis["adapted_fp"] > 0)
                ).sum()
            ),

            "Adapted FN Cases": int(
                (
                    mask
                    & (error_analysis["adapted_fn"] > 0)
                ).sum()
            )
        }
    )

category_summary = pd.DataFrame(
    category_rows
)


# ============================================================
# STEP 6F — PRINT ERROR CATEGORY TABLE
# ============================================================

print("\n" + "=" * 90)
print("ERROR CATEGORY SUMMARY")
print("=" * 90)

print(
    category_summary.to_string(
        index=False
    )
)


# ============================================================
# STEP 6G — PRINT IMPORTANT FP / FN EXAMPLES
# ============================================================

print("\n" + "=" * 90)
print("BASELINE FALSE NEGATIVE EXAMPLES")
print("=" * 90)

for _, row in baseline_fn_cases.head(10).iterrows():

    print("\nTest ID:", row["test_id"])
    print("Question:", row["question"])
    print("Answer:", row["answer"])
    print("Human spans:", row["hallucination_spans"])
    print("Baseline predictions:",
          row["baseline_predicted_spans"])
    print("Categories:",
          row["error_categories"])
    print("Notes:",
          row["annotator_notes"])


print("\n" + "=" * 90)
print("BASELINE FALSE POSITIVE EXAMPLES")
print("=" * 90)

for _, row in baseline_fp_cases.head(10).iterrows():

    print("\nTest ID:", row["test_id"])
    print("Question:", row["question"])
    print("Answer:", row["answer"])
    print("Human spans:", row["hallucination_spans"])
    print("Baseline predictions:",
          row["baseline_predicted_spans"])
    print("Categories:",
          row["error_categories"])
    print("Notes:",
          row["annotator_notes"])


print("\n" + "=" * 90)
print("MEDICAL-ADAPTED FALSE NEGATIVE EXAMPLES")
print("=" * 90)

for _, row in adapted_fn_cases.head(10).iterrows():

    print("\nTest ID:", row["test_id"])
    print("Question:", row["question"])
    print("Answer:", row["answer"])
    print("Human spans:", row["hallucination_spans"])
    print("Adapted predictions:",
          row["adapted_predicted_spans"])
    print("Categories:",
          row["error_categories"])
    print("Notes:",
          row["annotator_notes"])


print("\n" + "=" * 90)
print("MEDICAL-ADAPTED FALSE POSITIVE EXAMPLES")
print("=" * 90)

for _, row in adapted_fp_cases.head(10).iterrows():

    print("\nTest ID:", row["test_id"])
    print("Question:", row["question"])
    print("Answer:", row["answer"])
    print("Human spans:", row["hallucination_spans"])
    print("Adapted predictions:",
          row["adapted_predicted_spans"])
    print("Categories:",
          row["error_categories"])
    print("Notes:",
          row["annotator_notes"])


# ============================================================
# STEP 6H — SAVE COMPLETE ERROR ANALYSIS
# ============================================================

error_analysis.to_csv(
    "/kaggle/working/final_error_analysis.csv",
    index=False
)

category_summary.to_csv(
    "/kaggle/working/error_category_summary.csv",
    index=False
)

print("\nSaved error-analysis files:")
print("/kaggle/working/final_error_analysis.csv")
print("/kaggle/working/error_category_summary.csv")


# ============================================================
# STEP 6I — FINAL PROJECT SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FINAL STEP 5 + STEP 6 SUMMARY")
print("=" * 90)

print("\nFINAL TEST SET")
print("-" * 50)
print("Examples:", 40)
print("Human-positive examples:", 17)
print("Human hallucination spans:", 26)

print("\nBASELINE — ORIGINAL LETTUCEDETECT")
print("-" * 50)
print(
    f"Span Precision : {baseline_metrics['Span Precision']:.4f}"
)
print(
    f"Span Recall    : {baseline_metrics['Span Recall']:.4f}"
)
print(
    f"Span F1        : {baseline_metrics['Span F1']:.4f}"
)
print(
    f"Example F1     : {baseline_metrics['Example F1']:.4f}"
)
print(
    f"Accuracy       : {baseline_metrics['Example Accuracy']:.4f}"
)

print("\nMEDICAL FINE-TUNED LETTUCEDETECT")
print("-" * 50)
print(
    f"Span Precision : {adapted_metrics['Span Precision']:.4f}"
)
print(
    f"Span Recall    : {adapted_metrics['Span Recall']:.4f}"
)
print(
    f"Span F1        : {adapted_metrics['Span F1']:.4f}"
)
print(
    f"Example F1     : {adapted_metrics['Example F1']:.4f}"
)
print(
    f"Accuracy       : {adapted_metrics['Example Accuracy']:.4f}"
)

print("\nIMPROVEMENT")
print("-" * 50)
print(
    f"Span F1 change    : "
    f"{improvement['Span F1']:+.4f}"
)
print(
    f"Example F1 change : "
    f"{improvement['Example F1']:+.4f}"
)
print(
    f"Accuracy change   : "
    f"{improvement['Example Accuracy']:+.4f}"
)

print("\nSTEP 5 — BASELINE VS FINE-TUNED: COMPLETE")
print("STEP 6 — DOMAIN GAP / ERROR ANALYSIS: COMPLETE")

STEP 5A — FINAL TEST GROUND TRUTH VERIFICATION
Final test examples: 40
Human-positive examples: 17
Human hallucination spans: 26
Ground truth verification PASSED.

STEP 5C — PREPARING SAME 40 TEST INPUTS
Test examples: 40

STEP 5E — LOADING ORIGINAL PRETRAINED LETTUCEDETECT


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Baseline model loaded.
Model: KRLabsOrg/lettucedect-base-modernbert-en-v1

STEP 5F — BASELINE INFERENCE
Running model inference...
Batch 20/20 complete
Processed 40/40

Baseline inference completed.
Examples with predictions: 40

STEP 5G — BASELINE EVALUATION

BASELINE RESULTS
------------------------------------------------------------
Human spans: 26
Predicted spans: 3
TP: 0
FP: 3
FN: 26
Span Precision: 0.0000
Span Recall: 0.0000
Span F1: 0.0000
Human-positive examples: 17
Predicted-positive examples: 2
Example TP: 0
Example FP: 2
Example FN: 17
Example TN: 21
Example Precision: 0.0000
Example Recall: 0.0000
Example F1: 0.0000
Example Accuracy: 0.5250

Baseline model released from GPU.

STEP 5H — MEDICAL-ADAPTED MODEL EVALUATION
Using medical-adapted model from the completed training run.
Running model inference...
Batch 20/20 complete
Processed 40/40

Medical-adapted inference completed.

STEP 5I — MEDICAL-ADAPTED EVALUATION

MEDICAL-ADAPTED RESULTS
---------------------------------